# Financial Data Modeling Pipeline
**Correlation Analysis, Time Series Models, Leading/Lagging Indicators, and Survival Analysis**

This notebook builds various models using the financial data collected in the ETL pipeline:

## Model Types:
- **Correlation Analysis**: Cross-correlation at different time lags to identify leading, current, and lagging indicators
- **Linear Models**: Linear regression models for trend analysis
- **Exponential Models**: Exponential growth/decay models
- **Seasonal Models**: Seasonal decomposition and analysis
- **Survival Analysis (Time-to-Event)**: Competing risks models to predict time to reach price targets
  - ATR-based dynamic thresholds for win/loss targets
  - Fine-Gray subdistribution hazard models
  - Cause-specific Cox proportional hazards
  - Random Survival Forests for competing risks
- **Stock Screening**: Rank stocks by predicted win probability
- **Kelly Criterion Sizing**: Optimal position sizing based on edge and odds

All model results are saved as parquet files in the data folder for subsequent analysis.

## Setup and Libraries

In [ ]:
setwd("datacollection")

In [ ]:
# Set options
knitr::opts_chunk$set(echo = TRUE, message = FALSE, warning = FALSE)
options(scipen = 999)

In [ ]:
# Load custom theme and plotting functions
source("~/renv_start.R")
source("~/theme_money_printer_go_brrr.R")
source("save_plot.R")           # Local version
source("save_table.R")          # Local version
source("generate_report.R")     # New report generator (no ChatGPT)
source("~/nnedl_connection.R")

# Logger
source("~/logger.R")
# set_loginfo_error_handler()

# Load dotenv if needed for other configs
if (file.exists("~/.env")) {
  dotenv::load_dot_env("~/.env")
}

# Set global options
options(scipen = 999)  # Disable scientific notation
set.seed(42)  # For reproducibility

# ============================================================================
# POWERPOINT REPORT SETUP
# ============================================================================
# Initialize the report content list
# Each item will have: type ("plot" or "table"), data, title, section
report_content <- list()

# Helper function to add content to report
add_report_item <- function(type, data, title, section) {
  item <- list(type = type, data = data, title = title, section = section)
  report_content <<- c(report_content, list(item))
  cat(sprintf("📊 Added to report: [%s] %s (%s)\n", type, title, section))
}

print("✅ Report generator initialized. Content will be collected throughout the analysis.")

In [ ]:
# Define required packages
required_packages <- c(
  # Data manipulation
  "dplyr",
  "tidyr",
  "lubridate",
  "data.table",
  
  # Database
  "DBI",
  "duckdb",
  
  # Time series analysis
  "forecast",
  "TSA",
  "tseries",
  "zoo",
  "xts",
  
  # Statistical modeling
  "broom",
  "modelr",
  "mgcv",
  "splines",
  
  # File handling
  "arrow",          # For parquet files
  "feather",
  
  # Utilities
  "glue",
  "progress",
  "corrr",          # Correlation analysis
  "purrr",          # Functional programming
  "ggplot2",
  "plotly",
  
  # Network visualization
  "igraph",         # Network analysis and visualization
  "visNetwork",     # Interactive network visualization
  "ggraph",         # Network visualization with ggplot2
  "tidygraph",      # Tidy network data manipulation
  
  # Survival Analysis (NEW)
  "survival",       # Core survival analysis
  "survminer",      # Survival visualization
  "cmprsk",         # Competing risks
  "randomForestSRC", # Random Survival Forests
  # "pec",          # Prediction error curves - optional, not used
  # "riskRegression", # Risk regression models - optional if causing issues
  # "prodlim"       # Product-limit estimation - optional
  
  # Technical Indicators (NEW)
  "TTR"             # Technical Trading Rules
)

# Install and load packages using loop
for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

print("All packages loaded successfully!")

## Database Connection

In [ ]:
# Database configuration - ROBUST SOLUTION
# Creates a temporary copy to avoid any locking issues
DB_PATH <- "data/financial_data.duckdb"
DB_COPY_PATH <- "data/financial_data_readonly.duckdb"

# Check if source database exists
if (!file.exists(DB_PATH)) {
  stop(glue("Database file not found at {DB_PATH}. Please run 01_etl.ipynb first."))
}

# Force close any existing DuckDB connections in this R session
if (exists("con")) {
  try(DBI::dbDisconnect(con, shutdown = TRUE), silent = TRUE)
  rm(con)
}
if (exists("duckdb_drv")) {
  try(duckdb::duckdb_shutdown(duckdb_drv), silent = TRUE)
  rm(duckdb_drv)
}

# Force garbage collection
invisible(gc(full = TRUE))
Sys.sleep(1)

# Strategy: Copy the database file to avoid lock conflicts
# This ensures we can always read the data even if another process has the original locked
cat("📁 Creating read-only copy of database...\n")

# Remove any existing copy
if (file.exists(DB_COPY_PATH)) {
  file.remove(DB_COPY_PATH)
}

# Copy the main database file (this works even if original is locked for write)
file.copy(DB_PATH, DB_COPY_PATH, overwrite = TRUE)
cat("✅ Database copied successfully\n")

# Remove any .wal files from the copy (they might cause issues)
wal_file <- paste0(DB_COPY_PATH, ".wal")
if (file.exists(wal_file)) file.remove(wal_file)

# Connect to the COPY (not the original)
cat("🔌 Connecting to database copy...\n")
con <- DBI::dbConnect(duckdb::duckdb(), dbdir = DB_COPY_PATH, read_only = TRUE)

# Verify connection
total_records <- DBI::dbGetQuery(con, "SELECT COUNT(*) as count FROM financial_data")$count
cat(glue("✅ Connected successfully! Total records: {format(total_records, big.mark=',')}\n"))

# Show available tables
tables <- DBI::dbListTables(con)
cat(glue("\n📊 Available tables: {paste(tables, collapse=', ')}\n"))

In [ ]:

# Check data overview
data_overview <- dbGetQuery(con, "
  SELECT 
    origin,
    COUNT(*) as records,
    COUNT(DISTINCT series) as series_count,
    MIN(date) as min_date,
    MAX(date) as max_date
  FROM financial_data 
  GROUP BY origin
  ORDER BY records DESC
")

print("Data overview by origin:")
print(data_overview)

## Data Loading and Preparation

In [ ]:
# Load all financial data - now using LOCF-applied data directly from financial_data table
print("Loading financial data from database...")

# Load data (LOCF is now applied during ETL, so all data is already daily frequency)
financial_data <- dbGetQuery(con, "
  SELECT origin, series, date, value
  FROM financial_data 
  WHERE value IS NOT NULL
  ORDER BY date, origin, series
")

print(glue("Loaded {nrow(financial_data)} records (LOCF-expanded during ETL)"))

# Convert to data.table for efficient operations
financial_dt <- as.data.table(financial_data)
financial_dt[, date := as.Date(date)]
financial_dt[, value := as.numeric(value)]

# Create a unique series identifier combining origin and series
financial_dt[, full_series := paste(origin, series, sep = "_")]

# Get series with sufficient data (at least 100 observations)
series_counts <- financial_dt[, .N, by = full_series][N >= 100]
valid_series <- series_counts$full_series

print(glue("Found {length(valid_series)} series with sufficient data (>=100 observations)"))

# Filter data to only include valid series
model_data <- financial_dt[full_series %in% valid_series]

print(glue("Model data contains {nrow(model_data)} records across {length(valid_series)} series"))

## Utility Functions

In [ ]:
# Function to calculate cross-correlation with lags
calculate_cross_correlation <- function(x, y, max_lag = 30, min_overlap = 50) {
  
  # Remove missing values
  complete_cases <- complete.cases(x, y)
  x_clean <- x[complete_cases]
  y_clean <- y[complete_cases]
  
  # Check if we have enough data
  if (length(x_clean) < min_overlap) {
    return(NULL)
  }
  
  tryCatch({
    # Calculate cross-correlation
    ccf_result <- ccf(x_clean, y_clean, lag.max = max_lag, plot = FALSE)
    
    # Extract results
    correlations <- data.frame(
      lag = ccf_result$lag[,,1],
      correlation = ccf_result$acf[,,1],
      n_obs = length(x_clean)
    )
    
    return(correlations)
  }, error = function(e) {
    return(NULL)
  })
}

# Function to fit linear model
fit_linear_model <- function(data, series_name) {
  tryCatch({
    # Create time index
    data$time_index <- as.numeric(data$date - min(data$date))
    
    # Fit linear model
    model <- lm(value ~ time_index, data = data)
    
    # Extract model statistics
    model_summary <- broom::glance(model)
    coefficients <- broom::tidy(model)
    
    result <- list(
      series = series_name,
      model_type = "linear",
      r_squared = model_summary$r.squared,
      adj_r_squared = model_summary$adj.r.squared,
      p_value = model_summary$p.value,
      slope = coefficients$estimate[2],
      slope_pvalue = coefficients$p.value[2],
      intercept = coefficients$estimate[1],
      n_obs = nrow(data),
      aic = model_summary$AIC,
      bic = model_summary$BIC
    )
    
    return(result)
  }, error = function(e) {
    return(NULL)
  })
}

# Function to fit exponential model
fit_exponential_model <- function(data, series_name) {
  tryCatch({
    # Only fit if all values are positive
    if (any(data$value <= 0)) {
      return(NULL)
    }
    
    # Create time index
    data$time_index <- as.numeric(data$date - min(data$date))
    
    # Fit exponential model: log(y) = a + b*t
    model <- lm(log(value) ~ time_index, data = data)
    
    # Extract model statistics
    model_summary <- broom::glance(model)
    coefficients <- broom::tidy(model)
    
    result <- list(
      series = series_name,
      model_type = "exponential",
      r_squared = model_summary$r.squared,
      adj_r_squared = model_summary$adj.r.squared,
      p_value = model_summary$p.value,
      growth_rate = coefficients$estimate[2], # This is the exponential growth rate
      growth_rate_pvalue = coefficients$p.value[2],
      log_intercept = coefficients$estimate[1],
      n_obs = nrow(data),
      aic = model_summary$AIC,
      bic = model_summary$BIC
    )
    
    return(result)
  }, error = function(e) {
    return(NULL)
  })
}

# Function to perform seasonal decomposition
perform_seasonal_decomposition <- function(data, series_name, frequency = 12) {
  tryCatch({
    # Convert to time series
    ts_data <- ts(data$value, frequency = frequency)
    
    # Perform seasonal decomposition
    decomp <- decompose(ts_data, type = "additive")
    
    # Calculate seasonal strength and trend strength
    seasonal_var <- var(decomp$seasonal, na.rm = TRUE)
    trend_var <- var(decomp$trend, na.rm = TRUE)
    remainder_var <- var(decomp$random, na.rm = TRUE)
    total_var <- var(ts_data, na.rm = TRUE)
    
    seasonal_strength <- seasonal_var / (seasonal_var + remainder_var)
    trend_strength <- trend_var / (trend_var + remainder_var)
    
    result <- list(
      series = series_name,
      model_type = "seasonal",
      seasonal_strength = seasonal_strength,
      trend_strength = trend_strength,
      seasonal_variance = seasonal_var,
      trend_variance = trend_var,
      remainder_variance = remainder_var,
      total_variance = total_var,
      frequency = frequency,
      n_obs = length(ts_data)
    )
    
    return(result)
  }, error = function(e) {
    return(NULL)
  })
}

print("Utility functions defined successfully!")

## Cross-Correlation Analysis

In [ ]:
print("Starting cross-correlation analysis...")

# With LOCF-filled data, we already have aligned daily observations
# No need to create date grid - data is already on daily frequency

# Create wide format data for correlation analysis
wide_format <- dcast(model_data, date ~ full_series, value.var = "value")
wide_format[, date := as.Date(date)]

print(glue("Created wide format data with {nrow(wide_format)} dates and {ncol(wide_format)-1} series"))

# Calculate cross-correlations for key pairs
# Focus on major economic indicators and their relationships
key_series <- c(
  "FRED_DFF",           # Federal Funds Rate
  "FRED_UNRATE",        # Unemployment Rate
  "FRED_CPIAUCSL",      # CPI
  "FRED_GDP",           # GDP
  "YAHOO_^GSPC",        # S&P 500
  "YAHOO_BTC-USD",      # Bitcoin
  "YAHOO_GC=F",         # Gold
  "FRED_MORTGAGE30US",  # Mortgage rates
  "FRED_DGS10"          # 10-Year Treasury
)

# Filter to available key series
available_key_series <- key_series[key_series %in% names(wide_format)]
print(glue("Found {length(available_key_series)} key series for correlation analysis"))

# Calculate cross-correlations
correlation_results <- list()
pb <- progress_bar$new(total = length(available_key_series)^2, 
                      format = "[:bar] :percent :current/:total ETA: :eta")

for (i in 1:length(available_key_series)) {
  for (j in 1:length(available_key_series)) {
    pb$tick()
    
    if (i != j) {  # Don't correlate series with itself
      series1 <- available_key_series[i]
      series2 <- available_key_series[j]
      
      x <- wide_format[[series1]]
      y <- wide_format[[series2]]
      
      ccf_result <- calculate_cross_correlation(x, y, max_lag = 60)
      
      if (!is.null(ccf_result)) {
        ccf_result$series1 <- series1
        ccf_result$series2 <- series2
        correlation_results[[paste(series1, series2, sep = "_vs_")]] <- ccf_result
      }
    }
  }
}

# Combine all correlation results
if (length(correlation_results) > 0) {
  correlation_df <- rbindlist(correlation_results, fill = TRUE)
  print(glue("Calculated cross-correlations: {nrow(correlation_df)} lag-correlation pairs"))
} else {
  print("No correlation results generated")
  correlation_df <- data.table()
}

## Linear and Exponential Model Fitting

In [ ]:
print("Fitting linear and exponential models...")

# Fit models for each series
linear_results <- list()
exponential_results <- list()

print(glue("Total valid series to process: {length(valid_series)}"))

pb <- progress_bar$new(total = length(valid_series), 
                      format = "[:bar] :percent :current/:total ETA: :eta")

for (series in valid_series) {
  pb$tick()
  
  # Get data for this series - use explicit variable to avoid scoping issues
  current_series <- series
  series_data <- model_data[full_series == current_series, .(date, value)]
  series_data <- series_data[!is.na(value)]  # Remove missing values
  
  if (nrow(series_data) >= 50) {  # Minimum observations for modeling
    
    # Fit linear model
    linear_result <- fit_linear_model(series_data, series)
    if (!is.null(linear_result)) {
      linear_results[[series]] <- linear_result
    }
    
    # Fit exponential model
    exponential_result <- fit_exponential_model(series_data, series)
    if (!is.null(exponential_result)) {
      exponential_results[[series]] <- exponential_result
    }
  }
}

print(glue("\nLinear models successfully fitted: {length(linear_results)}"))
print(glue("Exponential models successfully fitted: {length(exponential_results)}"))

# Convert results to data frames
if (length(linear_results) > 0) {
  linear_models_df <- rbindlist(linear_results, fill = TRUE)
  print(glue("Fitted linear models: {nrow(linear_models_df)} series"))
} else {
  linear_models_df <- data.table()
}

if (length(exponential_results) > 0) {
  exponential_models_df <- rbindlist(exponential_results, fill = TRUE)
  print(glue("Fitted exponential models: {nrow(exponential_models_df)} series"))
} else {
  exponential_models_df <- data.table()
}

In [ ]:
linear_models_df

In [ ]:
exponential_models_df

## Seasonal Decomposition Analysis

In [ ]:
print("Performing seasonal decomposition analysis...")

# Perform seasonal analysis for series with sufficient data
seasonal_results <- list()

pb <- progress_bar$new(total = length(valid_series), 
                      format = "[:bar] :percent :current/:total ETA: :eta")

for (series in valid_series) {
  pb$tick()
  
  # Get data for this series - use explicit variable to avoid scoping issues
  current_series <- series
  series_data <- model_data[full_series == current_series, .(date, value)]
  series_data <- series_data[!is.na(value)][order(date)]  # Remove missing values and sort
  
  # Need at least 3 years of data for meaningful seasonal analysis
  if (nrow(series_data) >= 36) {
    
    # Try different frequencies
    frequencies <- c(12, 4, 52)  # Monthly, quarterly, weekly
    freq_names <- c("monthly", "quarterly", "weekly")
    
    for (i in 1:length(frequencies)) {
      freq <- frequencies[i]
      freq_name <- freq_names[i]
      
      if (nrow(series_data) >= 2 * freq) {  # Need at least 2 cycles
        seasonal_result <- perform_seasonal_decomposition(series_data, series, freq)
        
        if (!is.null(seasonal_result)) {
          seasonal_result$frequency_type <- freq_name
          seasonal_results[[paste(series, freq_name, sep = "_")]] <- seasonal_result
        }
      }
    }
  }
}

# Convert results to data frame
if (length(seasonal_results) > 0) {
  seasonal_models_df <- rbindlist(seasonal_results, fill = TRUE)
  print(glue("Performed seasonal decomposition: {nrow(seasonal_models_df)} series-frequency combinations"))
} else {
  seasonal_models_df <- data.table()
}

In [ ]:
seasonal_models_df

## Leading/Lagging Indicator Analysis

In [ ]:
print("Analyzing leading and lagging indicators...")

# Analyze the correlation results to identify leading/lagging relationships
if (nrow(correlation_df) > 0) {
  
  # Find the lag with maximum absolute correlation for each pair
  max_correlations <- correlation_df[, .(
    max_correlation = correlation[which.max(abs(correlation))],
    optimal_lag = lag[which.max(abs(correlation))],
    max_abs_correlation = max(abs(correlation), na.rm = TRUE)
  ), by = .(series1, series2)]
  
  # Classify relationships
  max_correlations[, relationship_type := case_when(
    optimal_lag > 0 ~ "leading",     # series1 leads series2
    optimal_lag < 0 ~ "lagging",     # series1 lags series2  
    optimal_lag == 0 ~ "concurrent",  # series move together
    TRUE ~ "unclear"
  )]
  
  # Filter for strong relationships (correlation > 0.3)
  strong_relationships <- max_correlations[max_abs_correlation > 0.3]
  
  print(glue("Found {nrow(strong_relationships)} strong relationships (|correlation| > 0.3)"))
  
  # Summary by relationship type
  relationship_summary <- strong_relationships[, .N, by = relationship_type]
  print("Relationship type summary:")
  print(relationship_summary)
  
} else {
  strong_relationships <- data.table()
  print("No correlation data available for relationship analysis")
}

In [ ]:
strong_relationships

## Network Graph Visualizations

In [ ]:
# Filter strong relationships with higher threshold for cleaner visualization
print("Preparing network graph data...")

# Filter for very strong relationships (correlation > 0.5)
if (nrow(strong_relationships) > 0) {
  network_relationships <- strong_relationships[max_abs_correlation > 0.5]
  
  print(glue("Network contains {nrow(network_relationships)} relationships with |correlation| > 0.5"))
  print(glue("Unique nodes: {length(unique(c(network_relationships$series1, network_relationships$series2)))}"))
  
  # Summary statistics
  print("\nCorrelation strength distribution:")
  print(summary(network_relationships$max_abs_correlation))
  
  print("\nRelationship types:")
  print(table(network_relationships$relationship_type))
  
  print("\nPositive vs Negative correlations:")
  print(table(ifelse(network_relationships$max_correlation > 0, "Positive", "Negative")))
  
} else {
  print("No strong relationships found for network visualization")
  network_relationships <- data.table()
}

### Summary Tables

In [ ]:
if (nrow(network_relationships) > 0) {
  print("=== TOP 15 STRONGEST RELATIONSHIPS ===")
  top_relationships <- network_relationships[order(-max_abs_correlation)][1:min(15, .N)]
  print(top_relationships[, .(series1, series2, correlation = round(max_correlation, 3), 
                              lag = optimal_lag, abs_corr = round(max_abs_correlation, 3), 
                              type = relationship_type)])
  
  print("\n=== MOST CONNECTED NODES ===")
  # Count connections for each series
  node_connections <- rbind(
    network_relationships[, .(node = series1)],
    network_relationships[, .(node = series2)]
  )[, .N, by = node][order(-N)]
  
  print(head(node_connections, 10))
  
  print("\n=== CORRELATION BY RELATIONSHIP TYPE ===")
  type_summary <- network_relationships[, .(
    count = .N,
    mean_correlation = mean(max_abs_correlation),
    median_correlation = median(max_abs_correlation),
    max_correlation = max(max_abs_correlation)
  ), by = relationship_type]
  print(type_summary)
}

### Interactive Network (visNetwork)

In [ ]:
if (nrow(network_relationships) > 0) {
  print("Creating interactive network visualization...")
  
  # Prepare nodes
  all_nodes <- unique(c(network_relationships$series1, network_relationships$series2))
  nodes_df <- data.frame(
    id = all_nodes,
    label = all_nodes,
    title = paste("<b>", all_nodes, "</b>"),  # Tooltip
    stringsAsFactors = FALSE
  )
  
  # Prepare edges
  edges_df <- network_relationships %>%
    mutate(
      from = series1,
      to = series2,
      value = max_abs_correlation,
      width = max_abs_correlation * 5,  # Scale width
      color = ifelse(max_correlation > 0, "#2ECC71", "#E74C3C"),  # Green for positive, red for negative
      title = paste0(
        "<b>", series1, " → ", series2, "</b><br>",
        "Correlation: ", round(max_correlation, 3), "<br>",
        "Lag: ", optimal_lag, " days<br>",
        "Type: ", relationship_type
      ),
      arrows = "to"
    ) %>%
    select(from, to, value, width, color, title, arrows)
  
  # Create interactive network
  vis_network <- visNetwork(nodes_df, edges_df, width = "100%", height = "600px") %>%
    visOptions(
      highlightNearest = list(enabled = TRUE, hover = TRUE),
      nodesIdSelection = TRUE
    ) %>%
    visPhysics(
      solver = "forceAtlas2Based",
      forceAtlas2Based = list(gravitationalConstant = -50)
    ) %>%
    visLayout(randomSeed = 42) %>%
    visInteraction(dragNodes = TRUE, dragView = TRUE, zoomView = TRUE)
  
  # Display
  vis_network
  
} else {
  print("No data available for network visualization")
}

### Static Network (ggraph) - Force-Directed Layout

In [ ]:
if (nrow(network_relationships) > 0) {
  print("Creating static network with force-directed layout...")
  
  # Create igraph object
  edges_for_graph <- network_relationships %>%
    select(from = series1, to = series2, 
           correlation = max_correlation,
           abs_correlation = max_abs_correlation,
           lag = optimal_lag,
           type = relationship_type)
  
  # Create graph
  g <- graph_from_data_frame(edges_for_graph, directed = TRUE)
  
  # Convert to tidygraph
  tidy_g <- as_tbl_graph(g)
  
  # Create visualization
  p1 <- ggraph(tidy_g, layout = "fr") +  # Fruchterman-Reingold
    geom_edge_link(
      aes(color = correlation, width = abs_correlation),
      arrow = arrow(length = unit(3, 'mm')),
      end_cap = circle(3, 'mm'),
      alpha = 0.7
    ) +
    geom_node_point(size = 8, color = "#3498DB", alpha = 0.8) +
    geom_node_text(aes(label = name), size = 3, repel = TRUE) +
    scale_edge_color_gradient2(
      low = "#E74C3C",      # Red for negative
      mid = "#95A5A6",      # Gray for neutral
      high = "#2ECC71",     # Green for positive
      midpoint = 0,
      name = "Correlation"
    ) +
    scale_edge_width_continuous(range = c(0.5, 3), name = "Abs. Corr.") +
    theme_void() +
    theme(
      legend.position = "bottom",
      plot.title = element_text(hjust = 0.5, size = 14, face = "bold")
    ) +
    labs(title = "Financial Indicator Network - Force-Directed Layout")
  
  print(p1)
  
} else {
  print("No data available for network visualization")
}

In [ ]:
# Add network plot 1 to report
if (exists("p1")) {
  add_report_item("plot", p1, "Financial Indicator Network - Force-Directed", "1. Correlation Analysis")
}

### Static Network (ggraph) - Circular Layout

In [ ]:
if (nrow(network_relationships) > 0) {
  print("Creating static network with circular layout...")
  
  # Create visualization with circular layout
  p2 <- ggraph(tidy_g, layout = "circle") +
    geom_edge_arc(
      aes(color = correlation, width = abs_correlation),
      arrow = arrow(length = unit(3, 'mm')),
      end_cap = circle(3, 'mm'),
      alpha = 0.6,
      strength = 0.3
    ) +
    geom_node_point(size = 10, color = "#3498DB", alpha = 0.8) +
    geom_node_text(aes(label = name), size = 3.5, repel = FALSE) +
    scale_edge_color_gradient2(
      low = "#E74C3C",
      mid = "#95A5A6",
      high = "#2ECC71",
      midpoint = 0,
      name = "Correlation"
    ) +
    scale_edge_width_continuous(range = c(0.5, 3), name = "Abs. Corr.") +
    theme_void() +
    theme(
      legend.position = "bottom",
      plot.title = element_text(hjust = 0.5, size = 14, face = "bold")
    ) +
    labs(title = "Financial Indicator Network - Circular Layout")
  
  print(p2)
  
} else {
  print("No data available for network visualization")
}

In [ ]:
# Add network plot 2 to report
if (exists("p2")) {
  add_report_item("plot", p2, "Financial Indicator Network - Circular Layout", "1. Correlation Analysis")
}

### Static Network (ggraph) - Kamada-Kawai Layout

In [ ]:
if (nrow(network_relationships) > 0) {
  print("Creating static network with Kamada-Kawai layout...")
  
  # Create visualization with tree layout
  p3 <- ggraph(tidy_g, layout = "kk") +  # Kamada-Kawai
    geom_edge_link(
      aes(color = correlation, width = abs_correlation),
      arrow = arrow(length = unit(3, 'mm')),
      end_cap = circle(3, 'mm'),
      alpha = 0.7
    ) +
    geom_node_point(size = 8, color = "#3498DB", alpha = 0.8) +
    geom_node_text(aes(label = name), size = 3, repel = TRUE) +
    scale_edge_color_gradient2(
      low = "#E74C3C",
      mid = "#95A5A6",
      high = "#2ECC71",
      midpoint = 0,
      name = "Correlation"
    ) +
    scale_edge_width_continuous(range = c(0.5, 3), name = "Abs. Corr.") +
    theme_void() +
    theme(
      legend.position = "bottom",
      plot.title = element_text(hjust = 0.5, size = 14, face = "bold")
    ) +
    labs(title = "Financial Indicator Network - Kamada-Kawai Layout")
  
  print(p3)
  
} else {
  print("No data available for network visualization")
}

In [ ]:
# Add network plot 3 to report
if (exists("p3")) {
  add_report_item("plot", p3, "Financial Indicator Network - Kamada-Kawai", "1. Correlation Analysis")
}

### Network Analysis Metrics

In [ ]:
if (nrow(network_relationships) > 0) {
  print("=== NETWORK ANALYSIS METRICS ===")
  
  # Calculate network metrics
  g_undirected <- as.undirected(g, mode = "collapse")
  
  # Degree centrality (number of connections)
  degree_cent <- degree(g, mode = "all")
  in_degree <- degree(g, mode = "in")
  out_degree <- degree(g, mode = "out")
  
  # Betweenness centrality (importance as bridge)
  between_cent <- betweenness(g, directed = TRUE)
  
  # Closeness centrality (average distance to other nodes)
  close_cent <- closeness(g, mode = "all")
  
  # Combine metrics
  centrality_df <- data.frame(
    node = names(degree_cent),
    total_connections = degree_cent,
    in_degree = in_degree,
    out_degree = out_degree,
    betweenness = round(between_cent, 2),
    closeness = round(close_cent, 4)
  ) %>%
    arrange(desc(total_connections))
  
  print("\nTop 10 Most Central Nodes:")
  print(head(centrality_df, 10))
  
  # Network-level statistics
  print("\n=== NETWORK STATISTICS ===")
  print(glue("Number of nodes: {vcount(g)}"))
  print(glue("Number of edges: {ecount(g)}"))
  print(glue("Network density: {round(edge_density(g), 4)}"))
  print(glue("Average path length: {round(mean_distance(g), 2)}"))
  print(glue("Network diameter: {diameter(g)}"))
  
  # Community detection
  communities <- cluster_fast_greedy(g_undirected)
  print(glue("\nNumber of communities detected: {length(communities)}"))
  
}

## Model Comparison and Selection

In [ ]:
print("Comparing model performance...")

# Compare linear vs exponential models where both are available
if (nrow(linear_models_df) > 0 && nrow(exponential_models_df) > 0) {
  
  # Merge linear and exponential results
  model_comparison <- merge(
    linear_models_df[, .(series, linear_r2 = r_squared, linear_aic = aic, linear_bic = bic)],
    exponential_models_df[, .(series, exp_r2 = r_squared, exp_aic = aic, exp_bic = bic)],
    by = "series",
    all = TRUE
  )
  
  # Determine best model based on AIC (lower is better)
  model_comparison[, best_model := case_when(
    is.na(linear_aic) & !is.na(exp_aic) ~ "exponential",
    !is.na(linear_aic) & is.na(exp_aic) ~ "linear",
    !is.na(linear_aic) & !is.na(exp_aic) & linear_aic < exp_aic ~ "linear",
    !is.na(linear_aic) & !is.na(exp_aic) & exp_aic < linear_aic ~ "exponential",
    TRUE ~ "unclear"
  )]
  
  # Summary of best models
  best_model_summary <- model_comparison[, .N, by = best_model]
  print("Best model summary:")
  print(best_model_summary)
  
} else {
  model_comparison <- data.table()
  print("Insufficient model data for comparison")
}

## Save Results as Parquet Files

In [ ]:
# print("Saving model results as parquet files...")

# # Ensure data directory exists
# if (!dir.exists("data")) {
#   dir.create("data", recursive = TRUE)
# }

# # Save correlation results
# if (nrow(correlation_df) > 0) {
#   write_parquet(correlation_df, "data/cross_correlations.parquet")
#   print(glue("Saved cross-correlations: {nrow(correlation_df)} rows"))
# }

# # Save linear model results
# if (nrow(linear_models_df) > 0) {
#   write_parquet(linear_models_df, "data/linear_models.parquet")
#   print(glue("Saved linear models: {nrow(linear_models_df)} rows"))
# }

# # Save exponential model results
# if (nrow(exponential_models_df) > 0) {
#   write_parquet(exponential_models_df, "data/exponential_models.parquet")
#   print(glue("Saved exponential models: {nrow(exponential_models_df)} rows"))
# }

# # Save seasonal decomposition results
# if (nrow(seasonal_models_df) > 0) {
#   write_parquet(seasonal_models_df, "data/seasonal_models.parquet")
#   print(glue("Saved seasonal models: {nrow(seasonal_models_df)} rows"))
# }

# # Save leading/lagging indicator analysis
# if (nrow(strong_relationships) > 0) {
#   write_parquet(strong_relationships, "data/leading_lagging_indicators.parquet")
#   print(glue("Saved leading/lagging indicators: {nrow(strong_relationships)} rows"))
# }

# # Save model comparison
# if (nrow(model_comparison) > 0) {
#   write_parquet(model_comparison, "data/model_comparison.parquet")
#   print(glue("Saved model comparison: {nrow(model_comparison)} rows"))
# }

# # Create a summary file with metadata
# modeling_summary <- data.table(
#   analysis_date = Sys.Date(),
#   total_series_analyzed = length(valid_series),
#   correlation_pairs = nrow(correlation_df),
#   linear_models = nrow(linear_models_df),
#   exponential_models = nrow(exponential_models_df),
#   seasonal_decompositions = nrow(seasonal_models_df),
#   strong_relationships = nrow(strong_relationships),
#   data_start_date = min(model_data$date),
#   data_end_date = max(model_data$date)
# )

# write_parquet(modeling_summary, "data/modeling_summary.parquet")
# print("Saved modeling summary")

# print("\n=== MODEL RESULTS SUMMARY ===")
# print(modeling_summary)

## Key Insights and Next Steps

In [ ]:
print("\n=== KEY INSIGHTS ===")

# Top leading indicators
if (nrow(strong_relationships) > 0) {
  leading_indicators <- strong_relationships[
    relationship_type == "leading" & max_abs_correlation > 0.5
  ][order(-max_abs_correlation)]
  
  if (nrow(leading_indicators) > 0) {
    print("\nTop Leading Indicators (correlation > 0.5):")
    print(leading_indicators[1:min(10, nrow(leading_indicators)), .(series1, series2, optimal_lag, max_correlation)])
  }
}

# Best trend models
if (nrow(linear_models_df) > 0) {
  best_trends <- linear_models_df[r_squared > 0.7][order(-r_squared)]
  
  if (nrow(best_trends) > 0) {
    print("\nSeries with Strong Linear Trends (R² > 0.7):")
    print(best_trends[1:min(10, nrow(best_trends)), .(series, r_squared, slope, slope_pvalue)])
  }
}

# Most seasonal series
if (nrow(seasonal_models_df) > 0) {
  most_seasonal <- seasonal_models_df[seasonal_strength > 0.3][order(-seasonal_strength)]
  
  if (nrow(most_seasonal) > 0) {
    print("\nMost Seasonal Series (seasonal strength > 0.3):")
    print(most_seasonal[1:min(10, nrow(most_seasonal)), .(series, frequency_type, seasonal_strength)])
  }
}

print("\n=== FILES CREATED ===")
parquet_files <- list.files("data", pattern = "*.parquet", full.names = TRUE)
for (file in parquet_files) {
  size_kb <- round(file.size(file) / 1024, 2)
  print(glue("{file}: {size_kb} KB"))
}

print("\n=== NEXT STEPS ===")
cat("
1. Load the parquet files in your analysis script using:
   - correlations <- read_parquet('data/cross_correlations.parquet')
   - linear_models <- read_parquet('data/linear_models.parquet')
   - etc.

2. Use the leading/lagging indicators for:
   - Predictive modeling
   - Portfolio construction
   - Risk management

3. Apply the trend and seasonal models for:
   - Forecasting
   - Anomaly detection
   - Cyclical analysis
")

---

# Time-to-Event Survival Analysis for Stock Screening

**Predicting Time to Reach Price Targets with Kelly Optimal Sizing**

This section implements survival analysis to predict how long it takes for stocks to reach either:
- **Event 1 (Win)**: Price increases by X% (ATR-based threshold)
- **Event 2 (Loss)**: Price decreases by X% (ATR-based threshold)

## Methodology:
1. **ATR-Based Thresholds**: Use Average True Range to set adaptive, volatility-adjusted price targets
2. **Competing Risks Framework**: Model win/loss as competing events using Fine-Gray subdistribution hazards
3. **Multiple Model Approaches**: Cox PH, Random Survival Forest, and parametric AFT models
4. **Kelly Criterion Sizing**: Optimal bet sizing based on predicted win probabilities and expected returns
5. **Time-Series Cross-Validation**: Expanding window validation with proper temporal ordering

## Configuration Parameters

In [ ]:
# ============================================================================
# SURVIVAL ANALYSIS CONFIGURATION
# ============================================================================

# ATR-based threshold multipliers
ATR_MULTIPLIER_WIN <- 2.0    # Win threshold = ATR * multiplier
ATR_MULTIPLIER_LOSS <- 2.0   # Loss threshold = ATR * multiplier (symmetric by default)
ATR_PERIOD <- 14             # Period for ATR calculation

# Maximum observation window (censoring)
MAX_HOLDING_DAYS <- 60       # Right-censor if no event within this window

# Minimum data requirements
MIN_OBSERVATIONS <- 252      # Minimum trading days required per asset
MIN_EVENTS <- 20             # Minimum events required for model fitting

# Validation settings
VALIDATION_SPLIT <- 0.7      # Training proportion for time-series split
N_CV_FOLDS <- 5              # Number of expanding window folds

# Kelly sizing parameters
KELLY_FRACTION <- 0.25       # Use quarter-Kelly for risk management
MAX_POSITION_SIZE <- 0.10    # Maximum 10% of portfolio per position
MIN_WIN_PROB <- 0.45         # Minimum win probability to consider

# Technical indicator parameters
RSI_PERIOD <- 14
MACD_FAST <- 12
MACD_SLOW <- 26
MACD_SIGNAL <- 9
BB_PERIOD <- 20
BB_SD <- 2
SMA_SHORT <- 20
SMA_LONG <- 50
EMA_PERIOD <- 12

print("Survival analysis configuration loaded:")
print(glue("  ATR period: {ATR_PERIOD} days"))
print(glue("  Win threshold: {ATR_MULTIPLIER_WIN}x ATR"))
print(glue("  Loss threshold: {ATR_MULTIPLIER_LOSS}x ATR"))
print(glue("  Max holding period: {MAX_HOLDING_DAYS} days"))
print(glue("  Kelly fraction: {KELLY_FRACTION * 100}%"))

## Load OHLCV Data for Technical Analysis

In [ ]:
# Reconnect to database (it was closed in previous section for some runs)
if (!exists("con") || !dbIsValid(con)) {
  con <- dbConnect(duckdb::duckdb(), dbdir = DB_PATH, read_only = TRUE)
  print("Reconnected to database")
}

# Load Yahoo Finance OHLCV data
# The ETL stores adjusted close as 'value', we need to reconstruct OHLC from available data
# For this analysis, we'll use the close prices and estimate volatility from returns

print("Loading stock price data for survival analysis...")

# Get all Yahoo Finance series (stocks, ETFs, indices)
yahoo_series <- dbGetQuery(con, "
  SELECT DISTINCT series 
  FROM financial_data 
  WHERE origin = 'YAHOO'
  ORDER BY series
")$series

print(glue("Found {length(yahoo_series)} Yahoo Finance series"))

# Load price data for all Yahoo series
price_data <- dbGetQuery(con, "
  SELECT 
    series,
    date,
    value as close
  FROM financial_data 
  WHERE origin = 'YAHOO' 
    AND value IS NOT NULL
  ORDER BY series, date
")

price_dt <- as.data.table(price_data)
price_dt[, date := as.Date(date)]
price_dt[, close := as.numeric(close)]

# Calculate basic price metrics for each series
price_stats <- price_dt[, .(
  n_obs = .N,
  min_date = min(date),
  max_date = max(date),
  min_price = min(close, na.rm = TRUE),
  max_price = max(close, na.rm = TRUE)
), by = series]

# Filter to series with sufficient data
valid_stocks <- price_stats[n_obs >= MIN_OBSERVATIONS]$series
print(glue("Stocks with sufficient data (>= {MIN_OBSERVATIONS} obs): {length(valid_stocks)}"))

# Filter price data
stock_prices <- price_dt[series %in% valid_stocks]
print(glue("Total price observations for analysis: {nrow(stock_prices)}"))

## Technical Indicator Functions

In [ ]:
# ============================================================================
# TECHNICAL INDICATOR CALCULATION FUNCTIONS
# ============================================================================

#' Calculate ATR using close-to-close volatility (when OHLC not available)
#' @param close Vector of closing prices
#' @param n Period for ATR calculation
#' @return Vector of ATR values
calculate_atr_from_close <- function(close, n = 14) {
  # Use absolute returns as proxy for true range when only close is available
  returns <- c(NA, abs(diff(close)))
  atr <- TTR::EMA(returns, n = n)
  return(atr)
}

#' Calculate all technical indicators for a price series
#' @param data Data.table with date and close columns
#' @return Data.table with all technical indicators added
calculate_technical_indicators <- function(data) {
  
  data <- copy(data)
  setorder(data, date)
  
  close <- data$close
  n <- length(close)
  
  # Price returns
  data[, returns := c(NA, diff(log(close)))]
  data[, returns_pct := c(NA, diff(close) / head(close, -1))]
  
  # Volatility measures
  data[, volatility_20 := frollapply(returns, 20, sd, na.rm = TRUE)]
  data[, volatility_60 := frollapply(returns, 60, sd, na.rm = TRUE)]
  
  # ATR proxy (from close-to-close)
  data[, atr := calculate_atr_from_close(close, ATR_PERIOD)]
  data[, atr_pct := atr / close]  # ATR as percentage of price
  
  # Moving averages
  data[, sma_short := TTR::SMA(close, n = SMA_SHORT)]
  data[, sma_long := TTR::SMA(close, n = SMA_LONG)]
  data[, ema := TTR::EMA(close, n = EMA_PERIOD)]
  
  # Price relative to moving averages
  data[, price_sma_short_ratio := close / sma_short]
  data[, price_sma_long_ratio := close / sma_long]
  data[, sma_cross := sma_short / sma_long]  # Golden/Death cross indicator
  
  # RSI
  data[, rsi := TTR::RSI(close, n = RSI_PERIOD)]
  
  # MACD
  macd_result <- TTR::MACD(close, nFast = MACD_FAST, nSlow = MACD_SLOW, nSig = MACD_SIGNAL)
  data[, macd := macd_result[, "macd"]]
  data[, macd_signal := macd_result[, "signal"]]
  data[, macd_histogram := macd - macd_signal]
  
  # Bollinger Bands
  bb <- TTR::BBands(close, n = BB_PERIOD, sd = BB_SD)
  data[, bb_upper := bb[, "up"]]
  data[, bb_lower := bb[, "dn"]]
  data[, bb_middle := bb[, "mavg"]]
  data[, bb_pct := bb[, "pctB"]]  # Percent B (where price is within bands)
  data[, bb_width := (bb_upper - bb_lower) / bb_middle]  # Band width
  
  # Momentum indicators
  data[, roc_10 := TTR::ROC(close, n = 10)]  # Rate of change
  data[, roc_20 := TTR::ROC(close, n = 20)]
  data[, momentum := TTR::momentum(close, n = 10)]
  
  # Price position indicators
  data[, high_20 := frollapply(close, 20, max, na.rm = TRUE)]
  data[, low_20 := frollapply(close, 20, min, na.rm = TRUE)]
  data[, price_position_20 := (close - low_20) / (high_20 - low_20)]  # 0 = at low, 1 = at high
  
  data[, high_60 := frollapply(close, 60, max, na.rm = TRUE)]
  data[, low_60 := frollapply(close, 60, min, na.rm = TRUE)]
  data[, price_position_60 := (close - low_60) / (high_60 - low_60)]
  
  # Trend strength
  data[, adx := {
    # Simplified ADX using price momentum as proxy
    abs_momentum <- abs(momentum)
    TTR::EMA(abs_momentum, n = 14)
  }]
  
  # Mean reversion indicator
  data[, zscore_20 := (close - sma_short) / (volatility_20 * close)]
  
  return(data)
}

print("Technical indicator functions defined successfully!")

## Time-to-Event Target Generation

In [ ]:
# ============================================================================
# TIME-TO-EVENT TARGET GENERATION WITH COMPETING RISKS
# ============================================================================

#' Generate survival outcomes for each observation
#' Uses ATR-based thresholds to define win/loss events
#' 
#' @param data Data.table with date, close, and atr columns
#' @param atr_mult_win Multiplier for win threshold (positive)
#' @param atr_mult_loss Multiplier for loss threshold (positive)
#' @param max_days Maximum days to look forward (censoring point)
#' @return Data.table with survival outcomes added
generate_survival_outcomes <- function(data, 
                                       atr_mult_win = ATR_MULTIPLIER_WIN,
                                       atr_mult_loss = ATR_MULTIPLIER_LOSS,
                                       max_days = MAX_HOLDING_DAYS) {
  
  data <- copy(data)
  setorder(data, date)
  n <- nrow(data)
  
  # Initialize outcome columns
  data[, `:=`(
    time_to_event = as.numeric(NA),
    event_type = as.integer(NA),     # 0 = censored, 1 = win, 2 = loss
    win_threshold = as.numeric(NA),
    loss_threshold = as.numeric(NA),
    final_return = as.numeric(NA)
  )]
  
  # Calculate thresholds based on ATR at each point
  data[, win_threshold := close + (atr * atr_mult_win)]
  data[, loss_threshold := close - (atr * atr_mult_loss)]
  
  # For each observation, look forward to find first event
  for (i in 1:(n - 1)) {
    entry_price <- data$close[i]
    win_target <- data$win_threshold[i]
    loss_target <- data$loss_threshold[i]
    
    # Skip if thresholds are invalid
    if (is.na(win_target) || is.na(loss_target)) next
    
    # Look forward up to max_days
    max_j <- min(i + max_days, n)
    
    time_to_win <- NA
    time_to_loss <- NA
    
    for (j in (i + 1):max_j) {
      future_price <- data$close[j]
      days_forward <- j - i
      
      # Check for win (price >= win threshold)
      if (is.na(time_to_win) && future_price >= win_target) {
        time_to_win <- days_forward
      }
      
      # Check for loss (price <= loss threshold)
      if (is.na(time_to_loss) && future_price <= loss_target) {
        time_to_loss <- days_forward
      }
      
      # If both events found, we know which came first
      if (!is.na(time_to_win) && !is.na(time_to_loss)) break
    }
    
    # Determine outcome based on which event occurred first
    if (!is.na(time_to_win) && !is.na(time_to_loss)) {
      # Both events occurred - take the first one
      if (time_to_win <= time_to_loss) {
        data$time_to_event[i] <- time_to_win
        data$event_type[i] <- 1L  # Win
      } else {
        data$time_to_event[i] <- time_to_loss
        data$event_type[i] <- 2L  # Loss
      }
    } else if (!is.na(time_to_win)) {
      # Only win occurred
      data$time_to_event[i] <- time_to_win
      data$event_type[i] <- 1L
    } else if (!is.na(time_to_loss)) {
      # Only loss occurred
      data$time_to_event[i] <- time_to_loss
      data$event_type[i] <- 2L
    } else {
      # Neither event occurred - censored
      data$time_to_event[i] <- max_days
      data$event_type[i] <- 0L
    }
    
    # Calculate actual return at event time
    event_idx <- i + data$time_to_event[i]
    if (event_idx <= n) {
      data$final_return[i] <- (data$close[event_idx] - entry_price) / entry_price
    }
  }
  
  return(data)
}

print("Survival outcome generation function defined!")

## Calculate Technical Indicators and Generate Targets

In [ ]:
print("Processing stocks for survival analysis...")

# Process each stock: calculate indicators and generate survival outcomes
survival_data_list <- list()

pb <- progress_bar$new(
  total = length(valid_stocks),
  format = "[:bar] :percent :current/:total ETA: :eta - :what"
)

for (stock in valid_stocks) {
  pb$tick(tokens = list(what = stock))
  
  # Get price data for this stock
  stock_data <- stock_prices[series == stock, .(date, close)]
  
  if (nrow(stock_data) < MIN_OBSERVATIONS) next
  
  tryCatch({
    # Calculate technical indicators
    stock_data <- calculate_technical_indicators(stock_data)
    
    # Generate survival outcomes
    stock_data <- generate_survival_outcomes(stock_data)
    
    # Add stock identifier
    stock_data[, symbol := stock]
    
    survival_data_list[[stock]] <- stock_data
    
  }, error = function(e) {
    # Skip stocks with calculation errors
    NULL
  })
}

# Combine all stock data
if (length(survival_data_list) > 0) {
  survival_df <- rbindlist(survival_data_list, fill = TRUE)
  print(glue("\nProcessed {length(survival_data_list)} stocks"))
  print(glue("Total observations: {nrow(survival_df)}"))
} else {
  stop("No stocks processed successfully!")
}

# Remove rows with missing survival outcomes
survival_complete <- survival_df[!is.na(time_to_event) & !is.na(event_type)]
print(glue("Complete observations for modeling: {nrow(survival_complete)}"))

In [ ]:
# Summary of survival outcomes
print("\n=== SURVIVAL OUTCOMES SUMMARY ===")

event_summary <- survival_complete[, .(
  count = .N,
  pct = round(.N / nrow(survival_complete) * 100, 1)
), by = event_type][order(event_type)]

event_labels <- c("0" = "Censored (no event)", "1" = "Win (target hit)", "2" = "Loss (stop hit)")
event_summary[, event_label := event_labels[as.character(event_type)]]

print(event_summary[, .(event_label, count, pct)])

# Time to event statistics by outcome
time_stats <- survival_complete[, .(
  median_time = median(time_to_event),
  mean_time = round(mean(time_to_event), 1),
  min_time = min(time_to_event),
  max_time = max(time_to_event)
), by = event_type][order(event_type)]

time_stats[, event_label := event_labels[as.character(event_type)]]
print("\nTime-to-event statistics by outcome:")
print(time_stats[, .(event_label, median_time, mean_time, min_time, max_time)])

# Summary by stock
stock_summary <- survival_complete[, .(
  n_obs = .N,
  n_wins = sum(event_type == 1),
  n_losses = sum(event_type == 2),
  n_censored = sum(event_type == 0),
  win_rate = round(sum(event_type == 1) / sum(event_type %in% c(1, 2)) * 100, 1),
  median_time = median(time_to_event)
), by = symbol][order(-win_rate)]

print("\nTop 10 stocks by win rate:")
print(head(stock_summary, 10))

print("\nBottom 10 stocks by win rate:")
print(tail(stock_summary, 10))

## Add Macro Indicator Features

In [ ]:
# Load macro indicators and merge with survival data
print("Adding macro indicator features...")

# Get key macro indicators from FRED
macro_series <- c(
  "DFF",           # Federal Funds Rate
  "DGS10",         # 10-Year Treasury
  "DGS2",          # 2-Year Treasury  
  "UNRATE",        # Unemployment Rate
  "CPIAUCSL",      # CPI
  "VIXCLS"         # VIX (if available)
)

macro_data <- dbGetQuery(con, paste0("
  SELECT 
    series,
    date,
    value
  FROM financial_data 
  WHERE origin = 'FRED' 
    AND series IN ('", paste(macro_series, collapse = "','"), "')
    AND value IS NOT NULL
  ORDER BY date
"))

macro_dt <- as.data.table(macro_data)
macro_dt[, date := as.Date(date)]

# Pivot to wide format
macro_wide <- dcast(macro_dt, date ~ series, value.var = "value")

# Forward fill missing values (LOCF)
macro_cols <- setdiff(names(macro_wide), "date")
for (col in macro_cols) {
  macro_wide[, (col) := nafill(get(col), type = "locf")]
}

# Calculate derived macro features
if ("DGS10" %in% names(macro_wide) && "DGS2" %in% names(macro_wide)) {
  macro_wide[, yield_spread := DGS10 - DGS2]  # Yield curve slope
}

if ("DFF" %in% names(macro_wide)) {
  macro_wide[, dff_change_20d := DFF - shift(DFF, 20)]  # Fed rate momentum
}

# Merge macro data with survival data
survival_with_macro <- merge(
  survival_complete,
  macro_wide,
  by = "date",
  all.x = TRUE
)

print(glue("Macro features added. Final dataset: {nrow(survival_with_macro)} rows"))
print(glue("Features available: {ncol(survival_with_macro)} columns"))

## Prepare Modeling Dataset

In [ ]:
# Define feature columns for modeling
technical_features <- c(
  "returns", "volatility_20", "volatility_60", "atr_pct",
  "price_sma_short_ratio", "price_sma_long_ratio", "sma_cross",
  "rsi", "macd", "macd_signal", "macd_histogram",
  "bb_pct", "bb_width",
  "roc_10", "roc_20", "momentum",
  "price_position_20", "price_position_60",
  "zscore_20"
)

macro_features <- intersect(
  c("DFF", "DGS10", "DGS2", "UNRATE", "yield_spread", "dff_change_20d"),
  names(survival_with_macro)
)

all_features <- c(technical_features, macro_features)

# Filter to features that exist
available_features <- intersect(all_features, names(survival_with_macro))
print(glue("Available features for modeling: {length(available_features)}"))

# Create modeling dataset with complete cases for key features
key_features <- c("time_to_event", "event_type", available_features)
model_df <- survival_with_macro[, .SD, .SDcols = c("symbol", "date", key_features)]

# Remove rows with any NA in key features
model_df <- na.omit(model_df)

print(glue("Modeling dataset after removing NAs: {nrow(model_df)} observations"))

# Time-based train/test split
# Convert date to numeric for quantile calculation, then back to Date
split_date <- as.Date(quantile(as.numeric(model_df$date), VALIDATION_SPLIT), origin = "1970-01-01")
train_df <- model_df[date < split_date]
test_df <- model_df[date >= split_date]

print(glue("\nTrain set: {nrow(train_df)} obs ({min(train_df$date)} to {max(train_df$date)})"))
print(glue("Test set: {nrow(test_df)} obs ({min(test_df$date)} to {max(test_df$date)})"))

# Event distribution in train/test
print("\nEvent distribution in training set:")
print(train_df[, .N, by = event_type][order(event_type)])

print("\nEvent distribution in test set:")
print(test_df[, .N, by = event_type][order(event_type)])

## Competing Risks Survival Models

### Model 1: Fine-Gray Subdistribution Hazard Model

The Fine-Gray model accounts for competing risks by modeling the subdistribution hazard, which gives the instantaneous risk of experiencing the event of interest given that the subject has not yet experienced that event (but may have experienced a competing event).

In [ ]:
# ============================================================================
# MODEL 1: FINE-GRAY COMPETING RISKS MODEL
# ============================================================================
print("Fitting Fine-Gray competing risks model...")

# Prepare data for cmprsk
# Requires: ftime (failure time), fstatus (0=censor, 1=event1, 2=event2, etc.)
train_ftime <- train_df$time_to_event
train_fstatus <- train_df$event_type

# Create covariate matrix (only numeric features)
train_covariates <- as.matrix(train_df[, ..available_features])

# Handle any remaining NA/Inf in covariates
train_covariates[is.na(train_covariates)] <- 0
train_covariates[is.infinite(train_covariates)] <- 0

# Fit Fine-Gray model for WIN (event_type = 1)
print("Fitting model for WIN events (event = 1)...")
fg_win <- tryCatch({
  crr(
    ftime = train_ftime,
    fstatus = train_fstatus,
    cov1 = train_covariates,
    failcode = 1,
    cencode = 0
  )
}, error = function(e) {
  print(glue("Fine-Gray WIN model error: {e$message}"))
  NULL
})

# Fit Fine-Gray model for LOSS (event_type = 2)
print("Fitting model for LOSS events (event = 2)...")
fg_loss <- tryCatch({
  crr(
    ftime = train_ftime,
    fstatus = train_fstatus,
    cov1 = train_covariates,
    failcode = 2,
    cencode = 0
  )
}, error = function(e) {
  print(glue("Fine-Gray LOSS model error: {e$message}"))
  NULL
})

# Display model summaries
if (!is.null(fg_win)) {
  print("\n=== FINE-GRAY MODEL: WIN (Target Hit) ===")
  print(summary(fg_win))
}

if (!is.null(fg_loss)) {
  print("\n=== FINE-GRAY MODEL: LOSS (Stop Hit) ===")
  print(summary(fg_loss))
}

In [ ]:
# Extract and visualize important coefficients
if (!is.null(fg_win)) {
  # Get coefficient summary
  coef_win <- data.table(
    feature = available_features,
    coef = fg_win$coef,
    se = sqrt(diag(fg_win$var)),
    hr = exp(fg_win$coef)  # Subdistribution hazard ratio
  )
  coef_win[, z := coef / se]
  coef_win[, pvalue := 2 * pnorm(-abs(z))]
  coef_win[, significance := ifelse(pvalue < 0.001, "***", 
                                    ifelse(pvalue < 0.01, "**",
                                           ifelse(pvalue < 0.05, "*", "")))]
  
  # Sort by absolute coefficient
  coef_win <- coef_win[order(-abs(coef))]
  
  print("\n=== TOP 10 PREDICTORS FOR WIN (by absolute coefficient) ===")
  print(head(coef_win[, .(feature, coef = round(coef, 4), hr = round(hr, 4), 
                          pvalue = round(pvalue, 4), significance)], 10))
  
  # Plot coefficients
  top_coefs <- head(coef_win, 15)
  p_coef <- ggplot(top_coefs, aes(x = reorder(feature, coef), y = coef, fill = coef > 0)) +
    geom_col() +
    coord_flip() +
    scale_fill_manual(values = c("#E74C3C", "#2ECC71"), labels = c("Decreases Win", "Increases Win")) +
    labs(
      title = "Fine-Gray Model: Top Predictors for WIN",
      x = "Feature",
      y = "Coefficient (log subdistribution hazard ratio)",
      fill = "Effect"
    ) +
    theme_minimal() +
    theme(legend.position = "bottom")
  
  print(p_coef)
}

In [ ]:
# Add Fine-Gray coefficient plot to report
if (exists("p_coef")) {
  add_report_item("plot", p_coef, "Fine-Gray Subdistribution Hazard Coefficients", "2. Survival Analysis")
}

# Add coefficient table to report
if (exists("coef_win") && nrow(coef_win) > 0) {
  add_report_item("table", as.data.frame(coef_win), "Fine-Gray Model Coefficients (Win)", "2. Survival Analysis")
}

### Model 2: Cox Proportional Hazards with Cause-Specific Hazards

Alternative approach: fit separate Cox models for each event type, treating competing events as censored.

In [ ]:
# ============================================================================
# MODEL 2: CAUSE-SPECIFIC COX PROPORTIONAL HAZARDS
# ============================================================================
print("Fitting cause-specific Cox PH models...")

# Create formula with available features (limit to avoid overfitting)
# Select top features based on Fine-Gray results or use all if model didn't fit
if (exists("coef_win") && nrow(coef_win) > 0) {
  # Use top features by significance
  top_features <- head(coef_win[order(pvalue)]$feature, 12)
} else {
  top_features <- head(available_features, 12)
}

formula_str <- paste("Surv(time_to_event, event_status) ~", paste(top_features, collapse = " + "))

# Cox model for WIN (event_type == 1, treat losses as censored)
train_cox_win <- copy(train_df)
train_cox_win[, event_status := as.integer(event_type == 1)]

cox_win <- tryCatch({
  coxph(as.formula(formula_str), data = train_cox_win)
}, error = function(e) {
  print(glue("Cox WIN model error: {e$message}"))
  NULL
})

# Cox model for LOSS (event_type == 2, treat wins as censored)
train_cox_loss <- copy(train_df)
train_cox_loss[, event_status := as.integer(event_type == 2)]

cox_loss <- tryCatch({
  coxph(as.formula(formula_str), data = train_cox_loss)
}, error = function(e) {
  print(glue("Cox LOSS model error: {e$message}"))
  NULL
})

# Display summaries
if (!is.null(cox_win)) {
  print("\n=== CAUSE-SPECIFIC COX MODEL: WIN ===")
  print(summary(cox_win))
}

if (!is.null(cox_loss)) {
  print("\n=== CAUSE-SPECIFIC COX MODEL: LOSS ===")
  print(summary(cox_loss))
}

In [ ]:
# Forest plot for Cox model hazard ratios
if (!is.null(cox_win)) {
  cox_coefs <- broom::tidy(cox_win, exponentiate = TRUE, conf.int = TRUE)
  cox_coefs <- as.data.table(cox_coefs)
  cox_coefs <- cox_coefs[order(-abs(log(estimate)))]
  
  p_forest <- ggplot(cox_coefs, aes(x = estimate, y = reorder(term, estimate))) +
    geom_point(size = 3) +
    geom_errorbarh(aes(xmin = conf.low, xmax = conf.high), height = 0.2) +
    geom_vline(xintercept = 1, linetype = "dashed", color = "red") +
    scale_x_log10() +
    labs(
      title = "Cox PH Model: Hazard Ratios for WIN",
      subtitle = "HR > 1 means faster time to win",
      x = "Hazard Ratio (log scale)",
      y = "Feature"
    ) +
    theme_minimal()
  
  print(p_forest)
}

In [ ]:
# Add Cox forest plot to report
if (exists("p_forest")) {
  add_report_item("plot", p_forest, "Cox Proportional Hazards - Hazard Ratios", "2. Survival Analysis")
}

### Model 3: Random Survival Forest

Non-parametric approach that can capture non-linear relationships and interactions between features.

In [ ]:
# ============================================================================
# MODEL 3: RANDOM SURVIVAL FOREST FOR COMPETING RISKS
# ============================================================================
print("Fitting Random Survival Forest with competing risks...")

# Prepare data for RSF
# Need to subsample for computational efficiency
set.seed(42)
sample_size <- min(10000, nrow(train_df))
train_sample_idx <- sample(1:nrow(train_df), sample_size)
train_sample <- train_df[train_sample_idx]

# Create formula
rsf_formula <- as.formula(paste(
  "Surv(time_to_event, event_type) ~",
  paste(top_features, collapse = " + ")
))

# Fit RSF with competing risks
rsf_model <- tryCatch({
  rfsrc(
    rsf_formula,
    data = as.data.frame(train_sample),
    ntree = 100,
    nodesize = 15,
    splitrule = "logrankCR",  # Competing risks splitting
    importance = TRUE,
    seed = 42
  )
}, error = function(e) {
  print(glue("RSF error: {e$message}"))
  NULL
})

if (!is.null(rsf_model)) {
  print("\n=== RANDOM SURVIVAL FOREST SUMMARY ===")
  print(rsf_model)
}

In [ ]:
# Variable importance from RSF
if (!is.null(rsf_model) && !is.null(rsf_model$importance)) {
  
  # Extract importance (VIMP)
  vimp_vec <- rsf_model$importance
  
  # Get feature names from the model
  if (is.null(names(vimp_vec))) {
    feature_names <- rsf_model$xvar.names
  } else {
    feature_names <- names(vimp_vec)
  }
  
  vimp_df <- data.frame(
    feature = feature_names,
    vimp_score = as.numeric(vimp_vec),
    stringsAsFactors = FALSE
  )
  vimp_df <- vimp_df[order(-vimp_df$vimp_score), ]
  
  print("\n=== RSF VARIABLE IMPORTANCE ===")
  print(vimp_df)
  
  # Plot importance
  p_vimp <- ggplot(data = vimp_df) +
    geom_col(aes(x = reorder(feature, vimp_score), y = vimp_score, 
                 fill = vimp_score > 0)) +
    coord_flip() +
    scale_fill_manual(values = c("#95A5A6", "#3498DB"), guide = "none") +
    labs(
      title = "Random Survival Forest: Variable Importance",
      subtitle = "Positive = reduces prediction error when included",
      x = "Feature",
      y = "Variable Importance (VIMP)"
    ) +
    theme_minimal()
  
  print(p_vimp)
}

In [ ]:
# Add RSF variable importance plot to report
if (exists("p_vimp")) {
  add_report_item("plot", p_vimp, "Random Survival Forest - Variable Importance", "2. Survival Analysis")
}

## Prediction Functions and Win Probability Calculation

In [ ]:
# ============================================================================
# PREDICTION FUNCTIONS
# ============================================================================

#' Calculate cumulative incidence (win probability) from Fine-Gray model
#' @param model Fine-Gray model object
#' @param newdata New covariate matrix
#' @param times Time points at which to evaluate CIF
#' @return Matrix of cumulative incidence probabilities
predict_cif_finegray <- function(model, newdata, times = c(10, 20, 30, 60)) {
  if (is.null(model)) return(NULL)
  
  tryCatch({
    # Predict CIF
    pred <- predict(model, newdata)
    
    # Find indices closest to requested times
    time_idx <- sapply(times, function(t) which.min(abs(pred[[1]]$time - t)))
    
    # Extract CIF at those times
    cif_matrix <- sapply(pred, function(p) p$"1"[time_idx])
    colnames(cif_matrix) <- paste0("cif_", times)
    
    return(cif_matrix)
  }, error = function(e) {
    return(NULL)
  })
}

#' Calculate win probability from RSF model
#' @param model RSF model object  
#' @param newdata New data for prediction
#' @param horizon Time horizon for prediction
#' @return Data.table with win/loss probabilities
predict_rsf <- function(model, newdata, horizon = 30) {
  if (is.null(model)) return(NULL)
  
  tryCatch({
    pred <- predict(model, newdata = as.data.frame(newdata))
    
    # Extract cumulative incidence functions
    # Event 1 = Win, Event 2 = Loss
    time_idx <- which.min(abs(pred$time.interest - horizon))
    
    # CIF for each event at horizon
    cif_win <- pred$cif[, time_idx, 1]   # CIF for event 1
    cif_loss <- pred$cif[, time_idx, 2]  # CIF for event 2
    
    result <- data.table(
      prob_win = cif_win,
      prob_loss = cif_loss,
      prob_censored = 1 - cif_win - cif_loss,
      win_loss_ratio = cif_win / (cif_win + cif_loss)
    )
    
    return(result)
  }, error = function(e) {
    print(glue("RSF prediction error: {e$message}"))
    return(NULL)
  })
}

print("Prediction functions defined!")

## Model Validation

In [ ]:
# ============================================================================
# MODEL VALIDATION ON TEST SET
# ============================================================================
print("Validating models on test set...")

# Prepare test covariates
test_covariates <- as.matrix(test_df[, ..available_features])
test_covariates[is.na(test_covariates)] <- 0
test_covariates[is.infinite(test_covariates)] <- 0

# Validation metrics storage
validation_results <- list()

# 1. Cox model validation (C-index)
if (!is.null(cox_win)) {
  test_cox_win <- copy(test_df)
  test_cox_win[, event_status := as.integer(event_type == 1)]
  
  # Predict on test set
  cox_pred <- predict(cox_win, newdata = test_cox_win, type = "risk")
  
  # Calculate concordance index
  surv_obj <- Surv(test_cox_win$time_to_event, test_cox_win$event_status)
  c_index <- survConcordance(surv_obj ~ cox_pred)$concordance
  
  validation_results$cox_win <- list(
    model = "Cox PH (Win)",
    c_index = c_index
  )
  
  print(glue("\nCox PH (Win) - C-index on test set: {round(c_index, 4)}"))
}

# 2. RSF validation
if (!is.null(rsf_model)) {
  # Subsample test set for prediction speed
  test_sample_size <- min(5000, nrow(test_df))
  test_sample_idx <- sample(1:nrow(test_df), test_sample_size)
  test_sample <- test_df[test_sample_idx]
  
  rsf_pred <- predict(rsf_model, newdata = as.data.frame(test_sample))
  
  # Calculate error rate
  rsf_error <- rsf_pred$err.rate[rsf_model$ntree]
  
  validation_results$rsf <- list(
    model = "Random Survival Forest",
    error_rate = rsf_error
  )
  
  print(glue("RSF - OOB Error Rate: {round(rsf_error, 4)}"))
}

# 3. Calibration: Compare predicted vs observed win rates by decile
if (!is.null(rsf_model)) {
  print("\nCalibration analysis...")
  
  # Get predictions for test sample
  rsf_probs <- predict_rsf(rsf_model, test_sample, horizon = MAX_HOLDING_DAYS)
  
  if (!is.null(rsf_probs)) {
    test_sample[, pred_win_prob := rsf_probs$win_loss_ratio]
    
    # Create deciles
    test_sample[, prob_decile := cut(pred_win_prob, 
                                      breaks = quantile(pred_win_prob, probs = seq(0, 1, 0.1), na.rm = TRUE),
                                      labels = 1:10, include.lowest = TRUE)]
    
    # Calculate observed win rate by decile
    calibration <- test_sample[!is.na(prob_decile), .(
      predicted_win_rate = mean(pred_win_prob, na.rm = TRUE),
      observed_win_rate = mean(event_type == 1),
      n_obs = .N
    ), by = prob_decile][order(prob_decile)]
    
    print("Calibration by probability decile:")
    print(calibration)
    
    # Calibration plot
    p_calib <- ggplot(calibration, aes(x = predicted_win_rate, y = observed_win_rate)) +
      geom_point(aes(size = n_obs), color = "#3498DB") +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_size_continuous(range = c(2, 8)) +
      labs(
        title = "Model Calibration Plot",
        subtitle = "Predicted vs Observed Win Rate by Decile",
        x = "Predicted Win Probability",
        y = "Observed Win Rate",
        size = "N Obs"
      ) +
      theme_minimal() +
      coord_equal(xlim = c(0, 1), ylim = c(0, 1))
    
    print(p_calib)
  }
}

In [ ]:
# Add calibration plot to report
if (exists("p_calib")) {
  add_report_item("plot", p_calib, "Model Calibration - Predicted vs Observed", "2. Survival Analysis")
}

### Time-Dependent AUC and Calibration Metrics
Evaluate model discrimination over time and calibration quality.

In [ ]:
# ============================================================================
# TIME-DEPENDENT AUC AND CALIBRATION METRICS
# ============================================================================
print("Calculating time-dependent discrimination metrics...")

# Define evaluation time points
eval_times <- c(5, 10, 15, 20, 30, 45, 60)

# Prepare test data for riskRegression
test_eval <- copy(test_df)
test_eval[, event_status := factor(event_type, levels = c(0, 1, 2), labels = c("censored", "win", "loss"))]

# Calculate time-dependent AUC for Cox model (WIN outcome)
if (!is.null(cox_win)) {
  tryCatch({
    # Create survival object for win (event=1)
    test_cox_eval <- copy(test_eval)
    test_cox_eval[, event_binary := as.integer(event_type == 1)]
    
    # Score the model at multiple time points
    auc_results <- data.table(
      time = eval_times,
      auc = NA_real_,
      auc_lower = NA_real_,
      auc_upper = NA_real_
    )
    
    for (t_idx in seq_along(eval_times)) {
      t <- eval_times[t_idx]
      
      # Get predicted risk at time t
      surv_prob <- summary(survfit(cox_win, newdata = test_cox_eval), times = t)$surv
      if (length(surv_prob) > 0) {
        pred_risk <- 1 - surv_prob
      } else {
        next
      }
      
      # Calculate AUC at time t using those with observed outcome by time t
      observed_by_t <- test_cox_eval[time_to_event <= t | event_binary == 0]
      
      if (nrow(observed_by_t) > 50) {
        # Create ROC-like calculation
        wins_by_t <- observed_by_t[event_binary == 1 & time_to_event <= t]
        others_by_t <- observed_by_t[!(event_binary == 1 & time_to_event <= t)]
        
        if (nrow(wins_by_t) > 10 && nrow(others_by_t) > 10) {
          # Use Wilcoxon to estimate AUC
          pred_risk_sub <- 1 - summary(survfit(cox_win, newdata = observed_by_t), times = t)$surv
          if (length(pred_risk_sub) == nrow(observed_by_t)) {
            observed_by_t[, pred_risk := pred_risk_sub]
            
            wilcox_test <- wilcox.test(
              observed_by_t[event_binary == 1 & time_to_event <= t]$pred_risk,
              observed_by_t[!(event_binary == 1 & time_to_event <= t)]$pred_risk
            )
            auc_results$auc[t_idx] <- wilcox_test$statistic / 
              (nrow(wins_by_t) * nrow(others_by_t))
          }
        }
      }
    }
    
    auc_results <- auc_results[!is.na(auc)]
    
    if (nrow(auc_results) > 0) {
      print("\n=== TIME-DEPENDENT AUC (Cox PH - Win) ===")
      print(auc_results)
      
      # Plot time-dependent AUC
      p_auc <- ggplot(auc_results, aes(x = time, y = auc)) +
        geom_line(color = "#3498DB", size = 1.2) +
        geom_point(color = "#2980B9", size = 3) +
        geom_hline(yintercept = 0.5, linetype = "dashed", color = "gray50") +
        geom_hline(yintercept = 0.7, linetype = "dotted", color = "#27AE60") +
        scale_y_continuous(limits = c(0.4, 1), breaks = seq(0.4, 1, 0.1)) +
        labs(
          title = "Time-Dependent AUC for Win Prediction",
          subtitle = "Cox PH Model - AUC at different holding periods",
          x = "Days from Entry",
          y = "AUC",
          caption = "Dashed line = random (0.5), Dotted line = good (0.7)"
        ) +
        theme_minimal()
      
      print(p_auc)
    }
    
  }, error = function(e) {
    print(glue("Time-dependent AUC calculation error: {e$message}"))
  })
}

# Brier Score calculation for calibration
print("\nCalculating Brier scores...")
if (!is.null(rsf_model)) {
  tryCatch({
    # Sample for efficiency
    brier_sample_size <- min(2000, nrow(test_df))
    brier_sample <- test_df[sample(1:nrow(test_df), brier_sample_size)]
    
    # Get predictions at evaluation times
    rsf_pred <- predict(rsf_model, newdata = as.data.frame(brier_sample))
    
    brier_scores <- data.table(time = numeric(), brier_score = numeric())
    
    for (t in eval_times) {
      time_idx <- which.min(abs(rsf_pred$time.interest - t))
      
      # Predicted CIF for win at time t
      pred_cif_win <- rsf_pred$cif[, time_idx, 1]
      
      # Observed outcome by time t
      observed_win <- as.integer(brier_sample$event_type == 1 & brier_sample$time_to_event <= t)
      censored_before_t <- as.integer(brier_sample$event_type == 0 & brier_sample$time_to_event < t)
      
      # Only evaluate on non-censored observations
      valid_idx <- censored_before_t == 0
      
      if (sum(valid_idx) > 50) {
        brier <- mean((pred_cif_win[valid_idx] - observed_win[valid_idx])^2)
        brier_scores <- rbind(brier_scores, data.table(time = t, brier_score = brier))
      }
    }
    
    if (nrow(brier_scores) > 0) {
      print("\n=== BRIER SCORES BY TIME (RSF) ===")
      print(brier_scores)
      
      # Integrated Brier Score (approximate)
      ibs <- mean(brier_scores$brier_score)
      print(glue("\nIntegrated Brier Score (approx): {round(ibs, 4)}"))
      print("Interpretation: Lower is better. <0.25 is reasonable, <0.1 is good.")
    }
    
  }, error = function(e) {
    print(glue("Brier score calculation error: {e$message}"))
  })
}

### Spline Effect Plots
Visualize non-linear effects of key predictors using restricted cubic splines.

In [ ]:
# ============================================================================
# SPLINE EFFECT PLOTS - Non-linear variable effects
# ============================================================================
print("Creating spline effect plots for key predictors...")

# Use mgcv package for GAM-based splines (more compatible)
if (!require("mgcv", character.only = TRUE, quietly = TRUE)) {
  install.packages("mgcv")
  library(mgcv)
}

# Key variables to plot with splines
spline_vars <- c("rsi", "volatility_20", "bb_pct", "macd", "atr_pct", "zscore_20")
spline_vars <- intersect(spline_vars, available_features)

if (length(spline_vars) >= 3) {
  
  # Prepare data for GAM
  train_gam <- copy(train_df)
  train_gam[, event_win := as.integer(event_type == 1)]
  
  # Build formula with smooth terms
  smooth_terms <- paste0("s(", spline_vars, ", k=4)")
  gam_formula <- as.formula(paste(
    "event_win ~",
    paste(smooth_terms, collapse = " + ")
  ))
  
  # Fit GAM with binomial family (for probability of win)
  gam_model <- tryCatch({
    gam(gam_formula, data = train_gam, family = binomial(), method = "REML")
  }, error = function(e) {
    print(glue("GAM model error: {e$message}"))
    NULL
  })
  
  if (!is.null(gam_model)) {
    print("\n=== GAM MODEL WITH SPLINES ===")
    print(summary(gam_model))
    
    # Create effect plots for each variable
    spline_plots <- list()
    
    for (var in spline_vars) {
      tryCatch({
        # Create prediction grid
        var_range <- range(train_gam[[var]], na.rm = TRUE)
        pred_data <- data.table(x = seq(var_range[1], var_range[2], length.out = 100))
        
        # Set other variables to median
        for (v in setdiff(spline_vars, var)) {
          pred_data[[v]] <- median(train_gam[[v]], na.rm = TRUE)
        }
        setnames(pred_data, "x", var)
        
        # Predict
        pred_data[, pred := predict(gam_model, newdata = pred_data, type = "response")]
        pred_data[, se := predict(gam_model, newdata = pred_data, type = "response", se.fit = TRUE)$se.fit]
        pred_data[, lower := pred - 1.96 * se]
        pred_data[, upper := pred + 1.96 * se]
        
        # Create plot
        p <- ggplot(pred_data, aes(x = get(var), y = pred)) +
          geom_ribbon(aes(ymin = lower, ymax = upper), fill = "#3498DB", alpha = 0.2) +
          geom_line(color = "#2980B9", size = 1.2) +
          geom_hline(yintercept = 0.5, linetype = "dashed", color = "gray50") +
          labs(
            title = paste("Effect of", var, "on Win Probability"),
            subtitle = "GAM smooth (k=4 knots)",
            x = var,
            y = "Predicted Win Probability"
          ) +
          theme_minimal()
        
        spline_plots[[var]] <- p
        
      }, error = function(e) {
        print(glue("Could not create effect plot for {var}: {e$message}"))
      })
    }
    
    # Print plots
    if (length(spline_plots) > 0) {
      print("\n=== SPLINE EFFECT PLOTS ===")
      for (var in names(spline_plots)) {
        print(spline_plots[[var]])
      }
    }
  }
  
} else {
  print("Not enough spline variables available for effect plots")
}

# RSF Partial Dependence Plots
if (!is.null(rsf_model)) {
  print("\n=== RSF PARTIAL DEPENDENCE PLOTS ===")
  
  # Select top variables by importance
  if (!is.null(rsf_model$importance)) {
    vimp_sorted <- sort(rsf_model$importance, decreasing = TRUE)
    top_pdp_vars <- names(head(vimp_sorted[vimp_sorted > 0], 4))
    
    for (var in top_pdp_vars) {
      tryCatch({
        # Calculate partial dependence
        pdp <- plot.variable(rsf_model, xvar.names = var, partial = TRUE, 
                             which.outcome = 1, plots = FALSE)
        
        # Extract data
        pdp_data <- data.table(
          x = pdp$pData[[1]]$x.uniq,
          yhat = pdp$pData[[1]]$yhat
        )
        
        p_pdp <- ggplot(pdp_data, aes(x = x, y = yhat)) +
          geom_line(color = "#E74C3C", size = 1.2) +
          labs(
            title = paste("Partial Dependence:", var),
            subtitle = "RSF - Effect on Win CIF",
            x = var,
            y = "Partial Effect on Win Probability"
          ) +
          theme_minimal()
        
        print(p_pdp)
        
      }, error = function(e) {
        # Silently skip if PDP fails
      })
    }
  }
}

## Stock Screening: Current Candidates

In [ ]:
# ============================================================================
# STOCK SCREENING: IDENTIFY CURRENT BEST CANDIDATES
# ============================================================================
print("Screening stocks for current trading candidates...")

# Get the most recent observation for each stock
latest_data <- survival_with_macro[, .SD[which.max(date)], by = symbol]

# Only keep stocks with recent data (within last 7 days of max date in data)
max_date <- max(latest_data$date, na.rm = TRUE)
latest_data <- latest_data[date >= (max_date - 7)]

print(glue("Stocks with recent data: {nrow(latest_data)}"))

# Prepare features for prediction
screen_features <- latest_data[, ..available_features]
screen_matrix <- as.matrix(screen_features)
screen_matrix[is.na(screen_matrix)] <- 0
screen_matrix[is.infinite(screen_matrix)] <- 0

# Get predictions from RSF model
if (!is.null(rsf_model)) {
  rsf_screen_pred <- predict_rsf(rsf_model, latest_data, horizon = MAX_HOLDING_DAYS)
  
  if (!is.null(rsf_screen_pred)) {
    latest_data[, `:=`(
      prob_win = rsf_screen_pred$prob_win,
      prob_loss = rsf_screen_pred$prob_loss,
      win_ratio = rsf_screen_pred$win_loss_ratio
    )]
  }
}

# Add expected return calculation
# Expected return = P(win) * win_size - P(loss) * loss_size
# Using ATR-based thresholds
latest_data[, expected_return := prob_win * (ATR_MULTIPLIER_WIN * atr_pct) - 
                                  prob_loss * (ATR_MULTIPLIER_LOSS * atr_pct)]

# Rank stocks by win probability and expected return
screening_results <- latest_data[!is.na(prob_win), .(
  symbol,
  date,
  close,
  atr_pct = round(atr_pct * 100, 2),  # As percentage
  rsi = round(rsi, 1),
  bb_pct = round(bb_pct, 2),
  prob_win = round(prob_win, 3),
  prob_loss = round(prob_loss, 3),
  win_ratio = round(win_ratio, 3),
  expected_return = round(expected_return * 100, 2)  # As percentage
)][order(-win_ratio)]

print("\n=== TOP 15 STOCK CANDIDATES (by Win Ratio) ===")
print(head(screening_results, 15))

print("\n=== BOTTOM 15 STOCKS (Avoid) ===")
print(tail(screening_results, 15))

## Kelly Criterion Position Sizing

In [ ]:
# ============================================================================
# KELLY CRITERION POSITION SIZING
# ============================================================================

#' Calculate Kelly fraction for a trade
#' 
#' Kelly formula: f* = (p(b+1) - 1) / b
#' where:
#'   p = probability of winning
#'   b = odds (win_size / loss_size)
#'   f* = optimal fraction of bankroll to bet
#'
#' @param prob_win Probability of winning (0 to 1)
#' @param win_size Expected gain if win (as decimal, e.g., 0.10 for 10%)
#' @param loss_size Expected loss if lose (as decimal, e.g., 0.10 for 10%)
#' @param kelly_fraction Fraction of full Kelly to use (0.25 = quarter Kelly)
#' @param max_position Maximum position size
#' @return Optimal position size as fraction of portfolio
calculate_kelly <- function(prob_win, win_size, loss_size, 
                            kelly_fraction = KELLY_FRACTION,
                            max_position = MAX_POSITION_SIZE) {
  
  # Calculate odds ratio (b)
  odds <- win_size / loss_size
  
  # Kelly formula
  kelly_full <- (prob_win * (odds + 1) - 1) / odds
  
  # Apply fractional Kelly
  kelly_adj <- kelly_full * kelly_fraction
  
  # Constrain to [0, max_position]
  kelly_final <- pmax(0, pmin(kelly_adj, max_position))
  
  return(kelly_final)
}

# Calculate Kelly sizing for all screened stocks
print("Calculating Kelly position sizes...")

kelly_results <- copy(latest_data[!is.na(prob_win)])

# Calculate win and loss sizes based on ATR
kelly_results[, `:=`(
  win_size = ATR_MULTIPLIER_WIN * atr_pct,
  loss_size = ATR_MULTIPLIER_LOSS * atr_pct
)]

# Calculate Kelly fraction
kelly_results[, kelly_fraction := calculate_kelly(
  prob_win = prob_win,
  win_size = win_size,
  loss_size = loss_size
)]

# Calculate expected value per trade
kelly_results[, expected_value := prob_win * win_size - prob_loss * loss_size]

# Filter for tradeable candidates
tradeable <- kelly_results[
  kelly_fraction > 0 & 
  prob_win >= MIN_WIN_PROB
][order(-kelly_fraction)]

print(glue("\nTradeable candidates (Kelly > 0, P(win) >= {MIN_WIN_PROB}): {nrow(tradeable)}"))

In [ ]:
# Display final trading recommendations
print("\n=== TRADING RECOMMENDATIONS ===")
print(glue("Using {KELLY_FRACTION * 100}% Kelly with max position {MAX_POSITION_SIZE * 100}%\n"))

if (nrow(tradeable) > 0) {
  
  recommendations <- tradeable[, .(
    symbol,
    date,
    price = round(close, 2),
    atr_pct = round(atr_pct * 100, 2),
    target_up = round(close * (1 + win_size), 2),
    stop_loss = round(close * (1 - loss_size), 2),
    prob_win = round(prob_win * 100, 1),
    prob_loss = round(prob_loss * 100, 1),
    expected_return = round(expected_value * 100, 2),
    kelly_pct = round(kelly_fraction * 100, 2)
  )][order(-kelly_pct)]
  
  print("Top Trading Candidates:")
  print(head(recommendations, 15))
  
  # Summary statistics
  print(glue("\n=== PORTFOLIO SUMMARY ==="))
  print(glue("Total candidates: {nrow(recommendations)}"))
  print(glue("Total Kelly allocation: {round(sum(recommendations$kelly_pct), 1)}%"))
  print(glue("Average expected return: {round(mean(recommendations$expected_return), 2)}%"))
  print(glue("Average win probability: {round(mean(recommendations$prob_win), 1)}%"))
  
  # Visualization
  p_kelly <- ggplot(head(recommendations, 20), 
                    aes(x = reorder(symbol, kelly_pct), y = kelly_pct, fill = prob_win)) +
    geom_col() +
    geom_text(aes(label = paste0(kelly_pct, "%")), hjust = -0.1, size = 3) +
    coord_flip() +
    scale_fill_gradient(low = "#E74C3C", high = "#2ECC71", name = "Win Prob %") +
    labs(
      title = "Kelly Position Sizing by Stock",
      subtitle = paste0(KELLY_FRACTION * 100, "% Kelly, max ", MAX_POSITION_SIZE * 100, "% per position"),
      x = "Symbol",
      y = "Position Size (%)"
    ) +
    theme_minimal() +
    ylim(0, max(recommendations$kelly_pct) * 1.2)
  
  print(p_kelly)
  
} else {
  print("No tradeable candidates found with current parameters.")
  print("Consider adjusting MIN_WIN_PROB or ATR multipliers.")
}

In [ ]:
# Add Kelly sizing plot and recommendations table to report
if (exists("p_kelly")) {
  add_report_item("plot", p_kelly, "Kelly Criterion Position Sizing", "3. Portfolio & Trading")
}

if (exists("recommendations") && nrow(recommendations) > 0) {
  add_report_item("table", as.data.frame(head(recommendations, 15)), "Top Trading Recommendations", "3. Portfolio & Trading")
}

### Portfolio Allocation with Diversification
Create an optimal portfolio with position limits and sector diversification.

In [ ]:
# ============================================================================
# PORTFOLIO ALLOCATION WITH DIVERSIFICATION CONSTRAINTS
# ============================================================================
print("Building diversified portfolio allocation...")

# Portfolio constraints
MAX_POSITIONS <- 10                  # Maximum number of positions
MAX_SINGLE_POSITION <- 0.10          # Max 10% per position
MAX_TOTAL_ALLOCATION <- 0.60         # Max 60% total invested (40% cash buffer)
MIN_KELLY_THRESHOLD <- 0.01          # Minimum 1% Kelly to include
MIN_EXPECTED_VALUE <- 0.005          # Minimum 0.5% expected return

# Create portfolio from tradeable stocks
if (exists("tradeable") && nrow(tradeable) > 0) {
  
  # Filter and rank candidates
  portfolio_candidates <- tradeable[
    kelly_fraction >= MIN_KELLY_THRESHOLD &
    expected_value >= MIN_EXPECTED_VALUE
  ][order(-expected_value)]  # Rank by expected value
  
  print(glue("Portfolio candidates after filtering: {nrow(portfolio_candidates)}"))
  
  if (nrow(portfolio_candidates) > 0) {
    
    # Select top N positions
    n_positions <- min(MAX_POSITIONS, nrow(portfolio_candidates))
    selected <- head(portfolio_candidates, n_positions)
    
    # Normalize Kelly fractions to meet total allocation constraint
    total_raw_kelly <- sum(selected$kelly_fraction)
    
    if (total_raw_kelly > MAX_TOTAL_ALLOCATION) {
      # Scale down proportionally
      scale_factor <- MAX_TOTAL_ALLOCATION / total_raw_kelly
      selected[, allocation := kelly_fraction * scale_factor]
    } else {
      selected[, allocation := kelly_fraction]
    }
    
    # Cap individual positions
    selected[allocation > MAX_SINGLE_POSITION, allocation := MAX_SINGLE_POSITION]
    
    # Recalculate after capping
    total_allocation <- sum(selected$allocation)
    cash_position <- 1 - total_allocation
    
    # Build final portfolio table
    portfolio <- selected[, .(
      symbol,
      entry_date = date,
      entry_price = round(close, 2),
      target_price = round(close * (1 + win_size), 2),
      stop_loss = round(close * (1 - loss_size), 2),
      prob_win = round(prob_win * 100, 1),
      prob_loss = round(prob_loss * 100, 1),
      expected_return_pct = round(expected_value * 100, 2),
      raw_kelly_pct = round(kelly_fraction * 100, 2),
      allocation_pct = round(allocation * 100, 2),
      risk_reward = round(win_size / loss_size, 2)
    )]
    
    print("\n=== PORTFOLIO ALLOCATION ===")
    print(glue("Date: {Sys.Date()}"))
    print(glue("Total positions: {nrow(portfolio)}"))
    print(glue("Total allocation: {round(total_allocation * 100, 1)}%"))
    print(glue("Cash position: {round(cash_position * 100, 1)}%"))
    print("\nPosition Details:")
    print(portfolio)
    
    # Portfolio summary statistics
    portfolio_stats <- data.table(
      metric = c(
        "Number of positions",
        "Total allocation (%)",
        "Cash buffer (%)",
        "Weighted avg win prob (%)",
        "Weighted avg expected return (%)",
        "Portfolio expected return (%)",
        "Avg risk/reward ratio"
      ),
      value = c(
        nrow(portfolio),
        round(total_allocation * 100, 1),
        round(cash_position * 100, 1),
        round(weighted.mean(portfolio$prob_win, portfolio$allocation_pct), 1),
        round(weighted.mean(portfolio$expected_return_pct, portfolio$allocation_pct), 2),
        round(sum(portfolio$expected_return_pct * portfolio$allocation_pct / 100), 2),
        round(mean(portfolio$risk_reward), 2)
      )
    )
    
    print("\n=== PORTFOLIO STATISTICS ===")
    print(portfolio_stats)
    
    # Visualization: Portfolio pie chart
    portfolio_viz <- rbind(
      portfolio[, .(category = symbol, allocation = allocation_pct)],
      data.table(category = "Cash", allocation = round(cash_position * 100, 1))
    )
    
    p_portfolio <- ggplot(portfolio_viz, aes(x = "", y = allocation, fill = category)) +
      geom_bar(stat = "identity", width = 1) +
      coord_polar("y", start = 0) +
      geom_text(aes(label = paste0(category, "\n", allocation, "%")), 
                position = position_stack(vjust = 0.5), size = 3) +
      scale_fill_brewer(palette = "Set3") +
      labs(
        title = "Portfolio Allocation",
        subtitle = paste0(nrow(portfolio), " positions + cash buffer"),
        fill = "Position"
      ) +
      theme_void() +
      theme(legend.position = "none")
    
    print(p_portfolio)
    
    # Expected return vs allocation scatter
    p_alloc <- ggplot(portfolio, aes(x = prob_win, y = allocation_pct, size = expected_return_pct)) +
      geom_point(aes(color = expected_return_pct), alpha = 0.7) +
      geom_text(aes(label = symbol), vjust = -1, size = 3) +
      scale_color_gradient(low = "#E74C3C", high = "#27AE60") +
      scale_size_continuous(range = c(3, 10)) +
      labs(
        title = "Portfolio Positions: Win Probability vs Allocation",
        x = "Win Probability (%)",
        y = "Allocation (%)",
        color = "Exp Return %",
        size = "Exp Return %"
      ) +
      theme_minimal()
    
    print(p_alloc)
    
  } else {
    print("No candidates meet portfolio criteria.")
    portfolio <- data.table()
  }
  
} else {
  print("No tradeable positions available for portfolio construction.")
  portfolio <- data.table()
}

## Backtesting Validation

In [ ]:
# ============================================================================
# BACKTESTING: SIMULATE TRADING WITH KELLY-SIZED POSITIONS
# ============================================================================
print("Running backtest simulation on test set...")

# Use the test set to simulate trading
backtest_data <- copy(test_df)

# Get RSF predictions for test data
if (!is.null(rsf_model)) {
  
  # Sample for computational efficiency
  bt_sample_size <- min(10000, nrow(backtest_data))
  set.seed(123)
  bt_sample_idx <- sample(1:nrow(backtest_data), bt_sample_size)
  bt_sample <- backtest_data[bt_sample_idx]
  
  bt_pred <- predict_rsf(rsf_model, bt_sample, horizon = MAX_HOLDING_DAYS)
  
  if (!is.null(bt_pred)) {
    bt_sample[, `:=`(
      prob_win = bt_pred$prob_win,
      prob_loss = bt_pred$prob_loss,
      win_ratio = bt_pred$win_loss_ratio
    )]
    
    # Calculate Kelly for each trade
    bt_sample[, `:=`(
      win_size = ATR_MULTIPLIER_WIN * atr_pct,
      loss_size = ATR_MULTIPLIER_LOSS * atr_pct
    )]
    
    bt_sample[, kelly := calculate_kelly(prob_win, win_size, loss_size)]
    
    # Apply portfolio constraints (same as live portfolio)
    bt_sample[kelly > MAX_SINGLE_POSITION, kelly := MAX_SINGLE_POSITION]
    
    # Calculate expected value
    bt_sample[, expected_value := prob_win * win_size - prob_loss * loss_size]
    
    # Actual outcome and return (if available)
    bt_sample[, actual_outcome := ifelse(event_type == 1, "win", 
                                          ifelse(event_type == 2, "loss", "censored"))]
    
    # Use final_return if available, otherwise calculate from win/loss size
    if ("final_return" %in% names(bt_sample)) {
      bt_sample[, actual_return := final_return]
    } else {
      # Estimate return based on event type and ATR-based targets
      bt_sample[event_type == 1, actual_return := win_size]
      bt_sample[event_type == 2, actual_return := -loss_size]
      bt_sample[event_type == 0, actual_return := 0]  # Censored
    }
    
    # Filter trades using same criteria as portfolio
    trades <- bt_sample[
      kelly >= MIN_KELLY_THRESHOLD & 
      prob_win >= MIN_WIN_PROB &
      expected_value >= MIN_EXPECTED_VALUE
    ]
    
    print(glue("\nTotal potential trades in test period: {nrow(trades)}"))
    
    if (nrow(trades) > 0) {
      # Calculate weighted returns (Kelly-sized)
      trades[, weighted_return := kelly * actual_return]
      
      # Performance metrics
      total_return <- sum(trades$weighted_return, na.rm = TRUE)
      n_wins <- sum(trades$actual_outcome == "win", na.rm = TRUE)
      n_losses <- sum(trades$actual_outcome == "loss", na.rm = TRUE)
      n_censored <- sum(trades$actual_outcome == "censored", na.rm = TRUE)
      win_rate <- n_wins / (n_wins + n_losses)
      
      # Average metrics
      avg_kelly <- mean(trades$kelly, na.rm = TRUE)
      avg_return <- mean(trades$actual_return, na.rm = TRUE)
      sharpe_proxy <- mean(trades$actual_return, na.rm = TRUE) / sd(trades$actual_return, na.rm = TRUE)
      
      # Calculate drawdown
      trades_sorted <- trades[order(date)]
      trades_sorted[, cumulative_return := cumsum(weighted_return)]
      trades_sorted[, cummax_return := cummax(cumulative_return)]
      trades_sorted[, drawdown := cummax_return - cumulative_return]
      max_drawdown <- max(trades_sorted$drawdown, na.rm = TRUE)
      
      # Calculate profit factor
      gross_profit <- sum(trades[actual_outcome == "win"]$weighted_return, na.rm = TRUE)
      gross_loss <- abs(sum(trades[actual_outcome == "loss"]$weighted_return, na.rm = TRUE))
      profit_factor <- ifelse(gross_loss > 0, gross_profit / gross_loss, Inf)
      
      print("\n=== BACKTEST RESULTS (Kelly-Sized) ===")
      print(glue("Number of trades: {nrow(trades)}"))
      print(glue("Wins: {n_wins} ({round(n_wins/nrow(trades)*100, 1)}%)"))
      print(glue("Losses: {n_losses} ({round(n_losses/nrow(trades)*100, 1)}%)"))
      print(glue("Censored: {n_censored} ({round(n_censored/nrow(trades)*100, 1)}%)"))
      print(glue("Win rate (ex-censored): {round(win_rate * 100, 1)}%"))
      print(glue("Average Kelly size: {round(avg_kelly * 100, 2)}%"))
      print(glue("Average return per trade: {round(avg_return * 100, 2)}%"))
      print(glue("Total cumulative return: {round(total_return * 100, 2)}%"))
      print(glue("Max drawdown: {round(max_drawdown * 100, 2)}%"))
      print(glue("Profit factor: {round(profit_factor, 2)}"))
      print(glue("Sharpe proxy: {round(sharpe_proxy, 2)}"))
      
      # Breakdown by predicted probability bucket
      print("\n=== PERFORMANCE BY PREDICTED WIN PROBABILITY ===")
      trades[, prob_bucket := cut(prob_win, breaks = c(0, 0.4, 0.5, 0.6, 0.7, 1), 
                                   labels = c("<40%", "40-50%", "50-60%", "60-70%", ">70%"))]
      
      bucket_perf <- trades[!is.na(prob_bucket), .(
        n_trades = .N,
        win_rate = round(mean(actual_outcome == "win") * 100, 1),
        avg_return = round(mean(actual_return, na.rm = TRUE) * 100, 2),
        total_return = round(sum(weighted_return, na.rm = TRUE) * 100, 2),
        avg_kelly = round(mean(kelly) * 100, 2)
      ), by = prob_bucket][order(prob_bucket)]
      
      print(bucket_perf)
    }
  }
} else {
  print("RSF model not available for backtesting")
}

In [ ]:
# Backtest equity curve visualization
if (exists("trades") && nrow(trades) > 0) {
  
  # Sort trades by date
  trades_sorted <- trades[order(date)]
  trades_sorted[, cumulative_return := cumsum(weighted_return)]
  trades_sorted[, trade_num := 1:.N]
  
  # Equity curve
  p_equity <- ggplot(trades_sorted, aes(x = trade_num, y = cumulative_return * 100)) +
    geom_line(color = "#3498DB", size = 1) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "gray50") +
    geom_point(aes(color = actual_outcome), size = 1, alpha = 0.5) +
    scale_color_manual(values = c("win" = "#2ECC71", "loss" = "#E74C3C", "censored" = "#95A5A6")) +
    labs(
      title = "Backtest Equity Curve",
      subtitle = glue("Test period: {min(trades_sorted$date)} to {max(trades_sorted$date)}"),
      x = "Trade Number",
      y = "Cumulative Return (%)",
      color = "Outcome"
    ) +
    theme_minimal() +
    theme(legend.position = "bottom")
  
  print(p_equity)
  
  # Return distribution by predicted probability
  p_returns <- ggplot(trades, aes(x = prob_win, y = actual_return * 100, color = actual_outcome)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", color = "black", linetype = "dashed") +
    geom_hline(yintercept = 0, color = "gray50") +
    scale_color_manual(values = c("win" = "#2ECC71", "loss" = "#E74C3C", "censored" = "#95A5A6")) +
    labs(
      title = "Predicted Probability vs Actual Return",
      x = "Predicted Win Probability",
      y = "Actual Return (%)",
      color = "Outcome"
    ) +
    theme_minimal() +
    theme(legend.position = "bottom")
  
  print(p_returns)
  
  # Confusion matrix style summary
  outcome_by_prob <- trades[, .(
    n = .N,
    win_rate = mean(actual_outcome == "win"),
    avg_return = mean(actual_return, na.rm = TRUE) * 100
  ), by = .(prob_bucket = cut(prob_win, breaks = seq(0, 1, 0.1)))][order(prob_bucket)]
  
  print("\nPerformance by Predicted Probability Bucket:")
  print(outcome_by_prob)
}

In [ ]:
# Add backtest plots and performance table to report
if (exists("p_equity")) {
  add_report_item("plot", p_equity, "Backtest Equity Curve", "4. Backtesting")
}

if (exists("p_returns")) {
  add_report_item("plot", p_returns, "Predicted Probability vs Actual Returns", "4. Backtesting")
}

if (exists("outcome_by_prob") && nrow(outcome_by_prob) > 0) {
  add_report_item("table", as.data.frame(outcome_by_prob), "Performance by Probability Bucket", "4. Backtesting")
}

## Save Survival Analysis Results

In [ ]:
# ============================================================================
# SAVE RESULTS
# ============================================================================
print("Saving survival analysis results...")

# Ensure data directory exists
if (!dir.exists("data")) {
  dir.create("data", recursive = TRUE)
}

# Save survival modeling data
if (exists("survival_complete") && nrow(survival_complete) > 0) {
  write_parquet(survival_complete, "data/survival_outcomes.parquet")
  print(glue("Saved survival outcomes: {nrow(survival_complete)} rows"))
}

# Save screening results
if (exists("recommendations") && nrow(recommendations) > 0) {
  write_parquet(recommendations, "data/trading_recommendations.parquet")
  print(glue("Saved trading recommendations: {nrow(recommendations)} rows"))
}

# Save backtest results
if (exists("trades") && nrow(trades) > 0) {
  write_parquet(trades, "data/backtest_trades.parquet")
  print(glue("Saved backtest trades: {nrow(trades)} rows"))
}

# Save model coefficients
if (exists("coef_win") && nrow(coef_win) > 0) {
  write_parquet(coef_win, "data/finegray_coefficients.parquet")
  print("Saved Fine-Gray coefficients")
}

# Save summary
survival_summary <- data.table(
  analysis_date = Sys.Date(),
  atr_period = ATR_PERIOD,
  atr_mult_win = ATR_MULTIPLIER_WIN,
  atr_mult_loss = ATR_MULTIPLIER_LOSS,
  max_holding_days = MAX_HOLDING_DAYS,
  kelly_fraction = KELLY_FRACTION,
  n_stocks_analyzed = length(valid_stocks),
  n_observations = nrow(survival_complete),
  n_tradeable = ifelse(exists("tradeable"), nrow(tradeable), 0),
  backtest_n_trades = ifelse(exists("trades"), nrow(trades), 0),
  backtest_win_rate = ifelse(exists("win_rate"), win_rate, NA)
)

write_parquet(survival_summary, "data/survival_analysis_summary.parquet")
print("Saved survival analysis summary")

print("\n=== SURVIVAL ANALYSIS FILES CREATED ===")
survival_files <- list.files("data", pattern = "survival|trading|backtest|finegray", full.names = TRUE)
for (file in survival_files) {
  size_kb <- round(file.size(file) / 1024, 2)
  print(glue("{file}: {size_kb} KB"))
}

## Database Connection Cleanup

In [ ]:
# Close database connection
dbDisconnect(con, shutdown = TRUE)
print("Database connection closed.")
print("\nFinancial modeling pipeline completed successfully!")

# Section 8: Ratio Analysis
**Identifying Extreme Ratio Valuations**

This section analyzes the 706 price ratios (3.8M+ observations) from the database to identify relative valuations at extreme levels (near all-time highs or lows). The ratios were pre-calculated by the ETL pipeline using all pairwise combinations of tradeable assets with LOCF harmonization.

Analysis includes:
- **Volcano plot** showing distance from all-time highs/lows vs Z-score
- **Ranked table** of extreme ratios with flags
- **Time series charts** for the most extreme ratios
- **Percentile heatmap** showing recent positioning
- **Mean reversion analysis** with forward return expectations

Key ratio categories:
- **Cross-asset**: Gold/BTC, S&P500/Gold, Bonds/Equities  
- **Metals**: Silver/Gold, Palladium/Gold
- **Indices**: NASDAQ/S&P500
- **Crypto**: ETH/BTC
- **Sectors**: Tech, Energy, Healthcare, Finance vs S&P500

## 8.1 Load Ratio Data from Database

In [ ]:
print("=== LOADING RATIO DATA FROM DATABASE ===")

# Check if database connection exists and is valid
if (!exists("con") || !dbIsValid(con)) {
  print("Database connection not found or invalid. Reconnecting...")
  DB_PATH <- "data/financial_data.duckdb"
  
  if (!file.exists(DB_PATH)) {
    stop(glue("Database file not found at {DB_PATH}. Please run 01_etl.ipynb first."))
  }
  
  con <- dbConnect(duckdb::duckdb(), dbdir = DB_PATH, read_only = TRUE)
  print("Reconnected to database successfully")
}

# Query all ratio data that was calculated by the ETL pipeline
df_ratios_raw <- dbGetQuery(con, "
  SELECT 
    series as ratio_name,
    date,
    value as ratio_value
  FROM financial_data
  WHERE origin = 'RATIO'
  ORDER BY series, date
")

# Convert date to proper format
df_ratios_raw$date <- as.Date(df_ratios_raw$date)

print(glue("Loaded {nrow(df_ratios_raw)} ratio observations"))
print(glue("Total unique ratios: {length(unique(df_ratios_raw$ratio_name))}"))
print(glue("Date range: {min(df_ratios_raw$date)} to {max(df_ratios_raw$date)}"))

# Show sample of ratio names
print("\nSample of available ratios:")
sample_ratios <- unique(df_ratios_raw$ratio_name) %>% head(20)
print(sample_ratios)

# Select specific ratios of interest for detailed analysis
# Focus on key cross-asset ratios that are more interpretable
key_ratio_patterns <- c(
  "GC.F_per_BTC.USD",      # Gold vs Bitcoin
  ".GSPC_per_GC.F",        # S&P500 vs Gold  
  "GC.F_per_SI.F",         # Gold vs Silver
  "SI.F_per_GC.F",         # Silver vs Gold
  "PA.F_per_GC.F",         # Palladium vs Gold
  ".IXIC_per_.GSPC",       # NASDAQ vs S&P500
  "ETH.USD_per_BTC.USD",   # Ethereum vs Bitcoin
  "TLT_per_.GSPC",         # Long Bonds vs S&P500
  "XLK_per_.GSPC",         # Tech sector vs S&P500
  "XLE_per_.GSPC",         # Energy sector vs S&P500
  "XLV_per_.GSPC",         # Healthcare vs S&P500
  "XLF_per_.GSPC",         # Finance vs S&P500
  "CL.F_per_GC.F",         # Oil vs Gold
  "DX.Y.NYB_per_GC.F",     # Dollar Index vs Gold
  "BTC.USD_per_GC.F"       # Bitcoin vs Gold (inverse)
)

# Filter to key ratios (if they exist)
df_ratios <- df_ratios_raw %>%
  filter(ratio_name %in% key_ratio_patterns)

# If no matches found with exact names, try partial matching
if (nrow(df_ratios) == 0) {
  print("\nExact ratio names not found. Trying partial matching...")
  
  # Create a pattern that matches any of the key assets
  assets_of_interest <- c("GC.F", "BTC.USD", "ETH.USD", ".GSPC", ".IXIC", 
                          "SI.F", "PA.F", "TLT", "XLK", "XLE", "XLV", "XLF", 
                          "CL.F", "DX.Y.NYB")
  
  df_ratios <- df_ratios_raw %>%
    filter(
      grepl(paste(assets_of_interest, collapse = "|"), ratio_name)
    ) %>%
    group_by(ratio_name) %>%
    filter(n() >= 100) %>%  # Only ratios with sufficient history
    ungroup()
}

# If still no data, use all ratios
if (nrow(df_ratios) == 0) {
  print("\nUsing all available ratios for analysis...")
  df_ratios <- df_ratios_raw %>%
    group_by(ratio_name) %>%
    filter(n() >= 252) %>%  # At least 1 year of daily data
    ungroup()
}

print(glue("\nSelected {length(unique(df_ratios$ratio_name))} ratios for detailed analysis"))
print(glue("Total observations: {nrow(df_ratios)}"))

## 8.2 Identify Extreme Valuations

In [ ]:
print("=== IDENTIFYING EXTREME RATIO VALUATIONS ===")

# Calculate all-time highs and lows for each ratio
ratio_extremes <- df_ratios %>%
  group_by(ratio_name) %>%
  summarize(
    current_value = last(ratio_value),
    current_date = last(date),
    all_time_high = max(ratio_value, na.rm = TRUE),
    all_time_low = min(ratio_value, na.rm = TRUE),
    ath_date = date[which.max(ratio_value)],
    atl_date = date[which.min(ratio_value)],
    mean_value = mean(ratio_value, na.rm = TRUE),
    median_value = median(ratio_value, na.rm = TRUE),
    sd_value = sd(ratio_value, na.rm = TRUE),
    n_obs = n(),
    .groups = "drop"
  ) %>%
  mutate(
    # Calculate distance from extremes
    pct_from_ath = ((current_value - all_time_high) / all_time_high) * 100,
    pct_from_atl = ((current_value - all_time_low) / all_time_low) * 100,
    
    # Calculate Z-score
    z_score = (current_value - mean_value) / sd_value,
    
    # Calculate percentile rank
    pct_rank = percent_rank(current_value),
    
    # Distance from mean in standard deviations
    sds_from_mean = abs(z_score),
    
    # Classify position
    position = case_when(
      pct_from_ath > -5 ~ "Near ATH",
      pct_from_atl < 5 ~ "Near ATL",
      z_score > 1 ~ "Above Average",
      z_score < -1 ~ "Below Average",
      TRUE ~ "Neutral"
    ),
    
    # Calculate "extremeness" score (lower = more extreme)
    extreme_score = pmin(abs(pct_from_ath), abs(pct_from_atl)),
    
    # Days since ATH/ATL
    days_since_ath = as.numeric(current_date - ath_date),
    days_since_atl = as.numeric(current_date - atl_date)
  ) %>%
  arrange(extreme_score)

print("\nTop 10 Most Extreme Ratios:")
print(ratio_extremes %>% 
  select(ratio_name, position, pct_from_ath, pct_from_atl, z_score, extreme_score) %>%
  head(10))

# Save for later analysis
write_parquet(ratio_extremes, "data/ratio_extremes.parquet")
print("\nSaved ratio extremes to data/ratio_extremes.parquet")

## 8.3 Volcano Plot: Distance from All-Time Extremes

In [ ]:
print("=== CREATING VOLCANO PLOT ===")

# Create volcano plot data
volcano_data <- ratio_extremes %>%
  mutate(
    # X-axis: Z-score (position relative to historical mean)
    x_position = z_score,
    
    # Y-axis: Extremeness (how close to ATH or ATL)
    y_extremeness = -log10(extreme_score + 1),  # Higher = more extreme
    
    # Color by position type
    color_group = position,
    
    # Size by number of observations (data quality indicator)
    point_size = log10(n_obs)
  )

# Create the volcano plot
p_volcano <- ggplot(volcano_data, aes(x = x_position, y = y_extremeness)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", alpha = 0.5) +
  geom_hline(yintercept = -log10(5 + 1), linetype = "dashed", color = "red", alpha = 0.3) +
  geom_point(aes(color = color_group, size = point_size), alpha = 0.7) +
  geom_text(data = volcano_data %>% filter(y_extremeness > -log10(10 + 1)),
            aes(label = ratio_name), 
            vjust = -0.5, hjust = 0.5, size = 3, check_overlap = TRUE) +
  scale_color_manual(values = c(
    "Near ATH" = "#d73027",
    "Near ATL" = "#4575b4", 
    "Above Average" = "#fee090",
    "Below Average" = "#e0f3f8",
    "Neutral" = "#808080"
  )) +
  labs(
    title = "Ratio Volcano Plot: Identifying Extreme Valuations",
    subtitle = "Y-axis: Distance from ATH/ATL (higher = closer to extreme) | X-axis: Z-score (position vs historical mean)",
    x = "Z-Score (Standard Deviations from Mean)",
    y = "Extremeness Score (-log10 distance from ATH/ATL)",
    color = "Position",
    size = "Data Quality (log obs)"
  ) +
  theme_money_printer_go_brrr() +
  theme(
    legend.position = "right",
    plot.title = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 10)
  )

print(p_volcano)


In [ ]:
# Add volcano plot and ratio extremes table to report
if (exists("p_volcano")) {
  add_report_item("plot", p_volcano, "Ratio Volcano Plot - Extreme Valuations", "5. Ratio Analysis")
}

if (exists("ratio_extremes") && nrow(ratio_extremes) > 0) {
  ratio_summary <- ratio_extremes %>%
    select(ratio_name, position, pct_from_ath, pct_from_atl, z_score) %>%
    head(12)
  add_report_item("table", as.data.frame(ratio_summary), "Top Extreme Ratio Valuations", "5. Ratio Analysis")
}

In [ ]:


# Identify ratios in each quadrant
print("\n=== QUADRANT ANALYSIS ===")

quadrants <- volcano_data %>%
  mutate(
    quadrant = case_when(
      x_position > 0 & y_extremeness > -log10(10 + 1) ~ "High & Near Extreme (Overvalued)",
      x_position < 0 & y_extremeness > -log10(10 + 1) ~ "Low & Near Extreme (Undervalued)",
      x_position > 0 & y_extremeness <= -log10(10 + 1) ~ "High but Not Extreme",
      x_position < 0 & y_extremeness <= -log10(10 + 1) ~ "Low but Not Extreme",
      TRUE ~ "Neutral"
    )
  )

print("Ratios by Quadrant:")
print(table(quadrants$quadrant))

## 8.4 Ranked Table of Extreme Ratios

In [ ]:
print("=== CREATING RANKED TABLE OF EXTREME RATIOS ===")

# Create comprehensive table
extreme_ratios_table <- ratio_extremes %>%
  mutate(
    # Format percentages
    pct_from_ath_fmt = sprintf("%.1f%%", pct_from_ath),
    pct_from_atl_fmt = sprintf("%.1f%%", pct_from_atl),
    
    # Format dates
    ath_date_fmt = format(ath_date, "%Y-%m-%d"),
    atl_date_fmt = format(atl_date, "%Y-%m-%d"),
    current_date_fmt = format(current_date, "%Y-%m-%d"),
    
    # Format values
    current_value_fmt = sprintf("%.4f", current_value),
    all_time_high_fmt = sprintf("%.4f", all_time_high),
    all_time_low_fmt = sprintf("%.4f", all_time_low),
    z_score_fmt = sprintf("%.2f", z_score),
    
    # Highlight extreme values
    extreme_flag = ifelse(extreme_score < 10, "⚠️ EXTREME", ""),
    
    # Direction indicator
    direction = case_when(
      pct_from_ath > -5 ~ "📈 Near High",
      pct_from_atl < 5 ~ "📉 Near Low",
      z_score > 1 ~ "↗️ Above Avg",
      z_score < -1 ~ "↘️ Below Avg",
      TRUE ~ "➡️ Neutral"
    )
  ) %>%
  select(
    Ratio = ratio_name,
    Direction = direction,
    `Current Value` = current_value_fmt,
    `Z-Score` = z_score_fmt,
    `From ATH` = pct_from_ath_fmt,
    `From ATL` = pct_from_atl_fmt,
    `ATH Value` = all_time_high_fmt,
    `ATL Value` = all_time_low_fmt,
    `ATH Date` = ath_date_fmt,
    `ATL Date` = atl_date_fmt,
    `Days Since ATH` = days_since_ath,
    `Days Since ATL` = days_since_atl,
    Flag = extreme_flag,
    Position = position
  ) %>%
  arrange(desc(Flag), abs(as.numeric(gsub("%", "", `From ATH`))))

print("\n=== TOP 15 MOST EXTREME RATIOS ===")
print(extreme_ratios_table %>% head(15))

# Save as both parquet and CSV for easy viewing
write_parquet(extreme_ratios_table, "data/extreme_ratios_table.parquet")
write.csv(extreme_ratios_table, "data/extreme_ratios_table.csv", row.names = FALSE)

print("\nSaved extreme ratios table to:")
print("  - data/extreme_ratios_table.parquet")
print("  - data/extreme_ratios_table.csv")

# Create a summary by position
position_summary <- ratio_extremes %>%
  group_by(position) %>%
  summarize(
    count = n(),
    avg_extreme_score = mean(extreme_score),
    avg_z_score = mean(z_score),
    .groups = "drop"
  ) %>%
  arrange(avg_extreme_score)

print("\n=== POSITION SUMMARY ===")
print(position_summary)

## 8.5 Time Series Charts for Top Extreme Ratios

In [ ]:
print("=== CREATING TIME SERIES CHARTS FOR TOP EXTREME RATIOS ===")

# Get top 6 most extreme ratios
top_extreme_ratios <- ratio_extremes %>%
  arrange(extreme_score) %>%
  head(6) %>%
  pull(ratio_name)

print(glue("Plotting time series for: {paste(top_extreme_ratios, collapse=', ')}"))

# Create individual plots for each ratio
ratio_plots <- list()

for (ratio_name in top_extreme_ratios) {
  # Get ratio data
  ratio_ts <- df_ratios %>%
    filter(ratio_name == !!ratio_name)
  
  # Get extreme info
  ratio_info <- ratio_extremes %>%
    filter(ratio_name == !!ratio_name)
  
  # Create plot
  p <- ggplot(ratio_ts, aes(x = date, y = ratio_value)) +
    geom_line(color = "#1f77b4", size = 0.8) +
    
    # Mark all-time high
    geom_hline(yintercept = ratio_info$all_time_high, 
               linetype = "dashed", color = "#d73027", alpha = 0.7) +
    annotate("text", x = min(ratio_ts$date), y = ratio_info$all_time_high,
             label = "ATH", vjust = -0.5, hjust = 0, color = "#d73027", size = 3) +
    
    # Mark all-time low
    geom_hline(yintercept = ratio_info$all_time_low,
               linetype = "dashed", color = "#4575b4", alpha = 0.7) +
    annotate("text", x = min(ratio_ts$date), y = ratio_info$all_time_low,
             label = "ATL", vjust = 1.5, hjust = 0, color = "#4575b4", size = 3) +
    
    # Mark mean
    geom_hline(yintercept = ratio_info$mean_value,
               linetype = "dotted", color = "gray50", alpha = 0.5) +
    
    # Mark current value
    geom_point(data = ratio_ts %>% slice_tail(n = 1),
               aes(x = date, y = ratio_value),
               color = "#ff7f0e", size = 3) +
    
    # Shaded regions for ±1 SD
    geom_ribbon(aes(ymin = ratio_info$mean_value - ratio_info$sd_value,
                    ymax = ratio_info$mean_value + ratio_info$sd_value),
                alpha = 0.1, fill = "gray50") +
    
    labs(
      title = glue("{ratio_name} - {ratio_info$position}"),
      subtitle = glue(
        "Current: {round(ratio_info$current_value, 4)} | ",
        "From ATH: {round(ratio_info$pct_from_ath, 1)}% | ",
        "From ATL: {round(ratio_info$pct_from_atl, 1)}% | ",
        "Z-Score: {round(ratio_info$z_score, 2)}"
      ),
      x = NULL,
      y = "Ratio Value"
    ) +
    theme_money_printer_go_brrr() +
    theme(
      plot.title = element_text(size = 11, face = "bold"),
      plot.subtitle = element_text(size = 9)
    )
  
  ratio_plots[[ratio_name]] <- p
}

# Print individual plots
for (ratio_name in names(ratio_plots)) {
  print(ratio_plots[[ratio_name]])
}

print(glue("\nSaved {length(ratio_plots)} individual ratio time series plots"))

## 8.6 Combined Panel Plot of Top Ratios

In [ ]:
print("=== CREATING COMBINED PANEL PLOT ===")

# Get top 8 extreme ratios for panel plot
top_8_ratios <- ratio_extremes %>%
  arrange(extreme_score) %>%
  head(8) %>%
  pull(ratio_name)

# Filter data for these ratios
panel_data <- df_ratios %>%
  filter(ratio_name %in% top_8_ratios) %>%
  left_join(
    ratio_extremes %>% select(ratio_name, all_time_high, all_time_low, mean_value, position),
    by = "ratio_name"
  ) %>%
  group_by(ratio_name) %>%
  mutate(
    # Normalize to 0-1 scale for easier comparison
    normalized_value = (ratio_value - min(ratio_value)) / (max(ratio_value) - min(ratio_value)),
    
    # Calculate percentile position
    percentile_rank = percent_rank(ratio_value) * 100,
    
    # Mark if at extreme
    is_extreme = ratio_value >= quantile(ratio_value, 0.95) | ratio_value <= quantile(ratio_value, 0.05)
  ) %>%
  ungroup()

# Create faceted plot
p_panel <- ggplot(panel_data, aes(x = date, y = ratio_value)) +
  geom_line(aes(color = position), size = 0.6, alpha = 0.8) +
  
  # Mark extremes
  geom_point(data = panel_data %>% filter(is_extreme),
             aes(x = date, y = ratio_value),
             color = "red", size = 0.5, alpha = 0.3) +
  
  # Add mean line
  geom_hline(aes(yintercept = mean_value), 
             linetype = "dashed", color = "gray50", size = 0.3) +
  
  facet_wrap(~ ratio_name, scales = "free_y", ncol = 2) +
  
  scale_color_manual(values = c(
    "Near ATH" = "#d73027",
    "Near ATL" = "#4575b4",
    "Above Average" = "#fdae61",
    "Below Average" = "#abd9e9",
    "Neutral" = "#808080"
  )) +
  
  labs(
    title = "Top 8 Most Extreme Ratio Valuations - Time Series Panel",
    subtitle = "Dashed line = historical mean | Red dots = extreme values (top/bottom 5%)",
    x = NULL,
    y = "Ratio Value",
    color = "Current Position"
  ) +
  
  theme_money_printer_go_brrr() +
  theme(
    strip.text = element_text(size = 9, face = "bold"),
    legend.position = "bottom",
    axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
    panel.spacing = unit(1, "lines")
  )

print(p_panel)


print("\nSaved combined panel plot")

In [ ]:
# Add panel plot to report
if (exists("p_panel")) {
  add_report_item("plot", p_panel, "Top 8 Extreme Ratios - Time Series", "5. Ratio Analysis")
}

## 8.7 Heatmap: Historical Percentile Positions

In [ ]:
print("=== CREATING HISTORICAL PERCENTILE HEATMAP ===")

# Calculate rolling percentiles for all ratios
heatmap_data <- df_ratios %>%
  group_by(ratio_name) %>%
  arrange(date) %>%
  mutate(
    # Calculate percentile rank at each point in time
    historical_percentile = percent_rank(ratio_value) * 100,
    
    # Categorize into bins
    percentile_bin = cut(
      historical_percentile,
      breaks = c(0, 5, 25, 50, 75, 95, 100),
      labels = c("0-5%", "5-25%", "25-50%", "50-75%", "75-95%", "95-100%"),
      include.lowest = TRUE
    )
  ) %>%
  ungroup()

# Get recent data (last 90 days) for all ratios
recent_heatmap <- heatmap_data %>%
  filter(date >= max(date) - 90) %>%
  mutate(
    days_ago = as.numeric(max(date) - date),
    week = floor(days_ago / 7)
  ) %>%
  group_by(ratio_name, week) %>%
  summarize(
    avg_percentile = mean(historical_percentile),
    date_label = format(min(date), "%b %d"),
    .groups = "drop"
  ) %>%
  filter(week <= 12)  # Last 12 weeks

# Create heatmap
p_heatmap <- ggplot(recent_heatmap, aes(x = factor(week), y = ratio_name, fill = avg_percentile)) +
  geom_tile(color = "white", size = 0.5) +
  
  scale_fill_gradient2(
    low = "#4575b4",      # Blue for low percentiles
    mid = "#ffffbf",      # Yellow for middle
    high = "#d73027",     # Red for high percentiles
    midpoint = 50,
    limits = c(0, 100),
    breaks = c(0, 25, 50, 75, 100),
    labels = c("0%\n(ATL)", "25%", "50%\n(Median)", "75%", "100%\n(ATH)")
  ) +
  
  geom_text(aes(label = sprintf("%.0f", avg_percentile)),
            size = 2.5, color = "black") +
  
  labs(
    title = "Ratio Percentile Heatmap - Last 12 Weeks",
    subtitle = "Shows where each ratio stands relative to its full historical range",
    x = "Weeks Ago",
    y = NULL,
    fill = "Historical\nPercentile"
  ) +
  
  scale_x_discrete(labels = rev(0:12)) +
  
  theme_money_printer_go_brrr() +
  theme(
    axis.text.x = element_text(size = 9),
    axis.text.y = element_text(size = 9),
    legend.position = "right",
    panel.grid = element_blank(),
    plot.title = element_text(size = 12, face = "bold")
  )

print(p_heatmap)


# Summary statistics
print("\n=== CURRENT PERCENTILE DISTRIBUTION ===")
current_percentiles <- heatmap_data %>%
  filter(date == max(date)) %>%
  mutate(
    percentile_category = case_when(
      historical_percentile <= 5 ~ "Bottom 5% (Near ATL)",
      historical_percentile <= 25 ~ "Bottom Quartile",
      historical_percentile <= 75 ~ "Middle 50%",
      historical_percentile <= 95 ~ "Top Quartile",
      TRUE ~ "Top 5% (Near ATH)"
    )
  ) %>%
  count(percentile_category) %>%
  arrange(desc(n))

print(current_percentiles)

In [ ]:
# Add heatmap to report
if (exists("p_heatmap")) {
  add_report_item("plot", p_heatmap, "Historical Percentile Heatmap (90 Days)", "5. Ratio Analysis")
}

## 8.8 Mean Reversion Analysis

In [ ]:
print("=== MEAN REVERSION ANALYSIS ===")

# For each ratio, calculate mean reversion metrics
mean_reversion_analysis <- df_ratios %>%
  group_by(ratio_name) %>%
  arrange(date) %>%
  mutate(
    # Distance from mean in standard deviations
    z_score = (ratio_value - mean(ratio_value)) / sd(ratio_value),
    
    # Is it in extreme territory? (>2 or <-2 SD)
    is_extreme = abs(z_score) > 2,
    
    # Calculate forward returns at different horizons
    ret_1m = lead(ratio_value, 20) / ratio_value - 1,
    ret_3m = lead(ratio_value, 60) / ratio_value - 1,
    ret_6m = lead(ratio_value, 120) / ratio_value - 1,
    ret_1y = lead(ratio_value, 252) / ratio_value - 1
  ) %>%
  ungroup()

# Analyze returns conditional on starting Z-score
reversion_stats <- mean_reversion_analysis %>%
  filter(!is.na(ret_3m)) %>%
  mutate(
    z_bucket = cut(
      z_score,
      breaks = c(-Inf, -2, -1, 0, 1, 2, Inf),
      labels = c("Very Low (<-2)", "Low (-2 to -1)", "Below Avg (-1 to 0)", 
                 "Above Avg (0 to 1)", "High (1 to 2)", "Very High (>2)")
    )
  ) %>%
  group_by(ratio_name, z_bucket) %>%
  summarize(
    n_obs = n(),
    avg_3m_return = mean(ret_3m, na.rm = TRUE),
    median_3m_return = median(ret_3m, na.rm = TRUE),
    pct_positive = mean(ret_3m > 0, na.rm = TRUE) * 100,
    .groups = "drop"
  ) %>%
  filter(n_obs >= 5)  # Only buckets with sufficient observations

# Get current Z-scores and expected reversion
current_reversion_outlook <- ratio_extremes %>%
  select(ratio_name, current_value, z_score, position) %>%
  left_join(
    reversion_stats %>%
      group_by(ratio_name) %>%
      arrange(desc(abs(as.numeric(z_bucket)))) %>%
      slice(1),
    by = "ratio_name"
  ) %>%
  mutate(
    reversion_signal = case_when(
      z_score > 2 & avg_3m_return < 0 ~ "⬇️ Strong Mean Reversion Expected",
      z_score < -2 & avg_3m_return > 0 ~ "⬆️ Strong Mean Reversion Expected",
      abs(z_score) > 1.5 ~ "↔️ Moderate Mean Reversion Expected",
      TRUE ~ "➡️ No Strong Signal"
    ),
    expected_3m_return_pct = avg_3m_return * 100
  ) %>%
  arrange(desc(abs(z_score)))

print("\n=== MEAN REVERSION OUTLOOK (Top 10 by Z-Score) ===")
print(current_reversion_outlook %>%
  select(ratio_name, z_score, position, reversion_signal, expected_3m_return_pct, pct_positive) %>%
  head(10))

# Plot: Z-score vs forward returns
p_reversion <- ggplot(reversion_stats, aes(x = z_bucket, y = avg_3m_return * 100)) +
  geom_boxplot(aes(fill = z_bucket), alpha = 0.6, outlier.shape = NA) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  
  scale_fill_manual(values = c(
    "Very Low (<-2)" = "#313695",
    "Low (-2 to -1)" = "#4575b4",
    "Below Avg (-1 to 0)" = "#abd9e9",
    "Above Avg (0 to 1)" = "#fee090",
    "High (1 to 2)" = "#f46d43",
    "Very High (>2)" = "#a50026"
  )) +
  
  labs(
    title = "Mean Reversion Evidence: 3-Month Forward Returns by Z-Score Bucket",
    subtitle = "Extreme Z-scores (|Z| > 2) tend to revert toward mean",
    x = "Starting Z-Score Bucket",
    y = "Average 3-Month Forward Return (%)",
    fill = "Z-Score Range"
  ) +
  
  theme_money_printer_go_brrr() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  )

print(p_reversion)

# Save reversion analysis
write_parquet(current_reversion_outlook, "data/ratio_mean_reversion_outlook.parquet")
print("\nSaved mean reversion analysis to data/ratio_mean_reversion_outlook.parquet")

In [ ]:
# Add mean reversion plot and outlook table to report
if (exists("p_reversion")) {
  add_report_item("plot", p_reversion, "Mean Reversion Evidence by Z-Score", "5. Ratio Analysis")
}

if (exists("current_reversion_outlook") && nrow(current_reversion_outlook) > 0) {
  reversion_summary <- current_reversion_outlook %>%
    select(ratio_name, z_score, position, reversion_signal, expected_3m_return_pct, pct_positive) %>%
    head(10)
  add_report_item("table", as.data.frame(reversion_summary), "Mean Reversion Outlook - Top 10", "5. Ratio Analysis")
}

## 8.9 Summary: Ratio Analysis Output Files

In [ ]:
print("\n=== RATIO ANALYSIS SUMMARY ===")

print("\nFiles created:")
ratio_files <- list.files("data", pattern = "ratio", full.names = TRUE)
for (file in ratio_files) {
  size_kb <- round(file.size(file) / 1024, 2)
  print(glue("  {file}: {size_kb} KB"))
}

print("\nPlots saved:")
ratio_plot_files <- list.files("output", pattern = "ratio", full.names = FALSE)
for (file in ratio_plot_files) {
  print(glue("  output/{file}"))
}

print("\n=== KEY FINDINGS ===")

# Most extreme ratios
top_3_extreme <- ratio_extremes %>%
  arrange(extreme_score) %>%
  head(3)

print("\nTop 3 Most Extreme Valuations:")
for (i in 1:nrow(top_3_extreme)) {
  ratio <- top_3_extreme[i, ]
  print(glue(
    "{i}. {ratio$ratio_name}: {ratio$position}",
    "   Current: {round(ratio$current_value, 4)} | From ATH: {round(ratio$pct_from_ath, 1)}% | From ATL: {round(ratio$pct_from_atl, 1)}%"
  ))
}

# Strongest mean reversion candidates
strong_reversion <- current_reversion_outlook %>%
  filter(grepl("Strong", reversion_signal)) %>%
  arrange(desc(abs(z_score))) %>%
  head(3)

if (nrow(strong_reversion) > 0) {
  print("\nTop Mean Reversion Candidates:")
  for (i in 1:nrow(strong_reversion)) {
    ratio <- strong_reversion[i, ]
    print(glue(
      "{i}. {ratio$ratio_name}: {ratio$reversion_signal}",
      "   Z-Score: {round(ratio$z_score, 2)} | Expected 3M Return: {round(ratio$expected_3m_return_pct, 1)}%"
    ))
  }
}

print("\n=== RATIO ANALYSIS COMPLETE ===")

# 🚀 Exponential Rise Detection Analysis

This section identifies securities experiencing potential exponential growth by analyzing rolling windows of 1-8 weeks.

**Methodology:**
1. For each security, fit a linear model to **log(price)** over time
2. R² > 0.8 indicates strong exponential fit (prices rising exponentially)
3. Calculate for rolling windows of 1, 2, 3, 4, 5, 6, 7, 8 weeks
4. Identify maximum duration of sustained exponential growth
5. Rank securities by a composite score considering both R² quality and duration

In [ ]:
# ============================================================================
# 11. EXPONENTIAL RISE DETECTION ANALYSIS
# ============================================================================
# Methodology: Fit linear model to log(price) ~ time
# R² > 0.8 indicates strong exponential growth
# Rolling windows: 1-8 weeks (5-40 trading days)
# ============================================================================

cat("=" |> rep(80) |> paste(collapse=""), "\n")
cat("🚀 EXPONENTIAL RISE DETECTION ANALYSIS\n")
cat("=" |> rep(80) |> paste(collapse=""), "\n")

# ------------------------------------------------------------------------------
# 11.1 Load All Financial Assets (Stocks, ETFs, Metals)
# ------------------------------------------------------------------------------

# Query YAHOO data (stocks, ETFs, metals)
yahoo_data <- dbGetQuery(con, "
  SELECT 
    series as symbol,
    date,
    value as close
  FROM financial_data 
  WHERE origin = 'YAHOO'
    AND value IS NOT NULL
    AND value > 0
  ORDER BY series, date
") %>%
  mutate(
    date = as.Date(date),
    close = as.numeric(close)
  ) %>%
  filter(!is.na(close) & close > 0)

# Show what securities we have
securities <- yahoo_data %>%
  group_by(symbol) %>%
  summarise(
    n_obs = n(),
    min_date = min(date),
    max_date = max(date),
    .groups = 'drop'
  ) %>%
  filter(n_obs >= 40)  # Need at least 8 weeks of data

cat(glue("\n📊 Loaded {nrow(securities)} securities with sufficient data\n"))
cat(glue("📅 Date range: {min(securities$min_date)} to {max(securities$max_date)}\n"))
cat(glue("📈 Securities: {paste(securities$symbol, collapse=', ')}\n\n"))

In [ ]:
# ------------------------------------------------------------------------------
# 11.2 Define Exponential Detection Function
# ------------------------------------------------------------------------------
# Fits log(price) ~ time and returns R² and other metrics

detect_exponential <- function(prices, dates) {
  # Need at least 5 data points for meaningful fit
  if (length(prices) < 5 || any(prices <= 0)) {
    return(list(r_squared = NA, slope = NA, price_change_pct = NA))
  }
  
  # Create time index (days from start)
  time_idx <- as.numeric(dates - min(dates))
  
  # Fit linear model to log(prices)
  log_prices <- log(prices)
  
  tryCatch({
    model <- lm(log_prices ~ time_idx)
    r_squared <- summary(model)$r.squared
    slope <- coef(model)[2]  # Daily growth rate (in log space)
    
    # Calculate total price change
    price_change_pct <- (tail(prices, 1) / head(prices, 1) - 1) * 100
    
    list(
      r_squared = r_squared,
      slope = slope,
      price_change_pct = price_change_pct
    )
  }, error = function(e) {
    list(r_squared = NA, slope = NA, price_change_pct = NA)
  })
}

# Define rolling window sizes (in trading days, ~5 days per week)
windows <- c(
  "1_week" = 5,
  "2_weeks" = 10,
  "3_weeks" = 15,
  "4_weeks" = 20,
  "5_weeks" = 25,
  "6_weeks" = 30,
  "7_weeks" = 35,
  "8_weeks" = 40
)

cat("✅ Exponential detection function defined\n")
cat(glue("📏 Rolling windows: {paste(names(windows), collapse=', ')}\n"))

In [ ]:
# ------------------------------------------------------------------------------
# 11.3 Calculate Exponential Fit for All Securities & Windows
# ------------------------------------------------------------------------------

# Get the most recent data for analysis
analysis_date <- max(yahoo_data$date)
cat(glue("📅 Analysis date: {analysis_date}\n\n"))

# Function to calculate exponential metrics for one security
calculate_exp_metrics <- function(symbol_data, window_days, analysis_date) {
  # Filter to recent window
  cutoff_date <- analysis_date - window_days
  recent_data <- symbol_data %>%
    filter(date > cutoff_date & date <= analysis_date) %>%
    arrange(date)
  
  if (nrow(recent_data) < 5) {
    return(NULL)
  }
  
  # Detect exponential
  result <- detect_exponential(recent_data$close, recent_data$date)
  
  data.frame(
    r_squared = result$r_squared,
    slope = result$slope,
    price_change_pct = result$price_change_pct,
    n_days = nrow(recent_data),
    start_price = head(recent_data$close, 1),
    end_price = tail(recent_data$close, 1)
  )
}

# Calculate for all securities and all windows
cat("📊 Calculating exponential fits for all securities and windows...\n")

exp_results <- list()

for (sym in unique(yahoo_data$symbol)) {
  sym_data <- yahoo_data %>% filter(symbol == sym)
  
  for (window_name in names(windows)) {
    window_days <- windows[window_name]
    
    metrics <- calculate_exp_metrics(sym_data, window_days, analysis_date)
    
    if (!is.null(metrics)) {
      exp_results[[length(exp_results) + 1]] <- data.frame(
        symbol = sym,
        window = window_name,
        window_days = window_days,
        metrics
      )
    }
  }
}

# Combine all results
exp_df <- bind_rows(exp_results)

cat(glue("✅ Calculated {nrow(exp_df)} exponential fit measurements\n"))
cat(glue("📈 Securities analyzed: {length(unique(exp_df$symbol))}\n"))

In [ ]:
# ------------------------------------------------------------------------------
# 11.4 Identify Securities with Strong Exponential Growth (R² > 0.8)
# ------------------------------------------------------------------------------

R2_THRESHOLD <- 0.8

# Filter to strong exponential fits
strong_exp <- exp_df %>%
  filter(r_squared >= R2_THRESHOLD & slope > 0) %>%  # Only positive growth
  arrange(symbol, window_days)

cat(glue("\n🎯 Securities with R² ≥ {R2_THRESHOLD} (strong exponential growth):\n"))
cat(glue("   Found {nrow(strong_exp)} qualifying measurements\n"))
cat(glue("   Covering {length(unique(strong_exp$symbol))} securities\n\n"))

# Show summary by window
strong_exp_summary <- strong_exp %>%
  group_by(window) %>%
  summarise(
    n_securities = n(),
    avg_r_squared = mean(r_squared, na.rm = TRUE),
    avg_price_change = mean(price_change_pct, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  arrange(factor(window, levels = names(windows)))

cat("📊 Strong exponential fits by window:\n")
print(strong_exp_summary)

In [ ]:
# ------------------------------------------------------------------------------
# 11.5 Calculate Maximum Duration of Sustained Exponential Growth
# ------------------------------------------------------------------------------
# For each security, find the longest consecutive window with R² > 0.8

# For each security, find the MAX duration where R² > 0.8
max_duration <- strong_exp %>%
  group_by(symbol) %>%
  summarise(
    max_window_days = max(window_days),
    max_window = window[which.max(window_days)],
    n_qualifying_windows = n(),
    avg_r_squared = mean(r_squared),
    max_r_squared = max(r_squared),
    total_price_change = price_change_pct[which.max(window_days)],
    .groups = 'drop'
  ) %>%
  arrange(desc(max_window_days), desc(avg_r_squared))

cat("\n📈 Maximum Duration of Sustained Exponential Growth (R² > 0.8):\n")
cat("-" |> rep(70) |> paste(collapse=""), "\n")

# Create a nice display table
display_max_duration <- max_duration %>%
  mutate(
    max_window_weeks = max_window_days / 5,
    avg_r_squared = round(avg_r_squared, 3),
    max_r_squared = round(max_r_squared, 3),
    total_price_change = round(total_price_change, 2)
  ) %>%
  select(
    Symbol = symbol,
    `Max Duration (Weeks)` = max_window_weeks,
    `# Windows R²>0.8` = n_qualifying_windows,
    `Avg R²` = avg_r_squared,
    `Max R²` = max_r_squared,
    `Price Change %` = total_price_change
  )

print(display_max_duration)

cat(glue("\n✅ {nrow(max_duration)} securities show sustained exponential growth\n"))

In [ ]:
# ------------------------------------------------------------------------------
# 11.6 Composite Scoring: R² vs Duration Tradeoff
# ------------------------------------------------------------------------------
# Score = (Normalized R²) * (Normalized Duration) * (1 + log(Price Change + 1))
# This balances quality of fit with sustainability of growth

# Normalize metrics to 0-1 scale
scored_securities <- max_duration %>%
  mutate(
    # Normalize duration (0-1)
    norm_duration = (max_window_days - min(max_window_days)) / 
                    (max(max_window_days) - min(max_window_days) + 1),
    # Normalize R² (0-1, already in 0.8-1.0 range)
    norm_r2 = (avg_r_squared - 0.8) / (1 - 0.8),
    # Price change factor (log scale to handle large values)
    price_factor = ifelse(total_price_change > 0, 
                          1 + log1p(total_price_change) / 5, 
                          0.5),  # Penalize negative returns
    # Composite score
    composite_score = norm_duration * norm_r2 * price_factor * 100,
    # Round for display
    composite_score = round(composite_score, 2)
  ) %>%
  arrange(desc(composite_score))

cat("\n🏆 COMPOSITE RANKING: R² vs Duration Tradeoff\n")
cat("=" |> rep(70) |> paste(collapse=""), "\n")
cat("Score = (Normalized Duration) × (Normalized R²) × (Price Factor)\n\n")

# Display ranking
ranking_display <- scored_securities %>%
  mutate(
    rank = row_number(),
    duration_weeks = max_window_days / 5,
    avg_r_squared = round(avg_r_squared, 3),
    total_price_change = paste0(round(total_price_change, 1), "%")
  ) %>%
  select(
    Rank = rank,
    Symbol = symbol,
    Score = composite_score,
    `Duration (Wks)` = duration_weeks,
    `Avg R²` = avg_r_squared,
    `Price Δ` = total_price_change
  )

print(ranking_display, n = 15)

# Identify top opportunities
top_3 <- head(scored_securities$symbol, 3)
cat(glue("\n🌟 TOP 3 EXPONENTIAL GROWTH OPPORTUNITIES: {paste(top_3, collapse=', ')}\n"))

In [ ]:
# ------------------------------------------------------------------------------
# 11.7 Best Opportunities by Window Duration
# ------------------------------------------------------------------------------
# Find the top exponential growers within each time window

cat("\n📊 BEST EXPONENTIAL OPPORTUNITIES BY TIME WINDOW\n")
cat("=" |> rep(70) |> paste(collapse=""), "\n")

# For each window, show top 5 securities with R² > 0.8
best_by_window <- strong_exp %>%
  group_by(window) %>%
  arrange(desc(r_squared)) %>%
  slice_head(n = 5) %>%
  mutate(
    rank = row_number(),
    r_squared = round(r_squared, 3),
    price_change_pct = round(price_change_pct, 1)
  ) %>%
  select(
    Window = window,
    Rank = rank,
    Symbol = symbol,
    `R²` = r_squared,
    `Price Δ%` = price_change_pct,
    `N Days` = n_days
  ) %>%
  ungroup()

# Display for each window
for (w in names(windows)) {
  window_data <- best_by_window %>% filter(Window == w)
  if (nrow(window_data) > 0) {
    cat(glue("\n🕐 {w} ({windows[w]} trading days):\n"))
    print(window_data %>% select(-Window), n = 5)
  }
}

# Summary table of best performer per window
cat("\n\n📈 TOP PERFORMER PER WINDOW:\n")
cat("-" |> rep(50) |> paste(collapse=""), "\n")

top_per_window <- strong_exp %>%
  group_by(window) %>%
  arrange(desc(r_squared)) %>%
  slice_head(n = 1) %>%
  ungroup() %>%
  arrange(window_days) %>%
  mutate(
    r_squared = round(r_squared, 3),
    price_change_pct = paste0(round(price_change_pct, 1), "%")
  ) %>%
  select(
    Window = window,
    `Top Symbol` = symbol,
    `R²` = r_squared,
    `Price Δ` = price_change_pct
  )

print(top_per_window)

In [ ]:
# ------------------------------------------------------------------------------
# 11.8 Visualization: R² Heatmap Across Securities & Windows
# ------------------------------------------------------------------------------

# Create heatmap data for all securities with at least one R² > 0.8
securities_of_interest <- unique(strong_exp$symbol)

heatmap_data <- exp_df %>%
  filter(symbol %in% securities_of_interest) %>%
  mutate(
    window = factor(window, levels = names(windows)),
    r_squared = pmin(r_squared, 1)  # Cap at 1
  )

# Order securities by max R²
sec_order <- heatmap_data %>%
  group_by(symbol) %>%
  summarise(max_r2 = max(r_squared, na.rm = TRUE), .groups = 'drop') %>%
  arrange(desc(max_r2)) %>%
  pull(symbol)

heatmap_data <- heatmap_data %>%
  mutate(symbol = factor(symbol, levels = sec_order))

# Create heatmap
p_heatmap <- ggplot(heatmap_data, aes(x = window, y = symbol, fill = r_squared)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = ifelse(r_squared >= 0.8, 
                               sprintf("%.2f", r_squared), "")),
            size = 3, color = "white", fontface = "bold") +
  scale_fill_gradient2(
    low = "#2c3e50",
    mid = "#f39c12",
    high = "#27ae60",
    midpoint = 0.8,
    limits = c(0, 1),
    name = "R²"
  ) +
  labs(
    title = "🔥 Exponential Growth Detection Heatmap",
    subtitle = "R² values for log(price) ~ time fit | Green cells indicate strong exponential growth (R² ≥ 0.8)",
    x = "Rolling Window",
    y = "Security"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 10, color = "gray50"),
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text.y = element_text(size = 9),
    panel.grid = element_blank(),
    legend.position = "right"
  )

print(p_heatmap)

# Add to report
add_report_item("plot", p_heatmap, "Exponential Growth Detection Heatmap", "11. Exponential Rise Detection")

In [ ]:
# ------------------------------------------------------------------------------
# 11.9 Visualization: Top Exponential Growers with Fit Lines
# ------------------------------------------------------------------------------

# Get top 6 securities by composite score
top_symbols <- head(scored_securities$symbol, 6)

# Prepare data for plotting (8-week window)
plot_data <- yahoo_data %>%
  filter(symbol %in% top_symbols) %>%
  filter(date > (analysis_date - 40)) %>%  # Last 8 weeks
  arrange(symbol, date) %>%
  group_by(symbol) %>%
  mutate(
    time_idx = as.numeric(date - min(date)),
    log_close = log(close)
  ) %>%
  ungroup()

# Calculate fit lines for each security
fit_lines <- plot_data %>%
  group_by(symbol) %>%
  do({
    model <- lm(log_close ~ time_idx, data = .)
    data.frame(
      time_idx = .$time_idx,
      date = .$date,
      fitted_log = predict(model),
      fitted_price = exp(predict(model))
    )
  }) %>%
  ungroup()

# Create faceted plot
p_top_exp <- ggplot(plot_data, aes(x = date)) +
  geom_line(aes(y = close), color = "#3498db", linewidth = 1, alpha = 0.8) +
  geom_point(aes(y = close), color = "#3498db", size = 1.5, alpha = 0.6) +
  geom_line(data = fit_lines, aes(y = fitted_price), 
            color = "#e74c3c", linewidth = 1.2, linetype = "dashed") +
  facet_wrap(~symbol, scales = "free_y", ncol = 3) +
  labs(
    title = "📈 Top Exponential Growth Candidates",
    subtitle = "Blue: Actual prices | Red dashed: Exponential fit | Last 8 weeks",
    x = "Date",
    y = "Price"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 10, color = "gray50"),
    strip.text = element_text(size = 11, face = "bold"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    panel.grid.minor = element_blank()
  )

print(p_top_exp)

# Add to report
add_report_item("plot", p_top_exp, "Top Exponential Growth Candidates with Fit Lines", "11. Exponential Rise Detection")

In [ ]:
# ------------------------------------------------------------------------------
# 11.10 Visualization: R² vs Duration Tradeoff Scatter
# ------------------------------------------------------------------------------

# Prepare scatter data
scatter_data <- scored_securities %>%
  mutate(
    duration_weeks = max_window_days / 5,
    is_top = symbol %in% top_3
  )

# Create scatter plot
p_scatter <- ggplot(scatter_data, aes(x = duration_weeks, y = avg_r_squared)) +
  geom_point(aes(size = abs(total_price_change), color = composite_score), 
             alpha = 0.8) +
  geom_text(aes(label = symbol), vjust = -1, hjust = 0.5, size = 3.5, 
            fontface = ifelse(scatter_data$is_top, "bold", "plain")) +
  geom_hline(yintercept = 0.9, linetype = "dashed", color = "red", alpha = 0.5) +
  geom_vline(xintercept = 6, linetype = "dashed", color = "blue", alpha = 0.5) +
  annotate("rect", xmin = 6, xmax = 9, ymin = 0.9, ymax = 1.0, 
           fill = "green", alpha = 0.1) +
  scale_color_gradient2(low = "#e74c3c", mid = "#f39c12", high = "#27ae60",
                        midpoint = 50, name = "Score") +
  scale_size_continuous(range = c(3, 12), name = "Price Δ%") +
  labs(
    title = "🎯 R² vs Duration Tradeoff Analysis",
    subtitle = "Green zone: Ideal (high R², long duration) | Size = Price change magnitude",
    x = "Maximum Sustained Duration (Weeks)",
    y = "Average R² (Exponential Fit Quality)"
  ) +
  scale_x_continuous(breaks = 1:8, limits = c(1, 9)) +
  scale_y_continuous(limits = c(0.8, 1.0)) +
  theme_minimal() +
  theme(
    plot.title = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 10, color = "gray50"),
    legend.position = "right"
  )

print(p_scatter)

# Add to report
add_report_item("plot", p_scatter, "R² vs Duration Tradeoff Analysis", "11. Exponential Rise Detection")

In [ ]:
# ------------------------------------------------------------------------------
# 11.11 Analysis by Asset Type (Stocks, ETFs, Metals, Crypto)
# ------------------------------------------------------------------------------

# Classify securities by type
classify_asset <- function(symbol) {
  case_when(
    symbol %in% c("GC=F", "SI=F", "PA=F", "CL=F") ~ "Commodities",
    symbol %in% c("BTC-USD", "ETH-USD") ~ "Crypto",
    str_detect(symbol, "^\\^") ~ "Index",
    str_detect(symbol, "=X$|=F$") ~ "Currency/Futures",
    symbol %in% c("XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", 
                  "XLRE", "XLU", "XLV", "XLY", "TLT", "IEF", "IEI", 
                  "SHV", "SHY", "SGOV", "MBB", "GOVT", "UBT", "UST",
                  "TBF", "TBT", "TBX", "TTT", "PST", "SKF", "SRS",
                  "SMN", "SCC", "SDP", "SIJ", "RXD", "REW", "DUG") ~ "ETF",
    TRUE ~ "Stock"
  )
}

# Add asset type classification
scored_with_type <- scored_securities %>%
  mutate(asset_type = classify_asset(symbol))

# Summary by asset type
cat("\n📊 EXPONENTIAL GROWTH BY ASSET TYPE\n")
cat("=" |> rep(60) |> paste(collapse=""), "\n")

type_summary <- scored_with_type %>%
  group_by(asset_type) %>%
  summarise(
    n_securities = n(),
    avg_duration_weeks = mean(max_window_days / 5),
    avg_r_squared = mean(avg_r_squared),
    avg_price_change = mean(total_price_change),
    avg_score = mean(composite_score),
    top_performer = symbol[which.max(composite_score)],
    .groups = 'drop'
  ) %>%
  arrange(desc(avg_score))

print(type_summary)

# Detailed breakdown
cat("\n\n📈 DETAILED BREAKDOWN BY TYPE:\n")
for (type in unique(scored_with_type$asset_type)) {
  type_data <- scored_with_type %>% 
    filter(asset_type == type) %>%
    arrange(desc(composite_score))
  
  cat(glue("\n--- {type} ({nrow(type_data)} securities) ---\n"))
  cat(glue("   Best: {type_data$symbol[1]} (Score: {type_data$composite_score[1]}, "))
  cat(glue("R²: {round(type_data$avg_r_squared[1], 3)}, "))
  cat(glue("Duration: {type_data$max_window_days[1]/5} wks)\n"))
}

In [ ]:
# ------------------------------------------------------------------------------
# 11.12 Final Summary Table & Investment Signals
# ------------------------------------------------------------------------------

cat("\n" |> rep(2) |> paste(collapse=""))
cat("🏆 " |> rep(25) |> paste(collapse=""), "\n")
cat("    EXPONENTIAL RISE DETECTION - FINAL SUMMARY\n")
cat("🏆 " |> rep(25) |> paste(collapse=""), "\n\n")

cat(glue("📅 Analysis Date: {analysis_date}\n"))
cat(glue("🔍 Securities Analyzed: {length(unique(exp_df$symbol))}\n"))
cat(glue("✅ Securities with R² ≥ 0.8: {nrow(scored_securities)}\n"))
cat(glue("📈 Average Duration of Exponential Growth: {round(mean(scored_securities$max_window_days/5), 1)} weeks\n\n"))

# Create final recommendations table
final_recommendations <- scored_securities %>%
  mutate(
    rank = row_number(),
    asset_type = classify_asset(symbol),
    duration_weeks = max_window_days / 5,
    signal_strength = case_when(
      composite_score >= 75 ~ "🟢 STRONG",
      composite_score >= 50 ~ "🟡 MODERATE",
      composite_score >= 25 ~ "🟠 WEAK",
      TRUE ~ "⚪ MINIMAL"
    ),
    annualized_return = round((1 + total_price_change/100)^(52/duration_weeks) - 1, 2) * 100,
    r_sq_disp = round(avg_r_squared, 3),
    price_disp = round(total_price_change, 1)
  ) %>%
  select(
    Rank = rank,
    Symbol = symbol,
    Type = asset_type,
    Signal = signal_strength,
    Score = composite_score,
    `R²` = r_sq_disp,
    `Duration (Wks)` = duration_weeks,
    `Price Δ%` = price_disp,
    `Ann. Return%` = annualized_return
  )

cat("📋 RANKED EXPONENTIAL GROWTH OPPORTUNITIES:\n")
cat("-" |> rep(90) |> paste(collapse=""), "\n")
print(final_recommendations, n = 15)

# Add to report
add_report_item("table", as.data.frame(final_recommendations), 
                "Exponential Rise Detection - Rankings", "11. Exponential Rise Detection")

In [ ]:
# ------------------------------------------------------------------------------
# 11.13 Visualization: Final Rankings Bar Chart
# ------------------------------------------------------------------------------

# Prepare data for bar chart
bar_data <- scored_securities %>%
  mutate(
    symbol = factor(symbol, levels = rev(symbol)),  # Reverse for horizontal bar
    asset_type = classify_asset(symbol),
    signal_color = case_when(
      composite_score >= 75 ~ "#27ae60",  # Green
      composite_score >= 50 ~ "#f1c40f",  # Yellow
      composite_score >= 25 ~ "#e67e22",  # Orange
      TRUE ~ "#95a5a6"  # Gray
    )
  )

# Create bar chart
p_ranking <- ggplot(bar_data, aes(x = symbol, y = composite_score, fill = signal_color)) +
  geom_col(alpha = 0.9) +
  geom_text(aes(label = paste0(round(composite_score, 1), " | R²:", round(avg_r_squared, 2))),
            hjust = -0.1, size = 3) +
  scale_fill_identity() +
  coord_flip(clip = "off") +
  labs(
    title = "🏆 Exponential Growth Rankings",
    subtitle = "Composite Score (Duration × R² × Price Factor) | Higher = Better Opportunity",
    x = "",
    y = "Composite Score"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 10, color = "gray50"),
    axis.text.y = element_text(size = 10, face = "bold"),
    panel.grid.major.y = element_blank(),
    plot.margin = ggplot2::margin(10, 80, 10, 10)  # Extra right margin for labels
  ) +
  scale_y_continuous(limits = c(0, max(bar_data$composite_score) * 1.3))

print(p_ranking)

# Add to report
add_report_item("plot", p_ranking, "Exponential Growth Rankings - Composite Score", "11. Exponential Rise Detection")

In [ ]:
# ------------------------------------------------------------------------------
# 11.14 Key Takeaways & Actionable Insights
# ------------------------------------------------------------------------------

cat("\n")
cat("=" |> rep(80) |> paste(collapse=""), "\n")
cat("📝 KEY TAKEAWAYS - EXPONENTIAL RISE DETECTION\n")
cat("=" |> rep(80) |> paste(collapse=""), "\n\n")

# Get top performers info
top_performer <- scored_securities[1,]
commodities_leader <- scored_securities %>% 
  filter(classify_asset(symbol) == "Commodities") %>% 
  head(1)
etf_leader <- scored_securities %>% 
  filter(classify_asset(symbol) == "ETF") %>% 
  head(1)

cat("🥇 TOP OVERALL PERFORMER:\n")
cat(glue("   {top_performer$symbol} - Score: {top_performer$composite_score}, "))
cat(glue("R²: {round(top_performer$avg_r_squared, 3)}, "))
cat(glue("Duration: {top_performer$max_window_days/5} weeks, "))
cat(glue("Price Δ: {round(top_performer$total_price_change, 1)}%\n\n"))

cat("🏅 CATEGORY LEADERS:\n")
cat(glue("   • Commodities: {commodities_leader$symbol} (Score: {round(commodities_leader$composite_score, 1)})\n"))
cat(glue("   • ETFs: {etf_leader$symbol} (Score: {round(etf_leader$composite_score, 1)})\n\n"))

# Calculate strong signals
strong_signals <- scored_securities %>% filter(composite_score >= 75)
moderate_signals <- scored_securities %>% filter(composite_score >= 50 & composite_score < 75)

cat("🚦 SIGNAL DISTRIBUTION:\n")
cat(glue("   🟢 STRONG signals (Score ≥ 75): {nrow(strong_signals)} securities\n"))
cat(glue("   🟡 MODERATE signals (50 ≤ Score < 75): {nrow(moderate_signals)} securities\n"))
cat(glue("   ⚪ Other: {nrow(scored_securities) - nrow(strong_signals) - nrow(moderate_signals)} securities\n\n"))

if (nrow(strong_signals) > 0) {
  cat("✨ ACTIONABLE OPPORTUNITIES (Strong Signals):\n")
  for (i in 1:nrow(strong_signals)) {
    s <- strong_signals[i,]
    cat(glue("   {i}. {s$symbol}: {s$max_window_days/5}-week exponential trend, "))
    cat(glue("R²={round(s$avg_r_squared, 2)}, +{round(s$total_price_change, 1)}%\n"))
  }
}

cat("\n")
cat("⚠️  CAVEATS:\n")
cat("   • Past exponential growth does not guarantee future performance\n")
cat("   • High R² indicates good fit, not future prediction\n")
cat("   • Consider fundamental analysis alongside technical signals\n")
cat("   • Use appropriate position sizing and risk management\n")
cat("\n")
cat("=" |> rep(80) |> paste(collapse=""), "\n")
cat(glue("📅 Analysis completed on {Sys.time()}\n"))
cat("=" |> rep(80) |> paste(collapse=""), "\n")

# Construction

In [ ]:

# ============================================================================
# 12. CONSTRUCTION ANALYSIS - DATA EXPLORATION
# ============================================================================
# Permits (issued), Active (under construction / starts), Completed
# DK: Danmarks Statistik  |  US: FRED (Census Bureau)
# ============================================================================

cat("\n")
cat(paste(rep("=", 70), collapse=""), "\n")
cat("   12. CONSTRUCTION ANALYSIS\n")
cat(paste(rep("=", 70), collapse=""), "\n\n")

# ── Discover all construction-related series in the database ────────────────
cat("🔍 Searching for construction data in database...\n\n")

construction_keywords <- c(
  "PERMIT", "HOUST", "UNDCONST", "COMPTS",   # US FRED series
  "BYGB",                                      # DK building stats (Byggestatistik)
  "construction", "byggetilladelse",
  "starts", "completions"
)

pattern <- paste(construction_keywords, collapse = "|")

db_construction <- dbGetQuery(con, sprintf("
  SELECT DISTINCT
    origin,
    series,
    COUNT(*)        AS n_records,
    MIN(date)       AS min_date,
    MAX(date)       AS max_date
  FROM financial_data
  WHERE LOWER(series)  LIKE '%%permit%%'
     OR LOWER(series)  LIKE '%%houst%%'
     OR LOWER(series)  LIKE '%%undcon%%'
     OR LOWER(series)  LIKE '%%compt%%'
     OR LOWER(origin)  LIKE '%%bygb%%'
     OR LOWER(origin)  LIKE '%%construction%%'
     OR LOWER(series)  LIKE '%%bygb%%'
     OR LOWER(series)  LIKE '%%bygge%%'
  GROUP BY origin, series
  ORDER BY origin, series
"))

if (nrow(db_construction) > 0) {
  cat("✅ Construction-related series found:\n\n")
  print(as.data.frame(db_construction))
} else {
  cat("ℹ️  No dedicated construction tags found — broadening search...\n")
  # Fall back: show all distinct origins so we know what's available
  all_origins <- dbGetQuery(con, "
    SELECT
      origin,
      COUNT(DISTINCT series) AS n_series,
      MIN(date) AS min_date,
      MAX(date) AS max_date
    FROM financial_data
    GROUP BY origin
    ORDER BY origin
  ")
  cat("\n📋 All data origins in database:\n")
  print(as.data.frame(all_origins))
}

# ── Also look for any mortgage/interest-rate series for correlation ──────────
cat("\n🔍 Searching for interest rate series...\n")
rate_series <- dbGetQuery(con, "
  SELECT DISTINCT origin, series, COUNT(*) AS n, MIN(date) AS min_date, MAX(date) AS max_date
  FROM financial_data
  WHERE LOWER(series) LIKE '%%rate%%'
     OR LOWER(series) LIKE '%%rentr%%'
     OR LOWER(series) LIKE '%%mortgage%%'
     OR LOWER(series) LIKE '%%rente%%'
     OR LOWER(series) LIKE '%%fedfunds%%'
     OR LOWER(series) LIKE '%%t10%%'
     OR LOWER(series) LIKE '%%gs10%%'
     OR LOWER(series) LIKE '%%-yr%%'
     OR LOWER(series) LIKE '%%yield%%'
     OR LOWER(series) LIKE '%%obs%%'
  GROUP BY origin, series
  ORDER BY origin, series
")

if (nrow(rate_series) > 0) {
  cat("✅ Interest/rate series found:\n")
  print(as.data.frame(rate_series))
} else {
  cat("ℹ️  No explicit rate series found — will derive from existing data.\n")
}


## 12.1 DK Building Permits & Construction Activity

In [ ]:

# ============================================================================
# 12.1  DK BUILDING PERMITS & CONSTRUCTION ACTIVITY
# ============================================================================
# Source priority:
#   1. financial_data table with origin = 'BYGB' (DST Byggestatistik)
#   2. financial_data with BOLIG_DK origin (if construction series exist)
# ============================================================================

print("=== 12.1 DK BUILDING PERMITS & CONSTRUCTION ACTIVITY ===")

# ── Load from database ───────────────────────────────────────────────────────
dk_construction_raw <- dbGetQuery(con, "
  SELECT series, date, value, origin
  FROM financial_data
  WHERE origin LIKE '%BYGB%'
     OR (origin LIKE '%DST%' AND (
           LOWER(series) LIKE '%permit%'
        OR LOWER(series) LIKE '%bygge%'
        OR LOWER(series) LIKE '%tilladel%'
        OR LOWER(series) LIKE '%starts%'
        OR LOWER(series) LIKE '%p%begynd%'
        OR LOWER(series) LIKE '%afslut%'
        OR LOWER(series) LIKE '%fuldfoert%'
        OR LOWER(series) LIKE '%color%'
     ))
  ORDER BY series, date
")

# ── If empty, synthesize from BOLIG_DK (Units listed/sold proxy) ─────────────
if (nrow(dk_construction_raw) == 0) {
  cat("ℹ️  No BYGB origin data found — deriving construction proxies from BOLIG_DK.\n\n")

  dk_construction_raw <- dbGetQuery(con, "
    SELECT series, date, value, origin
    FROM financial_data
    WHERE origin = 'BOLIG_DK'
      AND (   LOWER(series) LIKE '%solgte%'
           OR LOWER(series) LIKE '%udbudte%'
           OR LOWER(series) LIKE '%nytilkomne%'
           OR LOWER(series) LIKE '%nedtagne%'
      )
    ORDER BY series, date
  ")
  dk_src <- "BOLIG_DK (proxied)"
} else {
  dk_src <- "BYGB (DST)"
}

dk_construction_raw <- dk_construction_raw %>%
  mutate(date = as.Date(date), value = as.numeric(value)) %>%
  filter(!is.na(value))

cat(sprintf("📊 DK construction rows loaded (%s): %d\n", dk_src, nrow(dk_construction_raw)))
cat(sprintf("   Series: %d unique\n", n_distinct(dk_construction_raw$series)))
cat(sprintf("   Date range: %s to %s\n\n",
            min(dk_construction_raw$date), max(dk_construction_raw$date)))

# ── Classify each series into permits / active / completed ──────────────────
dk_construction <- dk_construction_raw %>%
  mutate(
    stage = case_when(
      # Permits / Approvals
      grepl("tilladel|permit|approved|godkendt", series, ignore.case = TRUE) ~ "Permits Issued",
      # Active / Under Construction / Starts
      grepl("p.begynd|start|igang|under|aktiv|begun|starts", series, ignore.case = TRUE) ~ "Construction Started",
      # Completions / Ready
      grepl("afslut|fuldfoert|f.rdig|complet|klar|delivered", series, ignore.case = TRUE) ~ "Completions",
      # Proxy: new listings ≈ permits, sold ≈ completions
      grepl("nytilkomne|udbudte", series, ignore.case = TRUE) ~ "Permits Issued",
      grepl("solgte|nedtagne", series, ignore.case = TRUE) ~ "Completions",
      TRUE ~ "Other"
    ),
    property_type = case_when(
      grepl("parcel|house|rækkehus|enfamilie", series, ignore.case = TRUE) ~ "Houses",
      grepl("lejlighed|ejerlejlighed|apartment", series, ignore.case = TRUE) ~ "Apartments",
      grepl("erhverv|commercial", series, ignore.case = TRUE) ~ "Commercial",
      TRUE ~ "All"
    )
  ) %>%
  filter(stage != "Other")

cat("Series classified by stage:\n")
print(dk_construction %>% count(stage, property_type) %>% as.data.frame())

# ── National aggregates per stage ────────────────────────────────────────────
dk_agg <- dk_construction %>%
  group_by(date, stage) %>%
  summarise(total = sum(value, na.rm = TRUE), .groups = "drop") %>%
  arrange(stage, date)

# ── Plot: Permits, Started, Completed over time ──────────────────────────────
p_dk_construction_stages <- dk_agg %>%
  ggplot(aes(x = date, y = total, color = stage)) +
  geom_line(linewidth = 0.8) +
  geom_smooth(method = "loess", se = FALSE, span = 0.25,
              linetype = "dashed", linewidth = 0.5, alpha = 0.7) +
  scale_y_continuous(labels = scales::comma_format()) +
  scale_color_brewer(type = "qual", palette = "Set1") +
  labs(
    title   = "DK Construction Pipeline: Permits → Started → Completed",
    subtitle = paste0("Source: ", dk_src, " | Loess trend dashed"),
    x = NULL, y = "Units", color = "Stage"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_dk_construction_stages)
add_report_item("plot", p_dk_construction_stages,
                "DK Construction Stages", "8. Construction Analysis")

# ── YoY growth rates ─────────────────────────────────────────────────────────
dk_yoy <- dk_agg %>%
  group_by(stage) %>%
  arrange(date) %>%
  mutate(
    # Detect frequency once per group from median spacing (avoids NA from lag())
    n_lag = as.integer(
      ifelse(median(as.numeric(diff(date)), na.rm = TRUE) < 60, 12L, 4L)
    ),
    yoy = (total / lag(total, n_lag[1]) - 1) * 100
  ) %>%
  filter(!is.na(yoy)) %>%
  ungroup()

p_dk_yoy <- dk_yoy %>%
  ggplot(aes(x = date, y = yoy, fill = yoy > 0)) +
  geom_col(show.legend = FALSE, width = 20) +
  geom_hline(yintercept = 0, color = "gray30") +
  facet_wrap(~stage, ncol = 1, scales = "free_y") +
  scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  labs(
    title   = "DK Construction Activity — Year-over-Year Growth",
    subtitle = "Green = expansion | Red = contraction",
    x = NULL, y = "YoY Change (%)"
  ) +
  theme_money_printer_go_brrr()

print(p_dk_yoy)
add_report_item("plot", p_dk_yoy,
                "DK Construction YoY Growth", "8. Construction Analysis")

cat("\n✅ DK construction data loaded and visualised.\n")


## 12.2 US Building Permits & Housing Starts (FRED)

In [ ]:

# ============================================================================
# 12.2  US BUILDING PERMITS & HOUSING STARTS  (FRED)
# ============================================================================
# Primary FRED series:
#   PERMIT     – New Privately-Owned Housing Units Authorized (permits, 1000s)
#   HOUST      – Housing Starts: Total  (1000s of units, seasonally adj.)
#   UNDCONST   – Housing Units Under Construction (1000s)
#   COMPTS     – Housing Completions: Total (1000s, SA)
# Also checked: PERMITNSA, HOUSTNSA, USPRIV, ITB (iShares US Home Construction ETF)
# ============================================================================

print("=== 12.2 US BUILDING PERMITS & HOUSING STARTS (FRED) ===")

# ── Primary candidate series ─────────────────────────────────────────────────
us_fred_series <- c("PERMIT", "PERMITNSA", "HOUST", "HOUSTNSA",
                    "UNDCONST", "COMPTS", "COMPTSNSA")

us_construction_raw <- dbGetQuery(con, sprintf("
  SELECT series, date, value, origin
  FROM financial_data
  WHERE series IN (%s)
  ORDER BY series, date
", paste0("'", us_fred_series, "'", collapse = ",")))

# ── Fall back to any FRED-tagged housing series ──────────────────────────────
if (nrow(us_construction_raw) == 0) {
  cat("ℹ️  FRED primary series not found — searching by origin/name keyword...\n")
  us_construction_raw <- dbGetQuery(con, "
    SELECT series, date, value, origin
    FROM financial_data
    WHERE (origin LIKE '%FRED%' OR origin LIKE '%fred%')
      AND (   LOWER(series) LIKE '%permit%'
           OR LOWER(series) LIKE '%houst%'
           OR LOWER(series) LIKE '%undcon%'
           OR LOWER(series) LIKE '%compt%'
      )
    ORDER BY series, date
  ")
}

# ── If still empty, try housing ETFs as soft proxies ─────────────────────────
use_proxy <- FALSE
if (nrow(us_construction_raw) == 0) {
  cat("ℹ️  No FRED construction series found — using ITB/XHB ETF as soft proxy.\n")
  us_construction_raw <- dbGetQuery(con, "
    SELECT series, date, value, origin
    FROM financial_data
    WHERE series IN ('ITB', 'XHB', 'DHI', 'LEN', 'PHM')
    ORDER BY series, date
  ")
  use_proxy <- TRUE
}

us_construction_raw <- us_construction_raw %>%
  mutate(date = as.Date(date), value = as.numeric(value)) %>%
  filter(!is.na(value))

cat(sprintf("📊 US construction rows: %d | Series: %d | Range: %s – %s\n\n",
            nrow(us_construction_raw),
            n_distinct(us_construction_raw$series),
            min(us_construction_raw$date),
            max(us_construction_raw$date)))

# ── Classify series into pipeline stages ────────────────────────────────────
us_construction <- us_construction_raw %>%
  mutate(
    stage = case_when(
      series %in% c("PERMIT", "PERMITNSA") ~ "Permits Issued",
      series %in% c("HOUST",  "HOUSTNSA")  ~ "Construction Started",
      series == "UNDCONST"                  ~ "Under Construction",
      series %in% c("COMPTS", "COMPTSNSA") ~ "Completions",
      # ETF proxy fallback
      use_proxy ~ paste("Proxy:", series),
      TRUE ~ "Other"
    ),
    seasonally_adj = !grepl("NSA", series),
    unit_label = ifelse(use_proxy, "Price (USD)", "Units (000s)")
  ) %>%
  filter(stage != "Other")

# ── Keep SA series where both SA and NSA exist ───────────────────────────────
us_pipeline <- us_construction %>%
  filter(seasonally_adj | use_proxy) %>%
  select(date, stage, value, unit_label)

cat("US pipeline stages present:\n")
print(us_pipeline %>% distinct(stage) %>% as.data.frame())

# ── Plot 1: Pipeline over time ───────────────────────────────────────────────
y_label <- unique(us_pipeline$unit_label)[1]

p_us_pipeline <- us_pipeline %>%
  ggplot(aes(x = date, y = value, color = stage)) +
  geom_line(linewidth = 0.75, alpha = 0.85) +
  geom_smooth(method = "loess", se = FALSE, span = 0.2,
              linetype = "dashed", linewidth = 0.4) +
  scale_y_continuous(labels = scales::comma_format()) +
  scale_color_brewer(type = "qual", palette = "Set2") +
  labs(
    title    = "US Housing Construction Pipeline",
    subtitle = "FRED: Permits Authorized → Starts → Under Construction → Completions",
    x = NULL, y = y_label, color = "Stage"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_us_pipeline)
add_report_item("plot", p_us_pipeline,
                "US Construction Pipeline", "8. Construction Analysis")

# ── Plot 2: YoY change ───────────────────────────────────────────────────────
us_yoy <- us_pipeline %>%
  group_by(stage) %>%
  arrange(date) %>%
  mutate(
    # Detect frequency once per group from median spacing (avoids NA edge cases)
    n_lag = as.integer(
      ifelse(median(as.numeric(diff(date)), na.rm = TRUE) < 45, 12L, 4L)
    ),
    yoy = (value / lag(value, n_lag[1]) - 1) * 100
  ) %>%
  filter(!is.na(yoy)) %>%
  ungroup()

p_us_yoy <- us_yoy %>%
  ggplot(aes(x = date, y = yoy, fill = yoy > 0)) +
  geom_col(show.legend = FALSE, width = 20) +
  geom_hline(yintercept = 0, color = "gray30") +
  facet_wrap(~stage, ncol = 1, scales = "free_y") +
  scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  labs(
    title    = "US Construction Activity — Year-over-Year Growth",
    subtitle = "Recessions typically show deep red across all stages",
    x = NULL, y = "YoY Change (%)"
  ) +
  theme_money_printer_go_brrr()

print(p_us_yoy)
add_report_item("plot", p_us_yoy,
                "US Construction YoY Growth", "8. Construction Analysis")

# ── Summary stats ─────────────────────────────────────────────────────────────
cat("\n📊 Current levels (most recent observation):\n")
us_pipeline %>%
  group_by(stage) %>%
  filter(date == max(date)) %>%
  select(stage, date, value) %>%
  print()

cat("\n✅ US construction data loaded and visualised.\n")


## 12.3 Time-Lag Analysis: Permits → Starts → Completions

In [ ]:

# ============================================================================
# 12.3  TIME-LAG ANALYSIS  (Cross-Correlation Functions)
# ============================================================================
# Cross-correlate permits vs starts, starts vs completions, permits vs completions
# Peak CCF lag = typical number of months between pipeline stages.
# Applies to both DK (if multi-stage data available) and US.
# ============================================================================

print("=== 12.3 TIME-LAG ANALYSIS ===")

# ── Helper: compute CCF and find dominant lag ────────────────────────────────
analyse_lag <- function(x_series, y_series, x_name, y_name,
                        max_lag = 24, country = "US") {
  # Align on common dates
  df_x <- x_series %>% rename(x = value)
  df_y <- y_series %>% rename(y = value)

  df_xy <- inner_join(df_x, df_y, by = "date") %>%
    arrange(date) %>%
    filter(!is.na(x), !is.na(y), x > 0, y > 0)

  if (nrow(df_xy) < max_lag * 2) {
    cat(sprintf("⚠️  Not enough overlap for %s vs %s (%s). Skipping.\n", x_name, y_name, country))
    return(invisible(NULL))
  }

  # Standardise (z-score) to make CCF unit-free
  df_xy <- df_xy %>%
    mutate(
      x_z = (x - mean(x)) / sd(x),
      y_z = (y - mean(y)) / sd(y)
    )

  # CCF: positive lag = x leads y (x happens BEFORE y)
  ccf_res <- ccf(df_xy$x_z, df_xy$y_z,
                 lag.max = max_lag, plot = FALSE)

  ccf_df <- data.frame(
    lag         = ccf_res$lag[,,1],
    correlation = ccf_res$acf[,,1]
  )

  # Peak positive-lag correlation (x leads y)
  peak_row <- ccf_df %>% filter(lag > 0) %>% slice_max(abs(correlation), n = 1)
  cat(sprintf("  %-18s → %-18s  peak lag = %+.0f months  (r = %.3f)  [%s]\n",
              x_name, y_name, peak_row$lag, peak_row$correlation, country))

  # Plot
  p <- ggplot(ccf_df, aes(x = lag, y = correlation, fill = correlation > 0)) +
    geom_col(show.legend = FALSE, width = 0.8) +
    geom_hline(yintercept = 0, color = "gray30") +
    geom_hline(yintercept = c(-1.96, 1.96) / sqrt(nrow(df_xy)),
               linetype = "dashed", color = "red", alpha = 0.6) +
    geom_vline(xintercept = peak_row$lag, linetype = "dotted",
               color = "#d62728", linewidth = 0.8) +
    scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
    annotate("text",
             x = peak_row$lag + 0.5, y = max(ccf_df$correlation) * 0.95,
             label = sprintf("Peak lag: +%d mo", as.integer(peak_row$lag)),
             hjust = 0, size = 3.2, color = "#d62728") +
    labs(
      title    = sprintf("[%s] CCF: %s → %s", country, x_name, y_name),
      subtitle = sprintf("Positive lag = %s leads %s  |  dashed red = 95%% CI",
                         x_name, y_name),
      x = "Lag (months, positive = x leads y)",
      y = "Cross-Correlation"
    ) +
    theme_money_printer_go_brrr()

  list(plot = p, peak_lag = peak_row$lag, peak_r = peak_row$correlation,
       x_name = x_name, y_name = y_name, country = country)
}

# ── US: pivot to wide, run CCFs ──────────────────────────────────────────────
cat("\n📐 US Pipeline Lags:\n")

us_wide <- us_pipeline %>%
  group_by(date, stage) %>%
  summarise(value = mean(value, na.rm = TRUE), .groups = "drop") %>%
  pivot_wider(names_from = stage, values_from = value)

# Rename for convenience
names(us_wide) <- make.names(names(us_wide))

us_lag_results <- list()

if ("Permits.Issued" %in% names(us_wide) & "Construction.Started" %in% names(us_wide)) {
  r <- analyse_lag(
    us_wide %>% select(date, value = Permits.Issued),
    us_wide %>% select(date, value = Construction.Started),
    "Permits", "Starts", country = "US"
  )
  if (!is.null(r)) {
    us_lag_results[["permits_to_starts"]] <- r
    add_report_item("plot", r$plot, "US: Permits→Starts Lag CCF", "8. Construction Analysis")
    print(r$plot)
  }
}

if ("Construction.Started" %in% names(us_wide) & "Completions" %in% names(us_wide)) {
  r <- analyse_lag(
    us_wide %>% select(date, value = Construction.Started),
    us_wide %>% select(date, value = Completions),
    "Starts", "Completions", country = "US"
  )
  if (!is.null(r)) {
    us_lag_results[["starts_to_completions"]] <- r
    add_report_item("plot", r$plot, "US: Starts→Completions Lag CCF", "8. Construction Analysis")
    print(r$plot)
  }
}

if ("Permits.Issued" %in% names(us_wide) & "Completions" %in% names(us_wide)) {
  r <- analyse_lag(
    us_wide %>% select(date, value = Permits.Issued),
    us_wide %>% select(date, value = Completions),
    "Permits", "Completions", country = "US"
  )
  if (!is.null(r)) {
    us_lag_results[["permits_to_completions"]] <- r
    add_report_item("plot", r$plot, "US: Permits→Completions Lag CCF", "8. Construction Analysis")
    print(r$plot)
  }
}

# ── DK: if multi-stage data available, run CCFs ──────────────────────────────
cat("\n📐 DK Pipeline Lags:\n")

dk_wide <- dk_agg %>%
  pivot_wider(names_from = stage, values_from = total)
names(dk_wide) <- make.names(names(dk_wide))

dk_lag_results <- list()

if ("Permits.Issued" %in% names(dk_wide) & "Completions" %in% names(dk_wide)) {
  r <- analyse_lag(
    dk_wide %>% select(date, value = Permits.Issued),
    dk_wide %>% select(date, value = Completions),
    "Permits", "Completions", country = "DK"
  )
  if (!is.null(r)) {
    dk_lag_results[["permits_to_completions"]] <- r
    add_report_item("plot", r$plot, "DK: Permits→Completions Lag CCF", "8. Construction Analysis")
    print(r$plot)
  }
}

if ("Permits.Issued" %in% names(dk_wide) & "Construction.Started" %in% names(dk_wide)) {
  r <- analyse_lag(
    dk_wide %>% select(date, value = Permits.Issued),
    dk_wide %>% select(date, value = Construction.Started),
    "Permits", "Starts", country = "DK"
  )
  if (!is.null(r)) {
    dk_lag_results[["permits_to_starts"]] <- r
    add_report_item("plot", r$plot, "DK: Permits→Starts Lag CCF", "8. Construction Analysis")
    print(r$plot)
  }
}

# ── DK vs US normalised permits comparison ──────────────────────────────────
cat("\n📊 DK vs US Permits: Normalised Comparison\n")

# Build combined normalised index (permits only)
dk_permits_norm <- dk_agg %>%
  filter(stage == "Permits Issued") %>%
  arrange(date) %>%
  mutate(
    index = 100 * total / mean(total, na.rm = TRUE),
    country = "Denmark"
  ) %>%
  select(date, index, country)

us_permits_norm <- us_pipeline %>%
  filter(stage == "Permits Issued") %>%
  arrange(date) %>%
  mutate(
    index = 100 * value / mean(value, na.rm = TRUE),
    country = "United States"
  ) %>%
  select(date, index, country)

combined_permits <- bind_rows(dk_permits_norm, us_permits_norm) %>%
  filter(!is.na(index))

if (nrow(combined_permits) > 0) {
  p_permits_combined <- combined_permits %>%
    ggplot(aes(x = date, y = index, color = country)) +
    geom_line(linewidth = 0.75, alpha = 0.85) +
    geom_smooth(method = "loess", se = TRUE, span = 0.2, alpha = 0.12, linewidth = 0) +
    geom_hline(yintercept = 100, linetype = "dashed", color = "gray40") +
    scale_color_manual(values = c("Denmark" = "#C0392B", "United States" = "#2E86C1")) +
    labs(
      title    = "Building Permits: Denmark vs USA (Normalised Index)",
      subtitle = "Index = 100 at long-run mean | Shaded: loess confidence band",
      x = NULL, y = "Normalised Permit Index (mean = 100)", color = "Country"
    ) +
    theme_money_printer_go_brrr() +
    theme(legend.position = "bottom")

  print(p_permits_combined)
  add_report_item("plot", p_permits_combined,
                  "DK vs US Permits Normalised", "8. Construction Analysis")
}

# ── Print lag summary table ──────────────────────────────────────────────────
all_lags <- c(us_lag_results, dk_lag_results)
if (length(all_lags) > 0) {
  lag_table <- bind_rows(
    lapply(all_lags, function(x)
      data.frame(Country   = x$country,
                 Lead      = x$x_name,
                 Lagged    = x$y_name,
                 Peak_Lag_Months = as.integer(x$peak_lag),
                 Correlation     = round(x$peak_r, 3)))
  )
  cat("\n📋 Lag Summary Table:\n")
  print(lag_table)
  add_report_item("table", lag_table, "Construction Lag Summary", "8. Construction Analysis")
} else {
  cat("ℹ️  Insufficient multi-stage data to compute lags.\n")
}

cat("\n✅ Time-lag analysis complete.\n")


## 12.4 Correlation: Construction Activity vs Prices & Interest Rates

In [ ]:

# ============================================================================
# 12.4  CORRELATION: CONSTRUCTION vs PRICES & INTEREST RATES
# ============================================================================
# Strategy:
#   A) DK: join construction -> DK house prices (df_bolig / df_national_avg)
#   B) US: join construction -> interest-rate series from DB
#   C) Rolling 12-month correlations + lagged correlations scan
#   D) Heatmap of correlations across series and lag offsets
# ============================================================================

print("=== 12.4 CORRELATION: CONSTRUCTION vs PRICES & INTEREST RATES ===")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  A. DK: Construction Permits vs Property Prices                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

cat("\n── A. DK: Permits vs Property Prices ──\n")

# Use national average house prices already computed in DK housing section
if (exists("df_national_avg") && nrow(df_national_avg) > 0) {
  dk_prices_q <- df_national_avg %>%
    select(date, avg_price) %>%
    filter(!is.na(avg_price))

  dk_permits_q <- dk_agg %>%
    filter(stage == "Permits Issued") %>%
    select(date, permits = total)

  dk_join <- inner_join(dk_prices_q, dk_permits_q, by = "date") %>%
    arrange(date) %>%
    filter(!is.na(permits), permits > 0)

  if (nrow(dk_join) >= 20) {
    # Static correlation
    r_static <- cor(dk_join$permits, dk_join$avg_price, use = "complete.obs")
    cat(sprintf("  Static Pearson r (permits vs house price): %.3f\n", r_static))

    # Scatter
    p_dk_scatter <- dk_join %>%
      ggplot(aes(x = permits, y = avg_price, color = as.numeric(date))) +
      geom_point(size = 2, alpha = 0.7) +
      geom_smooth(method = "lm", se = TRUE, color = "#2E86C1", fill = "#2E86C1", alpha = 0.2) +
      scale_color_viridis_c(option = "viridis",
                            labels = function(x) format(as.Date(x, origin = "1970-01-01"), "%Y")) +
      scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
      labs(
        title    = sprintf("DK: Building Permits vs House Prices  (r = %.2f)", r_static),
        subtitle = "Each point is one quarter; colour = year",
        x = "Building Permits Issued (units)", y = "Avg House Price (DKK/m²)",
        color = "Year"
      ) +
      theme_money_printer_go_brrr()

    print(p_dk_scatter)
    add_report_item("plot", p_dk_scatter,
                    "DK: Permits vs House Prices Scatter", "8. Construction Analysis")

    # Rolling 8-quarter correlation
    dk_join <- dk_join %>%
      mutate(
        roll_cor = zoo::rollapply(
          cbind(permits, avg_price), width = 8,
          FUN = function(m) cor(m[,1], m[,2]),
          fill = NA, align = "right", by.column = FALSE
        )
      )

    p_dk_roll_cor <- dk_join %>%
      filter(!is.na(roll_cor)) %>%
      ggplot(aes(x = date, y = roll_cor)) +
      geom_line(color = "#C0392B", linewidth = 0.8) +
      geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
      geom_hline(yintercept = c(-0.5, 0.5), linetype = "dotted", color = "gray60") +
      scale_y_continuous(limits = c(-1, 1)) +
      labs(
        title    = "DK: Rolling 8-Quarter Correlation — Permits vs House Prices",
        subtitle = "Positive = more permits precede higher prices | Negative = supply dampens prices",
        x = NULL, y = "Rolling Pearson r"
      ) +
      theme_money_printer_go_brrr()

    print(p_dk_roll_cor)
    add_report_item("plot", p_dk_roll_cor,
                    "DK: Rolling Permits-Price Correlation", "8. Construction Analysis")

    # Lagged correlation scan: does permit activity 1-8 qtrs ago predict prices?
    dk_lag_scan <- sapply(0:8, function(k) {
      x_lag <- lag(dk_join$permits, k)
      cor(x_lag, dk_join$avg_price, use = "complete.obs")
    })

    p_dk_lag_scan <- data.frame(lag_qtrs = 0:8, r = dk_lag_scan) %>%
      ggplot(aes(x = lag_qtrs, y = r, fill = r > 0)) +
      geom_col(show.legend = FALSE, width = 0.6) +
      geom_hline(yintercept = 0, color = "gray30") +
      scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
      scale_x_continuous(breaks = 0:8) +
      labs(
        title    = "DK: Lagged Correlation — Permits[t-k] vs Price[t]",
        subtitle = "Optimal lag shows how many quarters ahead permits predict price",
        x = "Lag k (quarters)", y = "Pearson r"
      ) +
      theme_money_printer_go_brrr()

    print(p_dk_lag_scan)
    add_report_item("plot", p_dk_lag_scan,
                    "DK: Permits→Price Lag Scan", "8. Construction Analysis")

    cat(sprintf("  Optimal DK lag (highest |r|): %d quarters\n",
                which.max(abs(dk_lag_scan)) - 1))
  }
} else {
  cat("  ℹ️  df_national_avg not found. Run DK housing section first.\n")
}

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  B. US: Construction vs Interest Rates                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

cat("\n── B. US: Construction vs Interest Rates ──\n")

# Attempt to load interest rate proxy series from DB
interest_candidates <- c("GS10", "GS30", "MORTGAGE30US", "FEDFUNDS",
                          "TB3MS", "DGS10", "DGS30", "MORTGAGE15US")

rates_raw <- dbGetQuery(con, sprintf("
  SELECT series, date, value
  FROM financial_data
  WHERE series IN (%s)
  ORDER BY series, date
", paste0("'", interest_candidates, "'", collapse=",")))

# Fall back: look for anything that sounds like a rate
if (nrow(rates_raw) == 0) {
  rates_raw <- dbGetQuery(con, "
    SELECT series, date, value
    FROM financial_data
    WHERE LOWER(series) LIKE '%gs10%'
       OR LOWER(series) LIKE '%mortgage%'
       OR LOWER(series) LIKE '%fedfunds%'
       OR LOWER(series) LIKE '%rente%'
    ORDER BY series, date
  ")
}

if (nrow(rates_raw) > 0) {
  rates_raw <- rates_raw %>%
    mutate(date = as.Date(date), value = as.numeric(value)) %>%
    filter(!is.na(value))

  cat(sprintf("  ✅ Interest rate series found: %s\n",
              paste(unique(rates_raw$series), collapse=", ")))

  # Use first available series as primary rate
  primary_rate_series <- unique(rates_raw$series)[1]
  rates_monthly <- rates_raw %>%
    filter(series == primary_rate_series) %>%
    select(date, rate = value) %>%
    arrange(date)

  # Join with US permits (monthly)
  us_permits_monthly <- us_pipeline %>%
    filter(stage == "Permits Issued") %>%
    select(date, permits = value) %>%
    arrange(date)

  us_rate_join <- inner_join(us_permits_monthly, rates_monthly,
                              by = "date") %>%
    arrange(date) %>%
    filter(!is.na(permits), permits > 0, !is.na(rate))

  if (nrow(us_rate_join) >= 24) {
    r_pts <- cor(us_rate_join$permits, us_rate_join$rate, use = "complete.obs")
    cat(sprintf("  Static r (US permits vs %s): %.3f\n", primary_rate_series, r_pts))

    # Dual-axis plot: permits (left) and rate (right) over time
    max_permits <- max(us_rate_join$permits, na.rm = TRUE)
    max_rate    <- max(us_rate_join$rate,    na.rm = TRUE)
    scale_factor <- max_permits / max_rate

    p_us_dual <- us_rate_join %>%
      ggplot(aes(x = date)) +
      geom_line(aes(y = permits), color = "#2E86C1", linewidth = 0.7, alpha = 0.85) +
      geom_line(aes(y = rate * scale_factor), color = "#C0392B",
                linewidth = 0.7, alpha = 0.85, linetype = "solid") +
      scale_y_continuous(
        name   = "Building Permits (000s)",
        labels = scales::comma_format(),
        sec.axis = sec_axis(~ . / scale_factor,
                            name = sprintf("%s (%%)", primary_rate_series))
      ) +
      labs(
        title    = sprintf("US Building Permits vs %s Interest Rate", primary_rate_series),
        subtitle = sprintf("Blue = Permits | Red = %s | Static r = %.2f",
                           primary_rate_series, r_pts),
        x = NULL
      ) +
      theme_money_printer_go_brrr() +
      theme(
        axis.title.y.left  = element_text(color = "#2E86C1"),
        axis.title.y.right = element_text(color = "#C0392B")
      )

    print(p_us_dual)
    add_report_item("plot", p_us_dual,
                    sprintf("US Permits vs %s", primary_rate_series),
                    "8. Construction Analysis")

    # Lagged correlation scan
    us_lag_scan <- sapply(0:18, function(k) {
      x_lag <- lag(us_rate_join$rate, k)
      cor(x_lag, us_rate_join$permits, use = "complete.obs")
    })

    p_us_lag_scan <- data.frame(lag_months = 0:18, r = us_lag_scan) %>%
      ggplot(aes(x = lag_months, y = r, fill = r > 0)) +
      geom_col(show.legend = FALSE, width = 0.7) +
      geom_hline(yintercept = 0, color = "gray30") +
      scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
      scale_x_continuous(breaks = 0:18) +
      labs(
        title    = sprintf("US: Lagged Correlation — Rate[t-k] vs Permits[t]  (%s)", primary_rate_series),
        subtitle = "Negative r expected: higher rates → fewer permits, typically 6-18 months later",
        x = "Lag k (months)", y = "Pearson r"
      ) +
      theme_money_printer_go_brrr()

    print(p_us_lag_scan)
    add_report_item("plot", p_us_lag_scan,
                    "US: Rate→Permits Lag Scan", "8. Construction Analysis")

    cat(sprintf("  Optimal lag (highest |r|): %d months  (r = %.3f)\n",
                which.max(abs(us_lag_scan)) - 1,
                us_lag_scan[which.max(abs(us_lag_scan))]))

    # Rolling 24-month correlation
    us_rate_join <- us_rate_join %>%
      mutate(
        roll_cor = zoo::rollapply(
          cbind(permits, rate), width = 24,
          FUN = function(m) cor(m[,1], m[,2]),
          fill = NA, align = "right", by.column = FALSE
        )
      )

    p_us_roll_cor <- us_rate_join %>%
      filter(!is.na(roll_cor)) %>%
      ggplot(aes(x = date, y = roll_cor)) +
      geom_ribbon(aes(ymin = 0, ymax = pmin(roll_cor, 0)), fill = "#d62728", alpha = 0.3) +
      geom_ribbon(aes(ymin = 0, ymax = pmax(roll_cor, 0)), fill = "#2ca02c", alpha = 0.3) +
      geom_line(color = "gray20", linewidth = 0.7) +
      geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
      scale_y_continuous(limits = c(-1, 1)) +
      labs(
        title    = sprintf("US: Rolling 24-Month Correlation — Permits vs %s", primary_rate_series),
        subtitle = "Red fill = negative (rate↑ → permits↓) | Green = positive co-movement",
        x = NULL, y = "Rolling Pearson r"
      ) +
      theme_money_printer_go_brrr()

    print(p_us_roll_cor)
    add_report_item("plot", p_us_roll_cor,
                    "US: Rolling Permits-Rate Correlation", "8. Construction Analysis")
  }
} else {
  cat("  ℹ️  No interest rate series found in DB. Add FRED 'GS10' or 'MORTGAGE30US'.\n")

  # ── Fallback: correlate US permits with Gold price as risk-off proxy ────────
  cat("  Using Gold (GC=F) as risk-sentiment proxy instead...\n")
  if (exists("df_metals")) {
    gold_monthly <- df_metals %>%
      filter(metal == "Gold") %>%
      mutate(month = floor_date(date, "month")) %>%
      group_by(month) %>%
      summarise(gold = mean(close, na.rm = TRUE), .groups = "drop") %>%
      rename(date = month)

    us_permits_monthly <- us_pipeline %>%
      filter(stage == "Permits Issued") %>%
      select(date, permits = value)

    proxy_join <- inner_join(us_permits_monthly, gold_monthly, by = "date") %>%
      filter(!is.na(permits), !is.na(gold))

    if (nrow(proxy_join) >= 24) {
      r_gold <- cor(proxy_join$permits, proxy_join$gold, use = "complete.obs")
      cat(sprintf("  US Permits vs Gold price  r = %.3f\n", r_gold))

      p_gold_proxy <- proxy_join %>%
        ggplot(aes(x = gold, y = permits)) +
        geom_point(aes(color = as.numeric(date)), size = 2, alpha = 0.7) +
        geom_smooth(method = "lm", se = TRUE, color = "#FFD700", fill = "#FFD700", alpha = 0.2) +
        scale_color_viridis_c(labels = function(x) format(as.Date(x, origin="1970-01-01"), "%Y")) +
        scale_y_continuous(labels = scales::comma_format()) +
        scale_x_continuous(labels = scales::dollar_format()) +
        labs(
          title    = sprintf("US Permits vs Gold Price (risk-off proxy)  r = %.2f", r_gold),
          subtitle = "Colour = year — rising gold often coincides with tighter credit",
          x = "Gold Price (USD/oz)", y = "Building Permits (000s)", color = "Year"
        ) +
        theme_money_printer_go_brrr()

      print(p_gold_proxy)
      add_report_item("plot", p_gold_proxy,
                      "US Permits vs Gold (proxy)", "8. Construction Analysis")
    }
  }
}

cat("\n✅ Correlation analysis complete.\n")


## 12.5 Construction Cycle Summary Dashboard

In [ ]:

# ============================================================================
# 12.5  CONSTRUCTION CYCLE SUMMARY DASHBOARD
# ============================================================================

cat("\n")
cat(paste(rep("=", 70), collapse=""), "\n")
cat("   📊 CONSTRUCTION ANALYSIS SUMMARY DASHBOARD\n")
cat(paste(rep("=", 70), collapse=""), "\n\n")

# ── Helper: safe "latest value + YoY" ───────────────────────────────────────
latest_yoy <- function(df, val_col = "value", periods = 12) {
  df <- df %>% arrange(date)
  if (nrow(df) < periods + 1) return(list(latest = NA_real_, yoy = NA_real_, date = NA))
  latest_val <- tail(df[[val_col]], 1)
  prior_val  <- df[[val_col]][nrow(df) - periods]
  yoy        <- (latest_val / prior_val - 1) * 100
  list(latest = latest_val, yoy = yoy, date = tail(df$date, 1))
}

# ── DK stats ─────────────────────────────────────────────────────────────────
dk_perm_stats <- dk_agg %>% filter(stage == "Permits Issued") %>%
  latest_yoy("total", periods = if (min(diff(dk_agg$date[dk_agg$stage=="Permits Issued"])) < 40) 12 else 4)
dk_comp_stats <- dk_agg %>% filter(stage == "Completions") %>%
  latest_yoy("total", periods = if (min(diff(dk_agg$date[dk_agg$stage=="Completions"])) < 40) 12 else 4)

# ── US stats ─────────────────────────────────────────────────────────────────
us_perm_stats <- us_pipeline %>% filter(stage == "Permits Issued") %>%
  latest_yoy("value", periods = 12)
us_start_stats <- us_pipeline %>% filter(stage == "Construction Started") %>%
  latest_yoy("value", periods = 12)
us_comp_stats  <- us_pipeline %>% filter(stage == "Completions") %>%
  latest_yoy("value", periods = 12)
us_under_stats <- us_pipeline %>% filter(stage == "Under Construction") %>%
  latest_yoy("value", periods = 12)

# ── Print text summary ───────────────────────────────────────────────────────
fmt_stat <- function(stat, unit = "", decimals = 1) {
  if (is.na(stat$latest)) return("N/A")
  yoy_str <- ifelse(is.na(stat$yoy), "",
                    sprintf("  YoY: %+.1f%%", stat$yoy))
  sprintf("%s%s%s", scales::comma(round(stat$latest, decimals)), unit, yoy_str)
}

cat("🇩🇰 DENMARK\n")
cat(paste(rep("-", 40), collapse=""), "\n")
cat(sprintf("  Permits Issued (latest):    %s\n", fmt_stat(dk_perm_stats)))
cat(sprintf("  Completions   (latest):    %s\n", fmt_stat(dk_comp_stats)))

cat("\n🇺🇸 UNITED STATES  (000s of units, SA)\n")
cat(paste(rep("-", 40), collapse=""), "\n")
cat(sprintf("  Permits Issued (latest):    %s\n", fmt_stat(us_perm_stats)))
cat(sprintf("  Housing Starts (latest):    %s\n", fmt_stat(us_start_stats)))
cat(sprintf("  Under Construction:         %s\n", fmt_stat(us_under_stats)))
cat(sprintf("  Completions   (latest):     %s\n", fmt_stat(us_comp_stats)))

# ── Structured summary table ──────────────────────────────────────────────────
build_row <- function(country, stage, stat, unit = "") {
  data.frame(
    Country       = country,
    Stage         = stage,
    Latest        = ifelse(is.na(stat$latest), NA_real_, round(stat$latest, 1)),
    Unit          = unit,
    YoY_pct       = ifelse(is.na(stat$yoy), NA_real_, round(stat$yoy, 1)),
    As_of_date    = ifelse(is.na(stat$date), NA_character_, as.character(stat$date)),
    stringsAsFactors = FALSE
  )
}

construction_summary_tbl <- bind_rows(
  build_row("DK", "Permits Issued",       dk_perm_stats,  "units"),
  build_row("DK", "Completions",          dk_comp_stats,  "units"),
  build_row("US", "Permits Issued",       us_perm_stats,  "000s"),
  build_row("US", "Construction Started", us_start_stats, "000s"),
  build_row("US", "Under Construction",   us_under_stats, "000s"),
  build_row("US", "Completions",          us_comp_stats,  "000s")
)

cat("\n📋 Structured Summary:\n")
print(construction_summary_tbl, row.names = FALSE)
add_report_item("table", construction_summary_tbl,
                "Construction Market Summary", "8. Construction Analysis")

# ── Combined 4-panel dashboard plot ─────────────────────────────────────────
# Panel rows: DK (top), US (bottom)  |  Panel cols: Index (left), YoY (right)

## DK indexed
dk_index_df <- dk_agg %>%
  group_by(stage) %>%
  arrange(date) %>%
  mutate(index = 100 * total / mean(total, na.rm = TRUE)) %>%
  ungroup()

p_dk_idx <- dk_index_df %>%
  ggplot(aes(x = date, y = index, color = stage)) +
  geom_line(linewidth = 0.75) +
  geom_hline(yintercept = 100, linetype = "dashed", color = "gray40") +
  scale_color_brewer(palette = "Set1") +
  labs(title = "DK: Normalised Index (mean = 100)",
       x = NULL, y = "Index", color = "Stage") +
  theme_money_printer_go_brrr() + theme(legend.position = "bottom", legend.text = element_text(size = 7))

## DK YoY
p_dk_yoy2 <- dk_yoy %>%
  ggplot(aes(x = date, y = yoy, color = stage)) +
  geom_line(linewidth = 0.65, alpha = 0.8) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
  scale_color_brewer(palette = "Set1") +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  labs(title = "DK: YoY Growth (%)",
       x = NULL, y = "YoY %", color = "Stage") +
  theme_money_printer_go_brrr() + theme(legend.position = "none")

## US indexed
us_index_df <- us_pipeline %>%
  group_by(stage) %>%
  arrange(date) %>%
  mutate(index = 100 * value / mean(value, na.rm = TRUE)) %>%
  ungroup()

p_us_idx <- us_index_df %>%
  ggplot(aes(x = date, y = index, color = stage)) +
  geom_line(linewidth = 0.75) +
  geom_hline(yintercept = 100, linetype = "dashed", color = "gray40") +
  scale_color_brewer(palette = "Set2") +
  labs(title = "US: Normalised Index (mean = 100)",
       x = NULL, y = "Index", color = "Stage") +
  theme_money_printer_go_brrr() + theme(legend.position = "bottom", legend.text = element_text(size = 7))

## US YoY
p_us_yoy2 <- us_yoy %>%
  ggplot(aes(x = date, y = yoy, color = stage)) +
  geom_line(linewidth = 0.65, alpha = 0.8) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
  scale_color_brewer(palette = "Set2") +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  labs(title = "US: YoY Growth (%)",
       x = NULL, y = "YoY %", color = "Stage") +
  theme_money_printer_go_brrr() + theme(legend.position = "none")

p_dashboard <- gridExtra::grid.arrange(
  p_dk_idx, p_dk_yoy2,
  p_us_idx, p_us_yoy2,
  ncol   = 2,
  top    = grid::textGrob(
    "Construction Activity Dashboard — DK & US",
    gp = grid::gpar(fontsize = 14, fontface = "bold")
  )
)

add_report_item("plot", p_dashboard,
                "Construction Dashboard (DK + US)", "8. Construction Analysis")

# ── Key takeaways ─────────────────────────────────────────────────────────────
cat("\n")
cat(paste(rep("=", 70), collapse=""), "\n")
cat("📝 KEY TAKEAWAYS — CONSTRUCTION ANALYSIS\n")
cat(paste(rep("=", 70), collapse=""), "\n\n")
cat("🔑 Typical pipeline lags (literature benchmarks):\n")
cat("   US: Permits → Starts:       ~1-3 months\n")
cat("   US: Starts → Completions:   ~6-8 months (single-family)\n")
cat("   US: Permits → Completions:  ~8-12 months\n")
cat("   DK: Permits → Completions:  ~12-18 months\n\n")
cat("📈 Construction & Interest Rate:\n")
cat("   Higher rates → higher financing cost → fewer permits (typically r < 0)\n")
cat("   Lag: rate changes precede permit changes by ~6-12 months\n\n")
cat("🏠 Construction & House Prices:\n")
cat("   Low supply (few completions) → upward price pressure\n")
cat("   Permits signal future supply; high permits → price moderation 1-2y out\n\n")
cat("⚠️  CAVEATS:\n")
cat("   • Correlations are not structural causal effects\n")
cat("   • DK data may use BOLIG_DK sales volume proxy if BYGB unavailable\n")
cat("   • US NSA series show strong seasonality; SA series preferred\n")
cat("\n")
cat(paste(rep("=", 70), collapse=""), "\n")
cat(sprintf("📅 Construction analysis completed: %s\n", Sys.time()))
cat(paste(rep("=", 70), collapse=""), "\n")


# Housing Data (DK)

In [ ]:
# ============================================================================
# DANISH HOUSING MARKET ANALYSIS
# ============================================================================
# Data: BOLIG_DK from FinansDanmark property statistics
# Tables: BM011 (prices), BM021 (market movements), BM031 (sale times)
#         UDB010 (properties on market), UDB020 (prices monthly), UDB030 (times monthly)
# ============================================================================

print("\n" %+% paste(rep("=", 70), collapse = "") %+% "\n")
print("    DANISH HOUSING MARKET ANALYSIS    ")
print(paste(rep("=", 70), collapse = "") %+% "\n")

# Load Danish housing data
df_bolig <- dbGetQuery(con, "
  SELECT * FROM financial_data 
  WHERE origin = 'BOLIG_DK'
  ORDER BY series, date
")

cat("\n📊 Danish Housing Data Summary:\n")
cat("   Total records:", nrow(df_bolig), "\n")
cat("   Unique series:", n_distinct(df_bolig$series), "\n")
cat("   Date range:", as.character(min(df_bolig$date)), "to", as.character(max(df_bolig$date)), "\n")

# Parse series names to extract components
df_bolig <- df_bolig %>%
  mutate(
    # Extract table code
    table_code = str_extract(series, "^[A-Z0-9]+"),
    
    # Clean series name for display
    series_clean = str_replace_all(series, "_", " "),
    
    # Extract postcode if present (4 digits)
    postcode = str_extract(series, "\\d{4}"),
    
    # Identify property type
    property_type = case_when(
      grepl("Parcel|r_kkehus|rækkehus", series, ignore.case = TRUE) ~ "Houses",
      grepl("Ejerlejlighed|lejlighed", series, ignore.case = TRUE) ~ "Apartments",
      TRUE ~ "Other"
    ),
    
    # Identify metric type
    metric_type = case_when(
      grepl("UDB020|BM011", series) & grepl("Udbudspris", series) ~ "Listing Price",
      grepl("UDB020|BM011", series) & grepl("Nedtagningspris|Salgspris", series) ~ "Sold Price",
      grepl("UDB020|BM011", series) & grepl("F_rste_udbudspris|Første", series) ~ "Initial List Price",
      grepl("UDB010", series) & grepl("Udbudte", series) ~ "Properties Listed",
      grepl("UDB010", series) & grepl("Nedtagne|Solgte", series) ~ "Properties Sold",
      grepl("UDB030|BM031", series) & grepl("Udbudstid|Liggetid", series) ~ "Time on Market",
      grepl("BM021", series) & grepl("Solgte", series) ~ "Sales Count",
      grepl("BM021", series) & grepl("Udbudte|markedet", series) ~ "Listings Count",
      grepl("UL30", series) ~ "Mortgage Lending",
      TRUE ~ "Other"
    ),
    
    # Extract region if present
    region = case_when(
      grepl("Hovedstaden", series) ~ "Capital Region",
      grepl("Sjælland", series) ~ "Zealand",
      grepl("Syddanmark", series) ~ "Southern Denmark",
      grepl("Midtjylland", series) ~ "Central Jutland", 
      grepl("Nordjylland", series) ~ "Northern Jutland",
      grepl("Hele_landet", series) ~ "All Denmark",
      !is.na(postcode) ~ paste0("Postcode ", postcode),
      TRUE ~ "Unknown"
    )
  )

# Summary by table
table_summary <- df_bolig %>%
  group_by(table_code) %>%
  summarise(
    n_series = n_distinct(series),
    n_records = n(),
    min_date = min(date),
    max_date = max(date),
    .groups = "drop"
  )

cat("\n📋 Data by Table:\n")
print(as.data.frame(table_summary))

## 9.1 Property Prices Over Time - Regional Overview

In [ ]:
# ============================================================================
# REGIONAL PRICE TRENDS - MONTHLY DATA (UDB020)
# ============================================================================

print("=== REGIONAL PRICE TRENDS ===")

# Get regional price data (monthly UDB020)
df_regional_prices <- df_bolig %>%
  filter(table_code == "UDB020") %>%
  filter(grepl("Udbudspriser", series)) %>%
  filter(grepl("Region|Hele_landet", series))

cat("\nRegional price series found:", n_distinct(df_regional_prices$series), "\n")

# Create faceted plot by property type and region
p_regional_prices <- df_regional_prices %>%
  ggplot(aes(x = date, y = value, color = region)) +
  geom_line(size = 0.8, alpha = 0.8) +
  facet_wrap(~ property_type, scales = "free_y", ncol = 1) +
  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
  scale_color_brewer(type = "qual", palette = "Set1") +
  labs(
    title = "Danish Property Listing Prices by Region",
    subtitle = "Monthly data from FinansDanmark (DKK per m²)",
    x = NULL,
    y = "Price (DKK/m²)",
    color = "Region"
  ) +
  theme_money_printer_go_brrr() +
  theme(
    legend.position = "bottom",
    strip.text = element_text(face = "bold", size = 11)
  )

print(p_regional_prices)

# Add to report
add_report_item("plot", p_regional_prices, "Regional Property Prices", "6. Danish Housing Market")

## 9.2 Property Prices by Postcode (Quarterly BM011)

In [ ]:
# ============================================================================
# POSTCODE-LEVEL PRICE ANALYSIS (BM011 - Quarterly)
# ============================================================================

print("=== POSTCODE PRICE ANALYSIS ===")

# Get postcode price data
df_postcode_prices <- df_bolig %>%
  filter(table_code == "BM011") %>%
  filter(!is.na(postcode)) %>%
  filter(grepl("F_rste_udbudspris|Første", series))  # Initial listing prices

# Calculate summary stats by postcode
postcode_summary <- df_postcode_prices %>%
  group_by(postcode, property_type) %>%
  summarise(
    n_quarters = n(),
    min_price = min(value, na.rm = TRUE),
    max_price = max(value, na.rm = TRUE),
    latest_price = last(value),
    first_price = first(value),
    pct_change = 100 * (latest_price / first_price - 1),
    .groups = "drop"
  ) %>%
  arrange(desc(latest_price))

cat("\nTop 10 most expensive postcodes (Houses):\n")
print(head(postcode_summary %>% filter(property_type == "Houses"), 10))

# Identify top 10 postcodes by latest price for visualization
top_postcodes <- postcode_summary %>%
  filter(property_type == "Houses") %>%
  arrange(desc(latest_price)) %>%
  head(10) %>%
  pull(postcode)

# Plot price evolution for top postcodes
p_postcode_prices <- df_postcode_prices %>%
  filter(postcode %in% top_postcodes) %>%
  filter(property_type == "Houses") %>%
  mutate(postcode_label = paste0(postcode)) %>%
  ggplot(aes(x = date, y = value, color = postcode_label)) +
  geom_line(size = 0.9, alpha = 0.85) +
  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
  scale_color_viridis_d(option = "turbo") +
  labs(
    title = "House Prices in Top 10 Most Expensive Postcodes",
    subtitle = "Quarterly initial listing prices (DKK per m²) - BM011",
    x = NULL,
    y = "Price (DKK/m²)",
    color = "Postcode"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "right")

print(p_postcode_prices)
add_report_item("plot", p_postcode_prices, "House Prices - Top 10 Postcodes", "6. Danish Housing Market")

## 9.3 Focus: 2800 Kgs. Lyngby - Comprehensive Analysis

In [ ]:
# ============================================================================
# FOCUS AREA: 2800 KGS. LYNGBY - COMPREHENSIVE ANALYSIS
# ============================================================================

print("=== FOCUS: 2800 KGS. LYNGBY ===")

FOCUS_POSTCODE <- "2800"

# Get all data for Lyngby
df_lyngby <- df_bolig %>%
  filter(postcode == FOCUS_POSTCODE)

cat("\n📍 Postcode 2800 Kgs. Lyngby:\n")
cat("   Total records:", nrow(df_lyngby), "\n")
cat("   Unique series:", n_distinct(df_lyngby$series), "\n")
cat("   Tables available:", paste(unique(df_lyngby$table_code), collapse = ", "), "\n")

# 1. PRICE EVOLUTION
df_lyngby_prices <- df_lyngby %>%
  filter(table_code == "BM011") %>%
  mutate(
    price_type = case_when(
      grepl("F_rste_udbudspris|Første", series) ~ "Initial Listing",
      grepl("Salgspris|salgspris", series) ~ "Final Sale",
      grepl("Nedtagnings|nedtagnings", series) ~ "Delisting",
      TRUE ~ "Other"
    )
  ) %>%
  filter(price_type != "Other")

# Calculate key metrics
lyngby_price_stats <- df_lyngby_prices %>%
  filter(property_type == "Houses", price_type == "Initial Listing") %>%
  summarise(
    first_date = min(date),
    last_date = max(date),
    first_price = first(value),
    latest_price = last(value),
    min_price = min(value),
    max_price = max(value),
    cagr = (latest_price / first_price)^(1/as.numeric(difftime(max(date), min(date), units = "days")/365.25)) - 1
  )

cat("\n📈 2800 Lyngby House Price Statistics:\n")
cat("   Period:", as.character(lyngby_price_stats$first_date), "to", as.character(lyngby_price_stats$last_date), "\n")
cat("   First recorded price:", scales::comma(lyngby_price_stats$first_price), "DKK/m²\n")
cat("   Latest price:", scales::comma(lyngby_price_stats$latest_price), "DKK/m²\n")
cat("   All-time high:", scales::comma(lyngby_price_stats$max_price), "DKK/m²\n")
cat("   CAGR:", scales::percent(lyngby_price_stats$cagr, accuracy = 0.1), "\n")

# Create Lyngby price evolution plot
p_lyngby_prices <- df_lyngby_prices %>%
  filter(property_type %in% c("Houses", "Apartments")) %>%
  ggplot(aes(x = date, y = value, color = price_type, linetype = property_type)) +
  geom_line(size = 1) +
  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
  scale_color_manual(values = c(
    "Initial Listing" = "#1f77b4",
    "Final Sale" = "#2ca02c",
    "Delisting" = "#d62728"
  )) +
  labs(
    title = "2800 Kgs. Lyngby - Property Price Evolution",
    subtitle = "Quarterly data from FinansDanmark (BM011)",
    x = NULL,
    y = "Price (DKK/m²)",
    color = "Price Type",
    linetype = "Property Type"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_lyngby_prices)
add_report_item("plot", p_lyngby_prices, "2800 Lyngby - Price Evolution", "6. Danish Housing Market")

In [ ]:
# ============================================================================
# LYNGBY: SALES ACTIVITY & TIME ON MARKET
# ============================================================================

# 2. MARKET ACTIVITY (Sales count from BM021)
df_lyngby_sales <- df_lyngby %>%
  filter(table_code == "BM021") %>%
  filter(grepl("Solgte", series))

# 3. TIME ON MARKET (BM031)
df_lyngby_times <- df_lyngby %>%
  filter(table_code == "BM031")

# Combined market activity plot
p_lyngby_activity <- df_lyngby_sales %>%
  ggplot(aes(x = date, y = value, fill = property_type)) +
  geom_col(position = "dodge", alpha = 0.8) +
  scale_fill_manual(values = c("Houses" = "#2ca02c", "Apartments" = "#1f77b4")) +
  labs(
    title = "2800 Kgs. Lyngby - Quarterly Sales Volume",
    subtitle = "Number of properties sold (BM021)",
    x = NULL,
    y = "Properties Sold",
    fill = "Property Type"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_lyngby_activity)
add_report_item("plot", p_lyngby_activity, "2800 Lyngby - Sales Volume", "6. Danish Housing Market")

# Time on market plot
if (nrow(df_lyngby_times) > 0) {
  p_lyngby_time_on_market <- df_lyngby_times %>%
    ggplot(aes(x = date, y = value, color = property_type)) +
    geom_line(size = 1) +
    geom_smooth(method = "loess", se = TRUE, alpha = 0.2, span = 0.3) +
    scale_color_manual(values = c("Houses" = "#2ca02c", "Apartments" = "#1f77b4")) +
    labs(
      title = "2800 Kgs. Lyngby - Average Days on Market",
      subtitle = "Time from listing to sale (BM031)",
      x = NULL,
      y = "Days on Market",
      color = "Property Type"
    ) +
    theme_money_printer_go_brrr() +
    theme(legend.position = "bottom")
  
  print(p_lyngby_time_on_market)
  add_report_item("plot", p_lyngby_time_on_market, "2800 Lyngby - Time on Market", "6. Danish Housing Market")
}

## 9.4 Listing vs Sold Price Gap Analysis

In [ ]:
# ============================================================================
# LISTING PRICE VS SOLD PRICE GAP ANALYSIS
# ============================================================================
# The gap between listing and sold prices indicates market conditions:
# - Negative gap = sellers having to reduce prices (buyer's market)
# - Positive gap = properties selling above listing (seller's market)
# ============================================================================

print("=== LISTING VS SOLD PRICE GAP ANALYSIS ===")

# Get regional data with both listing and sale prices (UDB020)
df_price_types <- df_bolig %>%
  filter(table_code == "UDB020") %>%
  filter(grepl("Region|Hele_landet", series)) %>%
  mutate(
    price_type = case_when(
      grepl("Udbudspriser", series) ~ "Listing",
      grepl("Nedtagningspriser", series) ~ "Sold",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(price_type))

# Pivot to calculate gap
df_price_gap <- df_price_types %>%
  select(date, region, property_type, price_type, value) %>%
  pivot_wider(
    names_from = price_type,
    values_from = value
  ) %>%
  filter(!is.na(Listing) & !is.na(Sold)) %>%
  mutate(
    price_gap = Sold - Listing,
    price_gap_pct = 100 * (Sold / Listing - 1)
  )

cat("\nPrice Gap Statistics (All Regions):\n")
gap_summary <- df_price_gap %>%
  group_by(property_type) %>%
  summarise(
    avg_gap_pct = mean(price_gap_pct, na.rm = TRUE),
    min_gap_pct = min(price_gap_pct, na.rm = TRUE),
    max_gap_pct = max(price_gap_pct, na.rm = TRUE),
    latest_gap_pct = last(price_gap_pct),
    .groups = "drop"
  )
print(as.data.frame(gap_summary))

# Plot price gap over time
p_price_gap <- df_price_gap %>%
  filter(region == "Capital Region") %>%
  ggplot(aes(x = date, y = price_gap_pct, color = property_type)) +
  geom_line(size = 0.8) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50") +
  geom_smooth(method = "loess", se = TRUE, alpha = 0.2, span = 0.2) +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  scale_color_manual(values = c("Houses" = "#2ca02c", "Apartments" = "#1f77b4")) +
  annotate("rect", xmin = min(df_price_gap$date), xmax = max(df_price_gap$date),
           ymin = 0, ymax = Inf, fill = "green", alpha = 0.05) +
  annotate("rect", xmin = min(df_price_gap$date), xmax = max(df_price_gap$date),
           ymin = -Inf, ymax = 0, fill = "red", alpha = 0.05) +
  annotate("text", x = min(df_price_gap$date) + 365, y = 3, 
           label = "Seller's Market", color = "darkgreen", size = 3, hjust = 0) +
  annotate("text", x = min(df_price_gap$date) + 365, y = -3, 
           label = "Buyer's Market", color = "darkred", size = 3, hjust = 0) +
  labs(
    title = "Capital Region: Price Gap (Sold vs Listed)",
    subtitle = "Negative = buyers getting discounts | Positive = sellers getting premiums",
    x = NULL,
    y = "Price Gap (%)",
    color = "Property Type"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_price_gap)
add_report_item("plot", p_price_gap, "Listing vs Sold Price Gap - Capital Region", "6. Danish Housing Market")

In [ ]:
# ============================================================================
# PRICE GAP HEATMAP BY REGION
# ============================================================================

# Prepare data for heatmap - average gap by year and region
df_gap_heatmap <- df_price_gap %>%
  mutate(year = year(date)) %>%
  filter(year >= 2010) %>%
  group_by(year, region, property_type) %>%
  summarise(
    avg_gap_pct = mean(price_gap_pct, na.rm = TRUE),
    .groups = "drop"
  )

# Create heatmap for Houses
p_gap_heatmap <- df_gap_heatmap %>%
  filter(property_type == "Houses") %>%
  ggplot(aes(x = year, y = region, fill = avg_gap_pct)) +
  geom_tile(color = "white") +
  geom_text(aes(label = sprintf("%.1f%%", avg_gap_pct)), size = 2.5) +
  scale_fill_gradient2(
    low = "#d73027", mid = "white", high = "#1a9850",
    midpoint = 0,
    name = "Gap %"
  ) +
  scale_x_continuous(breaks = seq(2010, 2026, 2)) +
  labs(
    title = "House Price Gap (Sold vs Listed) by Region and Year",
    subtitle = "Red = buyer's market (discounts) | Green = seller's market (premiums)",
    x = "Year",
    y = NULL
  ) +
  theme_money_printer_go_brrr() +
  theme(
    axis.text.x = element_text(angle = 0),
    legend.position = "right"
  )

print(p_gap_heatmap)
add_report_item("plot", p_gap_heatmap, "Price Gap Heatmap - Regional", "6. Danish Housing Market")

## 9.5 Time on Market Analysis

In [ ]:
# ============================================================================
# TIME ON MARKET ANALYSIS (UDB030 - Monthly)
# ============================================================================

print("=== TIME ON MARKET ANALYSIS ===")

# Get time on market data
df_time_on_market <- df_bolig %>%
  filter(table_code == "UDB030") %>%
  filter(grepl("Liggetider|liggetider", series)) %>%  # Days on market until sold
  filter(grepl("Region|Hele_landet", series))

cat("\nTime on market series:", n_distinct(df_time_on_market$series), "\n")

# Plot time on market by region
p_time_regional <- df_time_on_market %>%
  filter(property_type == "Houses") %>%
  ggplot(aes(x = date, y = value, color = region)) +
  geom_line(size = 0.7, alpha = 0.8) +
  scale_color_brewer(type = "qual", palette = "Set1") +
  labs(
    title = "Houses: Average Days on Market by Region",
    subtitle = "Monthly data from FinansDanmark (UDB030)",
    x = NULL,
    y = "Days on Market",
    color = "Region"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_time_regional)
add_report_item("plot", p_time_regional, "Days on Market - Regional", "6. Danish Housing Market")

# Compare houses vs apartments
p_time_comparison <- df_time_on_market %>%
  filter(region == "Capital Region") %>%
  ggplot(aes(x = date, y = value, color = property_type)) +
  geom_line(size = 1) +
  geom_smooth(method = "loess", se = TRUE, alpha = 0.15, span = 0.2) +
  scale_color_manual(values = c("Houses" = "#2ca02c", "Apartments" = "#1f77b4")) +
  labs(
    title = "Capital Region: Days on Market Comparison",
    subtitle = "Houses vs Apartments - with trend smoothing",
    x = NULL,
    y = "Days on Market",
    color = "Property Type"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "bottom")

print(p_time_comparison)
add_report_item("plot", p_time_comparison, "Days on Market - Houses vs Apartments", "6. Danish Housing Market")

## 9.6 Seasonality Analysis & Price Projections

In [ ]:
# ============================================================================
# SEASONALITY ANALYSIS & PRICE PROJECTIONS
# ============================================================================

print("=== SEASONALITY ANALYSIS ===")

# Use monthly regional data for seasonality (UDB020)
df_seasonality <- df_bolig %>%
  filter(table_code == "UDB020") %>%
  filter(grepl("Udbudspriser", series)) %>%
  filter(region == "Capital Region") %>%
  filter(property_type == "Houses") %>%
  arrange(date) %>%
  mutate(
    month = month(date),
    month_name = month(date, label = TRUE),
    year = year(date)
  )

# Calculate monthly seasonality factors
monthly_avg <- df_seasonality %>%
  group_by(month, month_name) %>%
  summarise(
    avg_price = mean(value, na.rm = TRUE),
    n_obs = n(),
    .groups = "drop"
  ) %>%
  mutate(
    overall_avg = mean(avg_price),
    seasonal_factor = avg_price / overall_avg,
    seasonal_pct = 100 * (seasonal_factor - 1)
  )

cat("\nMonthly Seasonality Factors (Capital Region - Houses):\n")
print(as.data.frame(monthly_avg))

# Plot seasonality
p_seasonality <- monthly_avg %>%
  ggplot(aes(x = month_name, y = seasonal_pct, fill = seasonal_pct > 0)) +
  geom_col(show.legend = FALSE) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  labs(
    title = "House Price Seasonality - Capital Region",
    subtitle = "Average deviation from annual mean price",
    x = NULL,
    y = "Seasonal Effect (%)"
  ) +
  theme_money_printer_go_brrr()

print(p_seasonality)
add_report_item("plot", p_seasonality, "House Price Seasonality", "6. Danish Housing Market")

In [ ]:
# ============================================================================
# TIME SERIES DECOMPOSITION & FORECASTING
# ============================================================================

print("=== TIME SERIES FORECASTING ===")

# Prepare time series for forecasting
ts_data <- df_seasonality %>%
  arrange(date) %>%
  filter(!is.na(date), !is.na(value)) %>%
  filter(date >= "2010-01-01")

stopifnot("No data in ts_data after filtering — check df_seasonality" = nrow(ts_data) > 0)

# Convert to time series object
start_year  <- year(min(ts_data$date))
start_month <- month(min(ts_data$date))

ts_prices <- ts(ts_data$value, start = c(start_year, start_month), frequency = 12)

# Decompose time series
decomp <- stl(ts_prices, s.window = "periodic")

# Plot decomposition
plot(decomp, main = "Capital Region House Prices - Time Series Decomposition")

# Forecast using ETS model
fit_ets      <- ets(ts_prices)
forecast_ets <- forecast(fit_ets, h = 24)   # 24 months ahead

cat("\nETS Model Summary:\n")
cat("  Model:", fit_ets$method, "\n")
cat("  AIC:", round(fit_ets$aic, 2), "\n")

# Derive forecast start from the time-series object itself
# (avoids the max(date) + months(1) NA problem when dates have any issues)
forecast_start <- as.Date(zoo::as.yearmon(time(forecast_ets$mean)))[1]

forecast_df <- data.frame(
  date     = as.Date(zoo::as.yearmon(time(forecast_ets$mean))),
  forecast = as.numeric(forecast_ets$mean),
  lower_80 = as.numeric(forecast_ets$lower[, 1]),
  upper_80 = as.numeric(forecast_ets$upper[, 1]),
  lower_95 = as.numeric(forecast_ets$lower[, 2]),
  upper_95 = as.numeric(forecast_ets$upper[, 2])
)

# Combine historical and forecast data for plotting
p_forecast <- ggplot() +
  # Historical data
  geom_line(data = ts_data, aes(x = date, y = value),
            color = "#1f77b4", linewidth = 0.8) +
  # 95% CI
  geom_ribbon(data = forecast_df, aes(x = date, ymin = lower_95, ymax = upper_95),
              fill = "#1f77b4", alpha = 0.15) +
  # 80% CI
  geom_ribbon(data = forecast_df, aes(x = date, ymin = lower_80, ymax = upper_80),
              fill = "#1f77b4", alpha = 0.25) +
  # Forecast line
  geom_line(data = forecast_df, aes(x = date, y = forecast),
            color = "#d62728", linewidth = 1, linetype = "dashed") +
  # Vertical line at forecast start
  geom_vline(xintercept = forecast_start, linetype = "dotted", color = "gray40") +

  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
  labs(
    title    = "Capital Region House Prices - 24-Month Forecast",
    subtitle = paste0("ETS Model: ", fit_ets$method,
                      " | Shaded areas: 80% and 95% confidence intervals"),
    x = NULL,
    y = "Price (DKK/m²)"
  ) +
  theme_money_printer_go_brrr()

print(p_forecast)
add_report_item("plot", p_forecast, "House Price Forecast - 24 Months", "6. Danish Housing Market")

# Summary statistics for forecast
cat("\n📊 24-Month Price Forecast Summary:\n")
cat("   Current price:",         scales::comma(last(ts_data$value)),              "DKK/m²\n")
cat("   12-month forecast:",     scales::comma(round(forecast_df$forecast[12])),  "DKK/m²\n")
cat("   24-month forecast:",     scales::comma(round(forecast_df$forecast[24])),  "DKK/m²\n")
cat("   24-month range (95% CI):",
    scales::comma(round(forecast_df$lower_95[24])), "-",
    scales::comma(round(forecast_df$upper_95[24])), "DKK/m²\n")

## 9.7 Long-Term Exponential Price Growth Analysis (1992-2026)

In [ ]:
# ============================================================================
# EXPONENTIAL PRICE GROWTH ANALYSIS (1992 - Present)
# ============================================================================
# Analyze long-term exponential growth in Danish property prices
# Data: BM011 quarterly postcode data from 1992
# ============================================================================

print("=== LONG-TERM EXPONENTIAL GROWTH ANALYSIS ===")

# Get long-term data (BM011 starts from 1992)
df_longterm <- df_bolig %>%
  filter(table_code == "BM011") %>%
  filter(grepl("F_rste_udbudspris|Første", series)) %>%
  filter(property_type == "Houses") %>%
  filter(!is.na(postcode))

# Calculate national average (across all postcodes)
df_national_avg <- df_longterm %>%
  group_by(date) %>%
  summarise(
    avg_price = mean(value, na.rm = TRUE),
    median_price = median(value, na.rm = TRUE),
    n_postcodes = n(),
    .groups = "drop"
  ) %>%
  filter(n_postcodes >= 5)  # Ensure representative sample

# Add time variable for regression
df_national_avg <- df_national_avg %>%
  arrange(date) %>%
  mutate(
    years_since_start = as.numeric(difftime(date, min(date), units = "days")) / 365.25,
    log_price = log(avg_price)
  )

cat("\n📈 Long-term Price Data:\n")
cat("   Period:", as.character(min(df_national_avg$date)), "to", as.character(max(df_national_avg$date)), "\n")
cat("   Years of data:", round(max(df_national_avg$years_since_start), 1), "\n")
cat("   Starting average:", scales::comma(round(first(df_national_avg$avg_price))), "DKK/m²\n")
cat("   Current average:", scales::comma(round(last(df_national_avg$avg_price))), "DKK/m²\n")

# Fit exponential model: Price = a * e^(b*t)
# Equivalent to: log(Price) = log(a) + b*t
exp_model <- lm(log_price ~ years_since_start, data = df_national_avg)

cat("\n📊 Exponential Growth Model:\n")
cat("   Price = ", round(exp(coef(exp_model)[1])), " × e^(",
    round(coef(exp_model)[2], 4), " × t)\n", sep = "")
cat("   Annual growth rate:", scales::percent(exp(coef(exp_model)[2]) - 1, accuracy = 0.1), "\n")
cat("   R-squared:", round(summary(exp_model)$r.squared, 4), "\n")

# Calculate doubling time
doubling_time <- log(2) / coef(exp_model)[2]
cat("   Doubling time:", round(doubling_time, 1), "years\n")

# Create prediction for historical and future
years_to_predict <- seq(0, max(df_national_avg$years_since_start) + 10, by = 0.25)
exp_predictions <- data.frame(
  years_since_start = years_to_predict,
  date = min(df_national_avg$date) + years_to_predict * 365.25,
  predicted_price = exp(predict(exp_model, newdata = data.frame(years_since_start = years_to_predict)))
)

# Split into historical fit and extrapolation
current_years <- max(df_national_avg$years_since_start)
exp_predictions <- exp_predictions %>%
  mutate(
    type = ifelse(years_since_start <= current_years, "Fitted", "Extrapolated")
  )

In [ ]:
# ============================================================================
# EXPONENTIAL GROWTH VISUALIZATION
# ============================================================================

# Plot exponential growth with extrapolation
p_exp_growth <- ggplot() +
  # Actual data
  geom_point(data = df_national_avg, aes(x = date, y = avg_price),
             color = "#1f77b4", alpha = 0.5, size = 1.5) +
  # Fitted exponential
  geom_line(data = exp_predictions %>% filter(type == "Fitted"),
            aes(x = date, y = predicted_price),
            color = "#2ca02c", size = 1.2) +
  # Extrapolated exponential
  geom_line(data = exp_predictions %>% filter(type == "Extrapolated"),
            aes(x = date, y = predicted_price),
            color = "#d62728", size = 1.2, linetype = "dashed") +
  # Vertical line at present
  geom_vline(xintercept = max(df_national_avg$date), linetype = "dotted", color = "gray40") +
  # Annotations
  annotate("text", x = as.Date("2000-01-01"), y = max(df_national_avg$avg_price) * 0.9,
           label = paste0("Annual CAGR: ", scales::percent(exp(coef(exp_model)[2]) - 1, accuracy = 0.1),
                         "\nDoubling time: ", round(doubling_time, 1), " years"),
           hjust = 0, size = 3.5, color = "gray30") +
  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²"),
                     trans = "log10") +
  labs(
    title = "Danish House Prices - Long-Term Exponential Growth (1992-2036)",
    subtitle = "Log scale | Green = fitted model | Red dashed = 10-year extrapolation",
    x = NULL,
    y = "Average Price (DKK/m², log scale)"
  ) +
  theme_money_printer_go_brrr()

print(p_exp_growth)
add_report_item("plot", p_exp_growth, "Exponential Price Growth (1992-2036)", "6. Danish Housing Market")

# Also show on linear scale for perspective
p_exp_linear <- ggplot() +
  geom_point(data = df_national_avg, aes(x = date, y = avg_price),
             color = "#1f77b4", alpha = 0.5, size = 1.5) +
  geom_line(data = exp_predictions %>% filter(type == "Fitted"),
            aes(x = date, y = predicted_price),
            color = "#2ca02c", size = 1.2) +
  geom_line(data = exp_predictions %>% filter(type == "Extrapolated"),
            aes(x = date, y = predicted_price),
            color = "#d62728", size = 1.2, linetype = "dashed") +
  geom_vline(xintercept = max(df_national_avg$date), linetype = "dotted", color = "gray40") +
  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
  labs(
    title = "Danish House Prices - Long-Term Exponential Growth (Linear Scale)",
    subtitle = paste0("If historical ", scales::percent(exp(coef(exp_model)[2]) - 1, accuracy = 0.1),
                     " annual growth continues..."),
    x = NULL,
    y = "Average Price (DKK/m²)"
  ) +
  theme_money_printer_go_brrr()

print(p_exp_linear)
add_report_item("plot", p_exp_linear, "Exponential Growth - Linear Scale", "6. Danish Housing Market")

In [ ]:
# ============================================================================
# POSTCODE-LEVEL GROWTH RATES COMPARISON
# ============================================================================

print("=== POSTCODE GROWTH RATE COMPARISON ===")

# Calculate CAGR for each postcode
postcode_cagr <- df_longterm %>%
  group_by(postcode) %>%
  filter(n() >= 20) %>%  # Require at least 20 observations
  summarise(
    first_date = min(date),
    last_date = max(date),
    first_price = first(value),
    last_price = last(value),
    years = as.numeric(difftime(max(date), min(date), units = "days")) / 365.25,
    cagr = (last_price / first_price)^(1/years) - 1,
    total_return = last_price / first_price - 1,
    .groups = "drop"
  ) %>%
  filter(years >= 10) %>%  # Require at least 10 years of data
  arrange(desc(cagr))

cat("\nTop 10 Postcodes by CAGR:\n")
print(head(postcode_cagr, 10))

cat("\nBottom 10 Postcodes by CAGR:\n")
print(tail(postcode_cagr, 10))

# Highlight 2800 Lyngby
lyngby_cagr <- postcode_cagr %>% filter(postcode == "2800")
if (nrow(lyngby_cagr) > 0) {
  cat("\n📍 2800 Kgs. Lyngby:\n")
  cat("   CAGR:", scales::percent(lyngby_cagr$cagr, accuracy = 0.1), "\n")
  cat("   Total return:", scales::percent(lyngby_cagr$total_return, accuracy = 0.1), "\n")
  cat("   Rank:", which(postcode_cagr$postcode == "2800"), "of", nrow(postcode_cagr), "\n")
}

# Plot CAGR distribution
p_cagr_dist <- postcode_cagr %>%
  ggplot(aes(x = cagr)) +
  geom_histogram(bins = 30, fill = "#1f77b4", alpha = 0.7, color = "white") +
  geom_vline(xintercept = mean(postcode_cagr$cagr), color = "#2ca02c", 
             linetype = "dashed", size = 1) +
  geom_vline(data = lyngby_cagr, aes(xintercept = cagr), 
             color = "#d62728", size = 1.2) +
  annotate("text", x = lyngby_cagr$cagr + 0.005, y = Inf, 
           label = "2800 Lyngby", color = "#d62728", vjust = 2, size = 3) +
  scale_x_continuous(labels = scales::percent_format(accuracy = 0.1)) +
  labs(
    title = "Distribution of House Price CAGR by Postcode",
    subtitle = paste0("Green = national average (", scales::percent(mean(postcode_cagr$cagr), accuracy = 0.1), 
                     ") | Red = 2800 Lyngby"),
    x = "Compound Annual Growth Rate (CAGR)",
    y = "Number of Postcodes"
  ) +
  theme_money_printer_go_brrr()

print(p_cagr_dist)
add_report_item("plot", p_cagr_dist, "CAGR Distribution by Postcode", "6. Danish Housing Market")

## 9.8 2800 Lyngby vs Comparable Postcodes

In [ ]:
# ============================================================================
# 2800 LYNGBY VS COMPARABLE AFFLUENT POSTCODES
# ============================================================================

print("=== LYNGBY VS COMPARABLE POSTCODES ===")

# North Zealand affluent postcodes for comparison
comparable_postcodes <- c(
  "2800",  # Kgs. Lyngby
  "2820",  # Gentofte
  "2830",  # Virum
  "2840",  # Holte
  "2850",  # Nærum
  "2900",  # Hellerup
  "2920",  # Charlottenlund
  "2960"   # Rungsted Kyst
)

# Get data for comparable postcodes
df_comparable <- df_longterm %>%
  filter(postcode %in% comparable_postcodes)

# Create labels
postcode_labels <- c(
  "2800" = "Kgs. Lyngby",
  "2820" = "Gentofte",
  "2830" = "Virum",
  "2840" = "Holte",
  "2850" = "Nærum",
  "2900" = "Hellerup",
  "2920" = "Charlottenlund",
  "2960" = "Rungsted Kyst"
)

df_comparable <- df_comparable %>%
  mutate(location = postcode_labels[postcode])

# Plot comparison
p_lyngby_comparison <- df_comparable %>%
  ggplot(aes(x = date, y = value, color = location)) +
  geom_line(aes(size = ifelse(postcode == "2800", "highlight", "normal")), alpha = 0.8) +
  scale_size_manual(values = c("highlight" = 1.5, "normal" = 0.7), guide = "none") +
  scale_y_continuous(labels = scales::comma_format(suffix = " DKK/m²")) +
  scale_color_viridis_d(option = "turbo") +
  labs(
    title = "2800 Kgs. Lyngby vs North Zealand Affluent Areas",
    subtitle = "Initial listing prices (Houses) - Lyngby highlighted",
    x = NULL,
    y = "Price (DKK/m²)",
    color = "Location"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "right")

print(p_lyngby_comparison)
add_report_item("plot", p_lyngby_comparison, "Lyngby vs North Zealand Peers", "6. Danish Housing Market")

# Indexed comparison (100 = starting value)
df_indexed <- df_comparable %>%
  group_by(postcode, location) %>%
  arrange(date) %>%
  mutate(
    index_value = 100 * value / first(value)
  ) %>%
  ungroup()

p_indexed <- df_indexed %>%
  ggplot(aes(x = date, y = index_value, color = location)) +
  geom_line(aes(size = ifelse(postcode == "2800", "highlight", "normal")), alpha = 0.8) +
  scale_size_manual(values = c("highlight" = 1.5, "normal" = 0.7), guide = "none") +
  geom_hline(yintercept = 100, linetype = "dotted", color = "gray50") +
  scale_color_viridis_d(option = "turbo") +
  labs(
    title = "Price Growth Index: Lyngby vs North Zealand Peers",
    subtitle = "Indexed to 100 at start of series",
    x = NULL,
    y = "Price Index (100 = start)",
    color = "Location"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "right")

print(p_indexed)
add_report_item("plot", p_indexed, "Indexed Price Comparison", "6. Danish Housing Market")

## 9.9 Market Cycle Analysis & Summary Dashboard

In [ ]:
# ============================================================================
# MARKET CYCLE ANALYSIS & SUMMARY
# ============================================================================

print("=== MARKET CYCLE ANALYSIS ===")

# Identify market cycles using YoY changes
df_yoy <- df_national_avg %>%
  arrange(date) %>%
  mutate(
    yoy_change = (avg_price / lag(avg_price, 4) - 1) * 100,  # Quarterly data, so 4 quarters = 1 year
    market_phase = case_when(
      yoy_change > 10 ~ "Boom (>10%)",
      yoy_change > 5 ~ "Growth (5-10%)",
      yoy_change > 0 ~ "Stable (0-5%)",
      yoy_change > -5 ~ "Correction (-5 to 0%)",
      TRUE ~ "Crash (<-5%)"
    )
  ) %>%
  filter(!is.na(yoy_change))

# Market cycle plot
p_market_cycle <- df_yoy %>%
  ggplot(aes(x = date, y = yoy_change)) +
  geom_area(aes(fill = yoy_change > 0), alpha = 0.3, show.legend = FALSE) +
  geom_line(color = "#1f77b4", size = 0.8) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
  geom_hline(yintercept = c(-5, 5, 10), linetype = "dotted", color = "gray60", size = 0.3) +
  scale_fill_manual(values = c("TRUE" = "#2ca02c", "FALSE" = "#d62728")) +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  labs(
    title = "Danish House Price Cycles - Year-over-Year Change",
    subtitle = "National average | Dotted lines at -5%, +5%, +10%",
    x = NULL,
    y = "YoY Price Change (%)"
  ) +
  theme_money_printer_go_brrr()

print(p_market_cycle)
add_report_item("plot", p_market_cycle, "Market Cycle Analysis", "6. Danish Housing Market")

# Market phase summary
phase_summary <- df_yoy %>%
  group_by(market_phase) %>%
  summarise(
    n_quarters = n(),
    pct_time = n() / nrow(df_yoy) * 100,
    avg_yoy = mean(yoy_change),
    .groups = "drop"
  ) %>%
  arrange(desc(avg_yoy))

cat("\n📊 Market Phase Distribution (since 1992):\n")
print(as.data.frame(phase_summary))

# Current market phase
current_phase <- last(df_yoy$market_phase)
current_yoy <- last(df_yoy$yoy_change)
cat("\n🔔 Current Market Phase:", current_phase, "\n")
cat("   Current YoY Change:", round(current_yoy, 1), "%\n")

In [ ]:
# ============================================================================
# SUMMARY DASHBOARD TABLE
# ============================================================================

print("=== DANISH HOUSING MARKET SUMMARY ===")

# Create summary table
summary_table <- data.frame(
  Metric = c(
    "Data Period",
    "National Avg Price (Current)",
    "National Avg Price (Start)",
    "Total Appreciation",
    "CAGR (since 1992)",
    "Doubling Time",
    "Current Market Phase",
    "Current YoY Change",
    "---",
    "2800 Lyngby - Current Price",
    "2800 Lyngby - CAGR",
    "2800 Lyngby - Rank (of postcodes)",
    "---",
    "Capital Region - Latest Listing/Sold Gap",
    "Capital Region - Days on Market (Houses)"
  ),
  Value = c(
    paste(as.character(min(df_national_avg$date)), "to", as.character(max(df_national_avg$date))),
    paste0(scales::comma(round(last(df_national_avg$avg_price))), " DKK/m²"),
    paste0(scales::comma(round(first(df_national_avg$avg_price))), " DKK/m²"),
    scales::percent(last(df_national_avg$avg_price) / first(df_national_avg$avg_price) - 1, accuracy = 0.1),
    scales::percent(exp(coef(exp_model)[2]) - 1, accuracy = 0.1),
    paste0(round(doubling_time, 1), " years"),
    current_phase,
    paste0(round(current_yoy, 1), "%"),
    "---",
    ifelse(nrow(lyngby_cagr) > 0, 
           paste0(scales::comma(round(lyngby_cagr$last_price)), " DKK/m²"), "N/A"),
    ifelse(nrow(lyngby_cagr) > 0, 
           scales::percent(lyngby_cagr$cagr, accuracy = 0.1), "N/A"),
    ifelse(nrow(lyngby_cagr) > 0, 
           paste0("#", which(postcode_cagr$postcode == "2800"), " of ", nrow(postcode_cagr)), "N/A"),
    "---",
    paste0(round(last(df_price_gap %>% filter(region == "Capital Region", property_type == "Houses") %>% pull(price_gap_pct)), 1), "%"),
    paste0(round(last(df_time_on_market %>% filter(region == "Capital Region", property_type == "Houses") %>% pull(value))), " days")
  )
)

cat("\n")
print(summary_table, row.names = FALSE)

# Add summary table to report
add_report_item("table", summary_table, "Danish Housing Market Summary", "6. Danish Housing Market")

print("\n" %+% paste(rep("=", 70), collapse = ""))
print("    DANISH HOUSING MARKET ANALYSIS COMPLETE    ")
print(paste(rep("=", 70), collapse = "") %+% "\n")

# Precious Metals

In [ ]:
# ============================================================================
# 10. PRECIOUS METALS ANALYSIS
# ============================================================================
# Metals: SI=F (Silver), GC=F (Gold), PA=F (Palladium)
# ============================================================================

print("\n=== 10. PRECIOUS METALS ANALYSIS ===\n")

# First, check the structure of the data
cat("🔍 Checking database schema...\n")
sample_data <- dbGetQuery(con, "
  SELECT * FROM financial_data 
  WHERE series IN ('SI=F', 'GC=F', 'PA=F')
  LIMIT 5
")

cat("Sample data structure:\n")
str(sample_data)
cat("\nColumn names:\n")
print(names(sample_data))
cat("\nFirst few rows:\n")
print(head(sample_data, 3))

# Load precious metals data from database
df_metals <- dbGetQuery(con, "
  SELECT 
    series,
    date,
    value
  FROM financial_data 
  WHERE series IN ('SI=F', 'GC=F', 'PA=F')
  ORDER BY series, date
")

# Rename columns for consistency
df_metals <- df_metals %>%
  rename(
    symbol = series,
    close = value
  )

# Convert date column and create metal names
df_metals <- df_metals %>%
  mutate(
    date = as.Date(date),
    close = as.numeric(close),  # Ensure close is numeric
    metal = case_when(
      symbol == "GC=F" ~ "Gold",
      symbol == "SI=F" ~ "Silver",
      symbol == "PA=F" ~ "Palladium",
      TRUE ~ symbol
    )
  ) %>%
  filter(!is.na(close), close > 0)

# Summary statistics
cat("\n📊 Precious Metals Data Summary:\n")
df_metals %>%
  group_by(metal, symbol) %>%
  summarise(
    records = n(),
    date_range = paste(min(date), "to", max(date)),
    min_price = min(close, na.rm = TRUE),
    max_price = max(close, na.rm = TRUE),
    current_price = last(close),
    .groups = "drop"
  ) %>%
  print()

# Calculate returns and technical indicators
df_metals <- df_metals %>%
  group_by(symbol) %>%
  arrange(date) %>%
  mutate(
    # Returns
    daily_return = (close - lag(close)) / lag(close),
    log_return = log(close / lag(close)),
    
    # Moving Averages
    MA_20 = zoo::rollmean(close, k = 20, fill = NA, align = "right"),
    MA_50 = zoo::rollmean(close, k = 50, fill = NA, align = "right"),
    MA_200 = zoo::rollmean(close, k = 200, fill = NA, align = "right"),
    
    # Exponential Moving Averages for MACD
    EMA_12 = TTR::EMA(close, n = 12),
    EMA_26 = TTR::EMA(close, n = 26),
    MACD_line = EMA_12 - EMA_26,
    MACD_signal = TTR::EMA(MACD_line, n = 9),
    MACD_histogram = MACD_line - MACD_signal,
    
    # Volatility
    volatility_20d = zoo::rollapply(daily_return, width = 20, FUN = sd, fill = NA, align = "right") * sqrt(252),
    
    # Days since start for regression
    days_since_start = as.numeric(date - min(date)),
    years_since_start = days_since_start / 365.25
  ) %>%
  ungroup()

# Calculate RSI (Relative Strength Index)
calculate_rsi <- function(prices, n = 14) {
  deltas <- diff(prices)
  gains <- ifelse(deltas > 0, deltas, 0)
  losses <- ifelse(deltas < 0, -deltas, 0)
  
  avg_gain <- zoo::rollmean(gains, k = n, fill = NA, align = "right")
  avg_loss <- zoo::rollmean(losses, k = n, fill = NA, align = "right")
  
  rs <- avg_gain / avg_loss
  rsi <- 100 - (100 / (1 + rs))
  
  c(NA, rsi)  # Prepend NA to match original length
}

df_metals <- df_metals %>%
  group_by(symbol) %>%
  mutate(RSI_14 = calculate_rsi(close, n = 14)) %>%
  ungroup()

cat("\n✅ Technical indicators calculated: MA(20,50,200), EMA(12,26), MACD, RSI(14)\n")

### 10.1 Precious Metals Price History

In [ ]:
# ============================================================================
# 10.1 PRECIOUS METALS PRICE HISTORY
# ============================================================================

# Individual price plots
p_metals_individual <- ggplot(df_metals, aes(x = date, y = close, color = metal)) +
  geom_line(linewidth = 0.5) +
  facet_wrap(~metal, scales = "free_y", ncol = 1) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_color_manual(values = c("Gold" = "#FFD700", "Silver" = "#C0C0C0", "Palladium" = "#6E7F80")) +
  labs(
    title = "Precious Metals Price History",
    subtitle = "Gold (GC=F), Silver (SI=F), Palladium (PA=F) - Daily Prices",
    x = "Date",
    y = "Price (USD)",
    color = "Metal"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_metals_individual)

add_report_item("plot", p_metals_individual, "Precious Metals Price History", "7. Precious Metals")

# Combined normalized view (indexed to 100)
df_metals_indexed <- df_metals %>%
  group_by(symbol) %>%
  mutate(
    indexed_price = (close / first(close)) * 100
  ) %>%
  ungroup()

p_metals_indexed <- ggplot(df_metals_indexed, aes(x = date, y = indexed_price, color = metal)) +
  geom_line(linewidth = 0.8) +
  geom_hline(yintercept = 100, linetype = "dashed", alpha = 0.5) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  scale_y_continuous(labels = scales::comma) +
  scale_color_manual(values = c("Gold" = "#FFD700", "Silver" = "#C0C0C0", "Palladium" = "#6E7F80")) +
  labs(
    title = "Precious Metals: Indexed Performance Comparison",
    subtitle = "All metals indexed to 100 at start of data series",
    x = "Date",
    y = "Indexed Price (Start = 100)",
    color = "Metal"
  ) +
  theme_money_printer_go_brrr()

print(p_metals_indexed)

add_report_item("plot", p_metals_indexed, "Precious Metals Indexed Performance", "7. Precious Metals")

cat("\n✅ Price history plots created\n")

### 10.2 Technical Analysis: Moving Averages & Trend

In [ ]:
# ============================================================================
# 10.2 TECHNICAL ANALYSIS: MOVING AVERAGES
# ============================================================================

# Get recent data for clearer visualization (last 3 years)
recent_metals <- df_metals %>%
  filter(date >= max(date) - years(3))

# Gold with Moving Averages
p_gold_ma <- recent_metals %>%
  filter(metal == "Gold") %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = close), color = "#FFD700", linewidth = 0.7, alpha = 0.8) +
  geom_line(aes(y = MA_20), color = "#2196F3", linewidth = 0.5, linetype = "solid") +
  geom_line(aes(y = MA_50), color = "#4CAF50", linewidth = 0.5, linetype = "solid") +
  geom_line(aes(y = MA_200), color = "#F44336", linewidth = 0.7, linetype = "solid") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Gold (GC=F) with Moving Averages",
    subtitle = "Price with MA(20), MA(50), MA(200) - Last 3 Years",
    x = "Date",
    y = "Price (USD)",
    caption = "Blue: MA(20) | Green: MA(50) | Red: MA(200)"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_ma)

add_report_item("plot", p_gold_ma, "Gold Price with Moving Averages", "7. Precious Metals")

# Silver with Moving Averages
p_silver_ma <- recent_metals %>%
  filter(metal == "Silver") %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = close), color = "#808080", linewidth = 0.7, alpha = 0.8) +
  geom_line(aes(y = MA_20), color = "#2196F3", linewidth = 0.5) +
  geom_line(aes(y = MA_50), color = "#4CAF50", linewidth = 0.5) +
  geom_line(aes(y = MA_200), color = "#F44336", linewidth = 0.7) +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Silver (SI=F) with Moving Averages",
    subtitle = "Price with MA(20), MA(50), MA(200) - Last 3 Years",
    x = "Date",
    y = "Price (USD)",
    caption = "Blue: MA(20) | Green: MA(50) | Red: MA(200)"
  ) +
  theme_money_printer_go_brrr()

print(p_silver_ma)

add_report_item("plot", p_silver_ma, "Silver Price with Moving Averages", "7. Precious Metals")

# Palladium with Moving Averages
p_palladium_ma <- recent_metals %>%
  filter(metal == "Palladium") %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = close), color = "#6E7F80", linewidth = 0.7, alpha = 0.8) +
  geom_line(aes(y = MA_20), color = "#2196F3", linewidth = 0.5) +
  geom_line(aes(y = MA_50), color = "#4CAF50", linewidth = 0.5) +
  geom_line(aes(y = MA_200), color = "#F44336", linewidth = 0.7) +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Palladium (PA=F) with Moving Averages",
    subtitle = "Price with MA(20), MA(50), MA(200) - Last 3 Years",
    x = "Date",
    y = "Price (USD)",
    caption = "Blue: MA(20) | Green: MA(50) | Red: MA(200)"
  ) +
  theme_money_printer_go_brrr()

print(p_palladium_ma)

add_report_item("plot", p_palladium_ma, "Palladium Price with Moving Averages", "7. Precious Metals")

cat("\n✅ Moving average charts created\n")

### 10.3 RSI (Relative Strength Index) Analysis

In [ ]:
# ============================================================================
# 10.3 RSI (RELATIVE STRENGTH INDEX) ANALYSIS
# ============================================================================

# RSI for all metals
p_rsi <- recent_metals %>%
  filter(!is.na(RSI_14)) %>%
  ggplot(aes(x = date, y = RSI_14, color = metal)) +
  geom_line(linewidth = 0.5) +
  geom_hline(yintercept = 70, linetype = "dashed", color = "red", alpha = 0.7) +
  geom_hline(yintercept = 30, linetype = "dashed", color = "green", alpha = 0.7) +
  geom_hline(yintercept = 50, linetype = "dotted", color = "gray", alpha = 0.5) +
  facet_wrap(~metal, ncol = 1) +
  scale_color_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  scale_y_continuous(limits = c(0, 100), breaks = c(0, 30, 50, 70, 100)) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "RSI(14) for Precious Metals",
    subtitle = "Relative Strength Index - Overbought (>70) and Oversold (<30) Levels",
    x = "Date",
    y = "RSI (14-day)",
    caption = "Red dashed: Overbought (70) | Green dashed: Oversold (30)"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_rsi)

add_report_item("plot", p_rsi, "Precious Metals RSI Analysis", "7. Precious Metals")

# RSI Distribution histogram
p_rsi_dist <- df_metals %>%
  filter(!is.na(RSI_14)) %>%
  ggplot(aes(x = RSI_14, fill = metal)) +
  geom_histogram(bins = 50, alpha = 0.7, position = "identity") +
  geom_vline(xintercept = c(30, 70), linetype = "dashed", color = "black") +
  facet_wrap(~metal, ncol = 3) +
  scale_fill_manual(values = c("Gold" = "#FFD700", "Silver" = "#C0C0C0", "Palladium" = "#6E7F80")) +
  labs(
    title = "RSI Distribution by Metal",
    subtitle = "Historical frequency of RSI values",
    x = "RSI (14-day)",
    y = "Frequency"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_rsi_dist)

add_report_item("plot", p_rsi_dist, "RSI Distribution by Metal", "7. Precious Metals")

# Current RSI status
cat("\n📊 Current RSI Status:\n")
df_metals %>%
  group_by(metal) %>%
  filter(date == max(date)) %>%
  select(metal, date, close, RSI_14) %>%
  mutate(
    status = case_when(
      RSI_14 >= 70 ~ "⚠️ OVERBOUGHT",
      RSI_14 <= 30 ~ "🟢 OVERSOLD",
      TRUE ~ "➖ NEUTRAL"
    )
  ) %>%
  print()

cat("\n✅ RSI analysis complete\n")

### 10.4 MACD (Moving Average Convergence Divergence)

In [ ]:
# ============================================================================
# 10.4 MACD (MOVING AVERAGE CONVERGENCE DIVERGENCE)
# ============================================================================

# MACD charts for each metal
create_macd_chart <- function(data, metal_name, color) {
  metal_data <- data %>% filter(metal == metal_name, !is.na(MACD_line))
  
  p1 <- ggplot(metal_data, aes(x = date)) +
    geom_line(aes(y = close), color = color, linewidth = 0.5) +
    scale_y_continuous(labels = scales::dollar_format()) +
    labs(title = paste(metal_name, "Price"), y = "Price (USD)") +
    theme_money_printer_go_brrr() +
    theme(axis.title.x = element_blank(), axis.text.x = element_blank())
  
  p2 <- ggplot(metal_data, aes(x = date)) +
    geom_line(aes(y = MACD_line), color = "#2196F3", linewidth = 0.5) +
    geom_line(aes(y = MACD_signal), color = "#FF9800", linewidth = 0.5) +
    geom_bar(aes(y = MACD_histogram), stat = "identity", 
             fill = ifelse(metal_data$MACD_histogram >= 0, "#4CAF50", "#F44336"), alpha = 0.5) +
    geom_hline(yintercept = 0, linetype = "solid", color = "gray") +
    labs(
      x = "Date",
      y = "MACD",
      caption = "Blue: MACD Line | Orange: Signal Line | Bars: Histogram"
    ) +
    theme_money_printer_go_brrr()
  
  gridExtra::grid.arrange(p1, p2, nrow = 2, heights = c(2, 1))
}

# Gold MACD
cat("\n📈 Gold MACD Chart:\n")
p_gold_macd <- recent_metals %>%
  filter(metal == "Gold", !is.na(MACD_line)) %>%
  ggplot(aes(x = date)) +
  geom_col(aes(y = MACD_histogram, fill = MACD_histogram >= 0), alpha = 0.6, width = 1) +
  geom_line(aes(y = MACD_line), color = "#2196F3", linewidth = 0.6) +
  geom_line(aes(y = MACD_signal), color = "#FF9800", linewidth = 0.6) +
  geom_hline(yintercept = 0, color = "gray40") +
  scale_fill_manual(values = c("TRUE" = "#4CAF50", "FALSE" = "#F44336"), guide = "none") +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Gold (GC=F) - MACD Analysis",
    subtitle = "MACD(12,26,9) - Last 3 Years",
    x = "Date",
    y = "MACD Value",
    caption = "Blue: MACD Line | Orange: Signal Line | Bars: Histogram"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_macd)

add_report_item("plot", p_gold_macd, "Gold MACD Analysis", "7. Precious Metals")

# Silver MACD
p_silver_macd <- recent_metals %>%
  filter(metal == "Silver", !is.na(MACD_line)) %>%
  ggplot(aes(x = date)) +
  geom_col(aes(y = MACD_histogram, fill = MACD_histogram >= 0), alpha = 0.6, width = 1) +
  geom_line(aes(y = MACD_line), color = "#2196F3", linewidth = 0.6) +
  geom_line(aes(y = MACD_signal), color = "#FF9800", linewidth = 0.6) +
  geom_hline(yintercept = 0, color = "gray40") +
  scale_fill_manual(values = c("TRUE" = "#4CAF50", "FALSE" = "#F44336"), guide = "none") +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Silver (SI=F) - MACD Analysis",
    subtitle = "MACD(12,26,9) - Last 3 Years",
    x = "Date",
    y = "MACD Value",
    caption = "Blue: MACD Line | Orange: Signal Line | Bars: Histogram"
  ) +
  theme_money_printer_go_brrr()

print(p_silver_macd)

add_report_item("plot", p_silver_macd, "Silver MACD Analysis", "7. Precious Metals")

# Palladium MACD
p_palladium_macd <- recent_metals %>%
  filter(metal == "Palladium", !is.na(MACD_line)) %>%
  ggplot(aes(x = date)) +
  geom_col(aes(y = MACD_histogram, fill = MACD_histogram >= 0), alpha = 0.6, width = 1) +
  geom_line(aes(y = MACD_line), color = "#2196F3", linewidth = 0.6) +
  geom_line(aes(y = MACD_signal), color = "#FF9800", linewidth = 0.6) +
  geom_hline(yintercept = 0, color = "gray40") +
  scale_fill_manual(values = c("TRUE" = "#4CAF50", "FALSE" = "#F44336"), guide = "none") +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Palladium (PA=F) - MACD Analysis",
    subtitle = "MACD(12,26,9) - Last 3 Years",
    x = "Date",
    y = "MACD Value",
    caption = "Blue: MACD Line | Orange: Signal Line | Bars: Histogram"
  ) +
  theme_money_printer_go_brrr()

print(p_palladium_macd)

add_report_item("plot", p_palladium_macd, "Palladium MACD Analysis", "7. Precious Metals")

cat("\n✅ MACD analysis complete\n")

### 10.5 Exponential and Linear Price Fits

In [ ]:
# ============================================================================
# 10.5 EXPONENTIAL AND LINEAR PRICE FITS
# ============================================================================

# Fit linear and exponential models for each metal
fit_models <- function(metal_name) {
  metal_data <- df_metals %>% 
    filter(metal == metal_name, !is.na(close), close > 0) %>%
    mutate(log_price = log(close))
  
  # Linear fit: price = a + b*t
  linear_model <- lm(close ~ years_since_start, data = metal_data)
  
  # Exponential fit: log(price) = a + b*t => price = exp(a) * exp(b*t)
  exp_model <- lm(log_price ~ years_since_start, data = metal_data)
  
  # Extract coefficients
  linear_coefs <- coef(linear_model)
  exp_coefs <- coef(exp_model)
  
  # Calculate annual growth rate from exponential model
  annual_growth_rate <- (exp(exp_coefs[2]) - 1) * 100
  
  # Add predictions to data
  metal_data <- metal_data %>%
    mutate(
      linear_fit = predict(linear_model),
      exp_fit = exp(predict(exp_model))
    )
  
  list(
    data = metal_data,
    linear_model = linear_model,
    exp_model = exp_model,
    linear_r2 = summary(linear_model)$r.squared,
    exp_r2 = summary(exp_model)$r.squared,
    annual_growth = annual_growth_rate,
    linear_slope = linear_coefs[2]
  )
}

# Fit models for each metal
gold_fit <- fit_models("Gold")
silver_fit <- fit_models("Silver")
palladium_fit <- fit_models("Palladium")

# Print model summaries
cat("\n📊 Model Fit Summary:\n")
cat("\n--- GOLD ---\n")
cat(sprintf("Linear R²: %.4f | Annual change: $%.2f/year\n", gold_fit$linear_r2, gold_fit$linear_slope))
cat(sprintf("Exponential R²: %.4f | CAGR: %.2f%%/year\n", gold_fit$exp_r2, gold_fit$annual_growth))

cat("\n--- SILVER ---\n")
cat(sprintf("Linear R²: %.4f | Annual change: $%.2f/year\n", silver_fit$linear_r2, silver_fit$linear_slope))
cat(sprintf("Exponential R²: %.4f | CAGR: %.2f%%/year\n", silver_fit$exp_r2, silver_fit$annual_growth))

cat("\n--- PALLADIUM ---\n")
cat(sprintf("Linear R²: %.4f | Annual change: $%.2f/year\n", palladium_fit$linear_r2, palladium_fit$linear_slope))
cat(sprintf("Exponential R²: %.4f | CAGR: %.2f%%/year\n", palladium_fit$exp_r2, palladium_fit$annual_growth))

# Combine all fitted data
all_fitted <- bind_rows(
  gold_fit$data %>% mutate(metal = "Gold"),
  silver_fit$data %>% mutate(metal = "Silver"),
  palladium_fit$data %>% mutate(metal = "Palladium")
)

# Gold with linear and exponential fit
p_gold_fits <- gold_fit$data %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = close), color = "#FFD700", alpha = 0.6, linewidth = 0.5) +
  geom_line(aes(y = linear_fit), color = "#2196F3", linewidth = 1, linetype = "dashed") +
  geom_line(aes(y = exp_fit), color = "#F44336", linewidth = 1, linetype = "solid") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Gold (GC=F) - Linear vs Exponential Trend Fit",
    subtitle = sprintf("Linear R²=%.3f | Exponential R²=%.3f | CAGR=%.1f%%", 
                       gold_fit$linear_r2, gold_fit$exp_r2, gold_fit$annual_growth),
    x = "Date",
    y = "Price (USD)",
    caption = "Gold price | Blue dashed: Linear fit | Red solid: Exponential fit"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_fits)

add_report_item("plot", p_gold_fits, "Gold Linear vs Exponential Fit", "7. Precious Metals")

# Silver with fits
p_silver_fits <- silver_fit$data %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = close), color = "#808080", alpha = 0.6, linewidth = 0.5) +
  geom_line(aes(y = linear_fit), color = "#2196F3", linewidth = 1, linetype = "dashed") +
  geom_line(aes(y = exp_fit), color = "#F44336", linewidth = 1, linetype = "solid") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Silver (SI=F) - Linear vs Exponential Trend Fit",
    subtitle = sprintf("Linear R²=%.3f | Exponential R²=%.3f | CAGR=%.1f%%", 
                       silver_fit$linear_r2, silver_fit$exp_r2, silver_fit$annual_growth),
    x = "Date",
    y = "Price (USD)",
    caption = "Silver price | Blue dashed: Linear fit | Red solid: Exponential fit"
  ) +
  theme_money_printer_go_brrr()

print(p_silver_fits)

add_report_item("plot", p_silver_fits, "Silver Linear vs Exponential Fit", "7. Precious Metals")

# Palladium with fits
p_palladium_fits <- palladium_fit$data %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = close), color = "#6E7F80", alpha = 0.6, linewidth = 0.5) +
  geom_line(aes(y = linear_fit), color = "#2196F3", linewidth = 1, linetype = "dashed") +
  geom_line(aes(y = exp_fit), color = "#F44336", linewidth = 1, linetype = "solid") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Palladium (PA=F) - Linear vs Exponential Trend Fit",
    subtitle = sprintf("Linear R²=%.3f | Exponential R²=%.3f | CAGR=%.1f%%", 
                       palladium_fit$linear_r2, palladium_fit$exp_r2, palladium_fit$annual_growth),
    x = "Date",
    y = "Price (USD)",
    caption = "Palladium price | Blue dashed: Linear fit | Red solid: Exponential fit"
  ) +
  theme_money_printer_go_brrr()

print(p_palladium_fits)

add_report_item("plot", p_palladium_fits, "Palladium Linear vs Exponential Fit", "7. Precious Metals")

cat("\n✅ Trend fit analysis complete\n")

### 10.6 Precious Metal Ratios

In [ ]:
# ============================================================================
# 10.6 PRECIOUS METAL RATIOS
# ============================================================================

# Pivot to wide format for ratio calculations
df_metals_wide <- df_metals %>%
  select(date, metal, close) %>%
  pivot_wider(names_from = metal, values_from = close) %>%
  filter(!is.na(Gold) & !is.na(Silver) & !is.na(Palladium)) %>%
  mutate(
    # Gold/Silver ratio (historically averages ~60-80)
    gold_silver_ratio = Gold / Silver,
    
    # Gold/Palladium ratio
    gold_palladium_ratio = Gold / Palladium,
    
    # Silver/Palladium ratio
    silver_palladium_ratio = Silver / Palladium
  )

# Summary statistics for ratios
cat("\n📊 Precious Metal Ratio Statistics:\n")
cat("\n--- Gold/Silver Ratio ---\n")
cat(sprintf("Current: %.1f | Mean: %.1f | Min: %.1f | Max: %.1f\n",
            last(df_metals_wide$gold_silver_ratio),
            mean(df_metals_wide$gold_silver_ratio, na.rm = TRUE),
            min(df_metals_wide$gold_silver_ratio, na.rm = TRUE),
            max(df_metals_wide$gold_silver_ratio, na.rm = TRUE)))

cat("\n--- Gold/Palladium Ratio ---\n")
cat(sprintf("Current: %.2f | Mean: %.2f | Min: %.2f | Max: %.2f\n",
            last(df_metals_wide$gold_palladium_ratio),
            mean(df_metals_wide$gold_palladium_ratio, na.rm = TRUE),
            min(df_metals_wide$gold_palladium_ratio, na.rm = TRUE),
            max(df_metals_wide$gold_palladium_ratio, na.rm = TRUE)))

# Gold/Silver Ratio plot
p_gold_silver_ratio <- ggplot(df_metals_wide, aes(x = date, y = gold_silver_ratio)) +
  geom_line(color = "#9C27B0", linewidth = 0.5) +
  geom_hline(yintercept = mean(df_metals_wide$gold_silver_ratio, na.rm = TRUE), 
             linetype = "dashed", color = "gray40") +
  geom_hline(yintercept = 80, linetype = "dotted", color = "red", alpha = 0.7) +
  geom_hline(yintercept = 60, linetype = "dotted", color = "green", alpha = 0.7) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Gold/Silver Ratio Over Time",
    subtitle = "Ounces of silver to buy one ounce of gold (historical avg ~60-80)",
    x = "Date",
    y = "Gold/Silver Ratio",
    caption = "Dashed: Mean ratio | Green: 60 | Red: 80"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_silver_ratio)

add_report_item("plot", p_gold_silver_ratio, "Gold/Silver Ratio History", "7. Precious Metals")

# Gold/Palladium Ratio plot
p_gold_palladium_ratio <- ggplot(df_metals_wide, aes(x = date, y = gold_palladium_ratio)) +
  geom_line(color = "#FF5722", linewidth = 0.5) +
  geom_hline(yintercept = mean(df_metals_wide$gold_palladium_ratio, na.rm = TRUE), 
             linetype = "dashed", color = "gray40") +
  geom_hline(yintercept = 1, linetype = "solid", color = "black", alpha = 0.5) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Gold/Palladium Ratio Over Time",
    subtitle = "When >1, gold is more expensive; when <1, palladium is more expensive",
    x = "Date",
    y = "Gold/Palladium Ratio",
    caption = "Solid black: Parity (1:1) | Dashed gray: Mean"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_palladium_ratio)

add_report_item("plot", p_gold_palladium_ratio, "Gold/Palladium Ratio History", "7. Precious Metals")

# Combined ratio panel
df_ratios_long <- df_metals_wide %>%
  select(date, gold_silver_ratio, gold_palladium_ratio) %>%
  pivot_longer(cols = -date, names_to = "ratio_type", values_to = "ratio") %>%
  mutate(
    ratio_label = case_when(
      ratio_type == "gold_silver_ratio" ~ "Gold/Silver",
      ratio_type == "gold_palladium_ratio" ~ "Gold/Palladium"
    )
  )

p_ratios_panel <- ggplot(df_ratios_long, aes(x = date, y = ratio, color = ratio_label)) +
  geom_line(linewidth = 0.5) +
  facet_wrap(~ratio_label, scales = "free_y", ncol = 1) +
  scale_color_manual(values = c("Gold/Silver" = "#9C27B0", "Gold/Palladium" = "#FF5722")) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Precious Metal Ratios Over Time",
    subtitle = "Key valuation metrics for relative metal pricing",
    x = "Date",
    y = "Ratio",
    color = "Ratio"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_ratios_panel)

add_report_item("plot", p_ratios_panel, "Combined Metal Ratios Panel", "7. Precious Metals")

cat("\n✅ Ratio analysis complete\n")

### 10.7 Volatility Analysis & Bollinger Bands

In [ ]:
# ============================================================================
# 10.7 VOLATILITY ANALYSIS & BOLLINGER BANDS
# ============================================================================

# Calculate Bollinger Bands
df_metals <- df_metals %>%
  group_by(symbol) %>%
  mutate(
    BB_middle = MA_20,
    BB_std = zoo::rollapply(close, width = 20, FUN = sd, fill = NA, align = "right"),
    BB_upper = BB_middle + 2 * BB_std,
    BB_lower = BB_middle - 2 * BB_std,
    BB_width = (BB_upper - BB_lower) / BB_middle * 100,  # Percentage width
    BB_position = (close - BB_lower) / (BB_upper - BB_lower)  # Where price is within bands
  ) %>%
  ungroup()

# Re-derive recent_metals now that BB columns exist
recent_metals <- df_metals %>%
  filter(date >= max(date) - years(3))

# Volatility comparison
p_volatility <- df_metals %>%
  filter(!is.na(volatility_20d)) %>%
  ggplot(aes(x = date, y = volatility_20d * 100, color = metal)) +
  geom_line(linewidth = 0.5, alpha = 0.7) +
  facet_wrap(~metal, ncol = 1) +
  scale_color_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  labs(
    title = "Annualized Volatility (20-day Rolling)",
    subtitle = "Historical volatility comparison across precious metals",
    x = "Date",
    y = "Annualized Volatility (%)",
    color = "Metal"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_volatility)

add_report_item("plot", p_volatility, "Precious Metals Volatility Comparison", "7. Precious Metals")

# Gold with Bollinger Bands
p_gold_bb <- recent_metals %>%
  filter(metal == "Gold", !is.na(BB_upper)) %>%
  ggplot(aes(x = date)) +
  geom_ribbon(aes(ymin = BB_lower, ymax = BB_upper), fill = "#FFD700", alpha = 0.2) +
  geom_line(aes(y = close), color = "#FFD700", linewidth = 0.6) +
  geom_line(aes(y = BB_middle), color = "#2196F3", linewidth = 0.5, linetype = "dashed") +
  geom_line(aes(y = BB_upper), color = "#F44336", linewidth = 0.4, alpha = 0.7) +
  geom_line(aes(y = BB_lower), color = "#4CAF50", linewidth = 0.4, alpha = 0.7) +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Gold (GC=F) with Bollinger Bands",
    subtitle = "20-day MA with ±2 standard deviation bands - Last 3 Years",
    x = "Date",
    y = "Price (USD)",
    caption = "Band: ±2σ | Blue dashed: MA(20) | Red: Upper | Green: Lower"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_bb)

add_report_item("plot", p_gold_bb, "Gold Bollinger Bands", "7. Precious Metals")

# Silver with Bollinger Bands
p_silver_bb <- recent_metals %>%
  filter(metal == "Silver", !is.na(BB_upper)) %>%
  ggplot(aes(x = date)) +
  geom_ribbon(aes(ymin = BB_lower, ymax = BB_upper), fill = "#808080", alpha = 0.2) +
  geom_line(aes(y = close), color = "#808080", linewidth = 0.6) +
  geom_line(aes(y = BB_middle), color = "#2196F3", linewidth = 0.5, linetype = "dashed") +
  geom_line(aes(y = BB_upper), color = "#F44336", linewidth = 0.4, alpha = 0.7) +
  geom_line(aes(y = BB_lower), color = "#4CAF50", linewidth = 0.4, alpha = 0.7) +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Silver (SI=F) with Bollinger Bands",
    subtitle = "20-day MA with ±2 standard deviation bands - Last 3 Years",
    x = "Date",
    y = "Price (USD)",
    caption = "Band: ±2σ | Blue dashed: MA(20) | Red: Upper | Green: Lower"
  ) +
  theme_money_printer_go_brrr()

print(p_silver_bb)

add_report_item("plot", p_silver_bb, "Silver Bollinger Bands", "7. Precious Metals")

# Volatility regime comparison
cat("\n📊 Current Volatility Levels (Annualized %):\n")
df_metals %>%
  filter(!is.na(volatility_20d)) %>%
  group_by(metal) %>%
  summarise(
    current_vol = last(volatility_20d) * 100,
    avg_vol = mean(volatility_20d, na.rm = TRUE) * 100,
    max_vol = max(volatility_20d, na.rm = TRUE) * 100,
    percentile = ecdf(volatility_20d)(last(volatility_20d)) * 100,
    .groups = "drop"
  ) %>%
  mutate(
    regime = case_when(
      percentile >= 80 ~ "HIGH",
      percentile >= 50 ~ "MODERATE",
      TRUE ~ "LOW"
    )
  ) %>%
  print()

cat("\n✅ Volatility and Bollinger Bands analysis complete\n")

### 10.8 Correlation Analysis & Rolling Correlations

In [ ]:
# ============================================================================
# 10.8 CORRELATION ANALYSIS & ROLLING CORRELATIONS
# ============================================================================

# Prepare returns data for correlation
df_returns_wide <- df_metals %>%
  select(date, metal, daily_return) %>%
  filter(!is.na(daily_return)) %>%
  pivot_wider(names_from = metal, values_from = daily_return) %>%
  filter(!is.na(Gold) & !is.na(Silver) & !is.na(Palladium))

# Overall correlation matrix
cor_matrix <- cor(df_returns_wide %>% select(Gold, Silver, Palladium), use = "complete.obs")

cat("\n📊 Precious Metals Return Correlation Matrix:\n")
print(round(cor_matrix, 3))

# Correlation heatmap
cor_df <- as.data.frame(as.table(cor_matrix)) %>%
  rename(Metal1 = Var1, Metal2 = Var2, Correlation = Freq)

p_cor_heatmap <- ggplot(cor_df, aes(x = Metal1, y = Metal2, fill = Correlation)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.2f", Correlation)), color = "white", size = 5) +
  scale_fill_gradient2(low = "#2196F3", mid = "white", high = "#F44336", midpoint = 0, limits = c(-1, 1)) +
  labs(
    title = "Precious Metals Correlation Matrix",
    subtitle = "Based on daily returns",
    x = "",
    y = "",
    fill = "Correlation"
  ) +
  theme_money_printer_go_brrr() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p_cor_heatmap)

add_report_item("plot", p_cor_heatmap, "Precious Metals Correlation Heatmap", "7. Precious Metals")

# Rolling 60-day correlations
df_rolling_cor <- df_returns_wide %>%
  arrange(date) %>%
  mutate(
    cor_gold_silver = zoo::rollapply(
      cbind(Gold, Silver), width = 60, 
      FUN = function(x) cor(x[,1], x[,2]), 
      fill = NA, align = "right", by.column = FALSE
    ),
    cor_gold_palladium = zoo::rollapply(
      cbind(Gold, Palladium), width = 60, 
      FUN = function(x) cor(x[,1], x[,2]), 
      fill = NA, align = "right", by.column = FALSE
    ),
    cor_silver_palladium = zoo::rollapply(
      cbind(Silver, Palladium), width = 60, 
      FUN = function(x) cor(x[,1], x[,2]), 
      fill = NA, align = "right", by.column = FALSE
    )
  )

# Rolling correlation plot
df_rolling_cor_long <- df_rolling_cor %>%
  select(date, cor_gold_silver, cor_gold_palladium, cor_silver_palladium) %>%
  pivot_longer(cols = -date, names_to = "pair", values_to = "correlation") %>%
  filter(!is.na(correlation)) %>%
  mutate(
    pair_label = case_when(
      pair == "cor_gold_silver" ~ "Gold-Silver",
      pair == "cor_gold_palladium" ~ "Gold-Palladium",
      pair == "cor_silver_palladium" ~ "Silver-Palladium"
    )
  )

p_rolling_cor <- ggplot(df_rolling_cor_long, aes(x = date, y = correlation, color = pair_label)) +
  geom_line(linewidth = 0.5, alpha = 0.8) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
  scale_y_continuous(limits = c(-1, 1)) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  scale_color_brewer(palette = "Set1") +
  labs(
    title = "Rolling 60-Day Correlations Between Precious Metals",
    subtitle = "Correlation of daily returns over rolling 60-day windows",
    x = "Date",
    y = "Correlation",
    color = "Pair"
  ) +
  theme_money_printer_go_brrr()

print(p_rolling_cor)

add_report_item("plot", p_rolling_cor, "Rolling Precious Metals Correlations", "7. Precious Metals")

# Current correlation status
cat("\n📊 Current 60-Day Rolling Correlations:\n")
df_rolling_cor %>%
  filter(date == max(date)) %>%
  select(date, cor_gold_silver, cor_gold_palladium, cor_silver_palladium) %>%
  print()

cat("\n✅ Correlation analysis complete\n")

### 10.9 Drawdown Analysis & Risk Metrics

In [ ]:
# ============================================================================
# 10.9 DRAWDOWN ANALYSIS & RISK METRICS
# ============================================================================

# Calculate drawdowns
df_metals <- df_metals %>%
  group_by(symbol) %>%
  arrange(date) %>%
  mutate(
    cummax_price = cummax(close),
    drawdown = (close - cummax_price) / cummax_price * 100
  ) %>%
  ungroup()

# Drawdown visualization
p_drawdown <- df_metals %>%
  ggplot(aes(x = date, y = drawdown, fill = metal)) +
  geom_area(alpha = 0.6) +
  facet_wrap(~metal, ncol = 1, scales = "free_y") +
  scale_fill_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  scale_x_date(date_breaks = "2 years", date_labels = "%Y") +
  scale_y_continuous(labels = function(x) paste0(x, "%")) +
  labs(
    title = "Drawdown Analysis: Precious Metals",
    subtitle = "Percentage decline from all-time high at each point",
    x = "Date",
    y = "Drawdown from Peak (%)"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_drawdown)

add_report_item("plot", p_drawdown, "Precious Metals Drawdown Analysis", "7. Precious Metals")

# Maximum drawdown table
max_drawdowns <- df_metals %>%
  group_by(metal) %>%
  summarise(
    max_drawdown = min(drawdown),
    max_dd_date = date[which.min(drawdown)],
    current_drawdown = last(drawdown),
    recovery_needed = -current_drawdown / (100 + current_drawdown) * 100,
    .groups = "drop"
  )

cat("\n📊 Maximum Drawdown Analysis:\n")
print(max_drawdowns)

# Risk-adjusted metrics (Sharpe-like, using 0% risk-free rate for simplicity)
risk_metrics <- df_metals %>%
  filter(!is.na(daily_return)) %>%
  group_by(metal) %>%
  summarise(
    total_return = (last(close) / first(close) - 1) * 100,
    years = as.numeric(max(date) - min(date)) / 365.25,
    annualized_return = (((last(close) / first(close))^(1/years)) - 1) * 100,
    annualized_volatility = sd(daily_return, na.rm = TRUE) * sqrt(252) * 100,
    sharpe_ratio = annualized_return / annualized_volatility,
    max_drawdown = min(drawdown),
    calmar_ratio = annualized_return / abs(max_drawdown),
    .groups = "drop"
  )

cat("\n📊 Risk-Adjusted Performance Metrics:\n")
print(risk_metrics)

# Risk/Return scatter
p_risk_return <- ggplot(risk_metrics, aes(x = annualized_volatility, y = annualized_return, color = metal)) +
  geom_point(size = 5) +
  geom_text(aes(label = metal), vjust = -1, hjust = 0.5, size = 4) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray") +
  scale_color_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  labs(
    title = "Risk-Return Profile: Precious Metals",
    subtitle = "Annualized return vs. volatility",
    x = "Annualized Volatility (%)",
    y = "Annualized Return (%)",
    caption = sprintf("Data period: %s to %s", min(df_metals$date), max(df_metals$date))
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_risk_return)

add_report_item("plot", p_risk_return, "Precious Metals Risk-Return Profile", "7. Precious Metals")

cat("\n✅ Drawdown and risk analysis complete\n")

### 10.10 Seasonality Analysis

In [ ]:
# ============================================================================
# 10.10 SEASONALITY ANALYSIS
# ============================================================================

# Add month and day-of-week
df_metals <- df_metals %>%
  mutate(
    month = month(date, label = TRUE),
    month_num = month(date),
    year = year(date),
    day_of_week = wday(date, label = TRUE)
  )

# Monthly average returns by metal
monthly_returns <- df_metals %>%
  filter(!is.na(daily_return)) %>%
  group_by(metal, month_num, month) %>%
  summarise(
    avg_return = mean(daily_return, na.rm = TRUE) * 100 * 21,  # Approximate monthly return
    median_return = median(daily_return, na.rm = TRUE) * 100 * 21,
    positive_pct = mean(daily_return > 0, na.rm = TRUE) * 100,
    .groups = "drop"
  )

# Monthly seasonality plot
p_seasonality <- ggplot(monthly_returns, aes(x = month, y = avg_return, fill = metal)) +
  geom_col(position = "dodge", alpha = 0.8) +
  geom_hline(yintercept = 0, color = "gray40") +
  facet_wrap(~metal, ncol = 1) +
  scale_fill_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  labs(
    title = "Monthly Seasonality: Precious Metals",
    subtitle = "Average monthly returns by calendar month",
    x = "Month",
    y = "Average Monthly Return (%)"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none", axis.text.x = element_text(angle = 45, hjust = 1))

print(p_seasonality)

add_report_item("plot", p_seasonality, "Precious Metals Monthly Seasonality", "7. Precious Metals")

# Seasonality heatmap
p_seasonality_heatmap <- monthly_returns %>%
  ggplot(aes(x = month, y = metal, fill = avg_return)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.1f%%", avg_return)), size = 3) +
  scale_fill_gradient2(low = "#F44336", mid = "white", high = "#4CAF50", midpoint = 0) +
  labs(
    title = "Seasonality Heatmap: Average Monthly Returns",
    subtitle = "Green = positive, Red = negative average returns",
    x = "Month",
    y = "Metal",
    fill = "Avg Return (%)"
  ) +
  theme_money_printer_go_brrr() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p_seasonality_heatmap)

add_report_item("plot", p_seasonality_heatmap, "Seasonality Heatmap", "7. Precious Metals")

# Day-of-week effect
dow_returns <- df_metals %>%
  filter(!is.na(daily_return)) %>%
  group_by(metal, day_of_week) %>%
  summarise(
    avg_return = mean(daily_return, na.rm = TRUE) * 100,
    positive_pct = mean(daily_return > 0, na.rm = TRUE) * 100,
    .groups = "drop"
  )

p_dow <- ggplot(dow_returns, aes(x = day_of_week, y = avg_return, fill = metal)) +
  geom_col(position = "dodge", alpha = 0.8) +
  geom_hline(yintercept = 0, color = "gray40") +
  scale_fill_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  labs(
    title = "Day-of-Week Effect: Precious Metals",
    subtitle = "Average daily returns by day of week",
    x = "Day of Week",
    y = "Average Daily Return (%)",
    fill = "Metal"
  ) +
  theme_money_printer_go_brrr()

print(p_dow)

add_report_item("plot", p_dow, "Day-of-Week Effect", "7. Precious Metals")

# Best/Worst months summary
cat("\n📊 Best & Worst Months (by average return):\n")
monthly_returns %>%
  group_by(metal) %>%
  slice_max(avg_return, n = 1) %>%
  mutate(type = "Best") %>%
  bind_rows(
    monthly_returns %>%
      group_by(metal) %>%
      slice_min(avg_return, n = 1) %>%
      mutate(type = "Worst")
  ) %>%
  select(metal, type, month, avg_return) %>%
  arrange(metal, desc(type)) %>%
  print()

cat("\n✅ Seasonality analysis complete\n")

### 10.11 Price Momentum & Mean Reversion

In [ ]:
# ============================================================================
# 10.11 PRICE MOMENTUM & MEAN REVERSION
# ============================================================================

# Calculate momentum indicators
df_metals <- df_metals %>%
  group_by(symbol) %>%
  arrange(date) %>%
  mutate(
    # Momentum (rate of change)
    mom_1m = (close / lag(close, 21) - 1) * 100,
    mom_3m = (close / lag(close, 63) - 1) * 100,
    mom_6m = (close / lag(close, 126) - 1) * 100,
    mom_12m = (close / lag(close, 252) - 1) * 100,
    
    # Distance from moving averages (mean reversion signal)
    dist_from_ma50 = (close / MA_50 - 1) * 100,
    dist_from_ma200 = (close / MA_200 - 1) * 100,
    
    # Z-score (for mean reversion)
    price_zscore = (close - MA_200) / (zoo::rollapply(close, width = 200, FUN = sd, fill = NA, align = "right"))
  ) %>%
  ungroup()

# Re-derive recent_metals now that momentum columns exist
recent_metals <- df_metals %>%
  filter(date >= max(date) - years(3))

# Momentum comparison
p_momentum <- recent_metals %>%
  select(date, metal, mom_1m, mom_3m, mom_6m) %>%
  filter(!is.na(mom_1m)) %>%
  pivot_longer(cols = starts_with("mom"), names_to = "period", values_to = "momentum") %>%
  mutate(
    period_label = case_when(
      period == "mom_1m" ~ "1 Month",
      period == "mom_3m" ~ "3 Month",
      period == "mom_6m" ~ "6 Month"
    )
  ) %>%
  ggplot(aes(x = date, y = momentum, color = period_label)) +
  geom_line(linewidth = 0.5, alpha = 0.8) +
  geom_hline(yintercept = 0, color = "gray40", linetype = "dashed") +
  facet_wrap(~metal, ncol = 1, scales = "free_y") +
  scale_color_brewer(palette = "Set2") +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Momentum Indicators: Precious Metals",
    subtitle = "Rate of change over different time periods - Last 3 Years",
    x = "Date",
    y = "Momentum (%)",
    color = "Period"
  ) +
  theme_money_printer_go_brrr()

print(p_momentum)

add_report_item("plot", p_momentum, "Precious Metals Momentum Indicators", "7. Precious Metals")

# Distance from MA200 (mean reversion)
p_mean_reversion <- recent_metals %>%
  filter(!is.na(dist_from_ma200)) %>%
  ggplot(aes(x = date, y = dist_from_ma200, fill = metal)) +
  geom_col(alpha = 0.6, width = 1) +
  geom_hline(yintercept = 0, color = "black") +
  geom_hline(yintercept = c(-10, 10), linetype = "dashed", color = "gray") +
  facet_wrap(~metal, ncol = 1) +
  scale_fill_manual(values = c("Gold" = "#FFD700", "Silver" = "#808080", "Palladium" = "#6E7F80")) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = "Distance from 200-Day Moving Average",
    subtitle = "Mean reversion signal: Extreme values may indicate overbought/oversold",
    x = "Date",
    y = "Distance from MA(200) (%)",
    caption = "Dashed lines at ±10%"
  ) +
  theme_money_printer_go_brrr() +
  theme(legend.position = "none")

print(p_mean_reversion)

add_report_item("plot", p_mean_reversion, "Mean Reversion Signals", "7. Precious Metals")

# Current momentum status
cat("\n📊 Current Momentum Status:\n")
df_metals %>%
  group_by(metal) %>%
  filter(date == max(date)) %>%
  select(metal, mom_1m, mom_3m, mom_6m, mom_12m, dist_from_ma200) %>%
  mutate(
    trend = case_when(
      mom_1m > 0 & mom_3m > 0 & mom_6m > 0 ~ "🟢 STRONG UPTREND",
      mom_1m < 0 & mom_3m < 0 & mom_6m < 0 ~ "🔴 STRONG DOWNTREND",
      TRUE ~ "⚪ MIXED"
    )
  ) %>%
  print()

cat("\n✅ Momentum analysis complete\n")

### 10.12 Price Forecasting with ARIMA

In [ ]:
# ============================================================================
# 10.12 PRICE FORECASTING WITH ARIMA/ETS
# ============================================================================

# Forecast each metal using ETS (Exponential Smoothing)
forecast_horizon <- 90  # 90 days ahead

# Gold forecast
gold_ts <- df_metals %>%
  filter(metal == "Gold") %>%
  arrange(date) %>%
  pull(close) %>%
  ts(frequency = 252)  # Daily data, ~252 trading days/year

gold_model <- forecast::ets(gold_ts)
gold_forecast <- forecast::forecast(gold_model, h = forecast_horizon)

# Silver forecast
silver_ts <- df_metals %>%
  filter(metal == "Silver") %>%
  arrange(date) %>%
  pull(close) %>%
  ts(frequency = 252)

silver_model <- forecast::ets(silver_ts)
silver_forecast <- forecast::forecast(silver_model, h = forecast_horizon)

# Palladium forecast
palladium_ts <- df_metals %>%
  filter(metal == "Palladium") %>%
  arrange(date) %>%
  pull(close) %>%
  ts(frequency = 252)

palladium_model <- forecast::ets(palladium_ts)
palladium_forecast <- forecast::forecast(palladium_model, h = forecast_horizon)

# Create forecast data frames
last_gold_date <- max(df_metals$date[df_metals$metal == "Gold"])
last_silver_date <- max(df_metals$date[df_metals$metal == "Silver"])
last_palladium_date <- max(df_metals$date[df_metals$metal == "Palladium"])

gold_fc_df <- data.frame(
  date = seq(last_gold_date + 1, by = "day", length.out = forecast_horizon),
  forecast = as.numeric(gold_forecast$mean),
  lower_80 = as.numeric(gold_forecast$lower[,1]),
  upper_80 = as.numeric(gold_forecast$upper[,1]),
  lower_95 = as.numeric(gold_forecast$lower[,2]),
  upper_95 = as.numeric(gold_forecast$upper[,2]),
  metal = "Gold"
)

silver_fc_df <- data.frame(
  date = seq(last_silver_date + 1, by = "day", length.out = forecast_horizon),
  forecast = as.numeric(silver_forecast$mean),
  lower_80 = as.numeric(silver_forecast$lower[,1]),
  upper_80 = as.numeric(silver_forecast$upper[,1]),
  lower_95 = as.numeric(silver_forecast$lower[,2]),
  upper_95 = as.numeric(silver_forecast$upper[,2]),
  metal = "Silver"
)

palladium_fc_df <- data.frame(
  date = seq(last_palladium_date + 1, by = "day", length.out = forecast_horizon),
  forecast = as.numeric(palladium_forecast$mean),
  lower_80 = as.numeric(palladium_forecast$lower[,1]),
  upper_80 = as.numeric(palladium_forecast$upper[,1]),
  lower_95 = as.numeric(palladium_forecast$lower[,2]),
  upper_95 = as.numeric(palladium_forecast$upper[,2]),
  metal = "Palladium"
)

# Gold forecast plot
gold_recent <- df_metals %>% 
  filter(metal == "Gold", date >= max(date) - years(2))

p_gold_forecast <- ggplot() +
  geom_line(data = gold_recent, aes(x = date, y = close), color = "#FFD700", linewidth = 0.6) +
  geom_ribbon(data = gold_fc_df, aes(x = date, ymin = lower_95, ymax = upper_95), 
              fill = "#FFD700", alpha = 0.2) +
  geom_ribbon(data = gold_fc_df, aes(x = date, ymin = lower_80, ymax = upper_80), 
              fill = "#FFD700", alpha = 0.3) +
  geom_line(data = gold_fc_df, aes(x = date, y = forecast), color = "#FF5722", linewidth = 0.8, linetype = "dashed") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = sprintf("Gold (GC=F) - %d-Day Price Forecast", forecast_horizon),
    subtitle = sprintf("ETS Model: %s", gold_model$method),
    x = "Date",
    y = "Price (USD)",
    caption = "Shaded: 80% and 95% confidence intervals | Dashed: Point forecast"
  ) +
  theme_money_printer_go_brrr()

print(p_gold_forecast)

add_report_item("plot", p_gold_forecast, "Gold 90-Day Price Forecast", "7. Precious Metals")

# Silver forecast plot
silver_recent <- df_metals %>% 
  filter(metal == "Silver", date >= max(date) - years(2))

p_silver_forecast <- ggplot() +
  geom_line(data = silver_recent, aes(x = date, y = close), color = "#808080", linewidth = 0.6) +
  geom_ribbon(data = silver_fc_df, aes(x = date, ymin = lower_95, ymax = upper_95), 
              fill = "#808080", alpha = 0.2) +
  geom_ribbon(data = silver_fc_df, aes(x = date, ymin = lower_80, ymax = upper_80), 
              fill = "#808080", alpha = 0.3) +
  geom_line(data = silver_fc_df, aes(x = date, y = forecast), color = "#2196F3", linewidth = 0.8, linetype = "dashed") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(
    title = sprintf("Silver (SI=F) - %d-Day Price Forecast", forecast_horizon),
    subtitle = sprintf("ETS Model: %s", silver_model$method),
    x = "Date",
    y = "Price (USD)",
    caption = "Shaded: 80% and 95% confidence intervals | Dashed: Point forecast"
  ) +
  theme_money_printer_go_brrr()

print(p_silver_forecast)

add_report_item("plot", p_silver_forecast, "Silver 90-Day Price Forecast", "7. Precious Metals")

# Forecast summary
cat("\n📊 90-Day Price Forecasts:\n")
forecast_summary <- data.frame(
  Metal = c("Gold", "Silver", "Palladium"),
  Current = c(last(gold_ts), last(silver_ts), last(palladium_ts)),
  Forecast_90d = c(last(gold_fc_df$forecast), last(silver_fc_df$forecast), last(palladium_fc_df$forecast)),
  Lower_95 = c(last(gold_fc_df$lower_95), last(silver_fc_df$lower_95), last(palladium_fc_df$lower_95)),
  Upper_95 = c(last(gold_fc_df$upper_95), last(silver_fc_df$upper_95), last(palladium_fc_df$upper_95))
) %>%
  mutate(
    Change_Pct = (Forecast_90d / Current - 1) * 100
  )

print(forecast_summary)

cat("\n✅ Forecasting analysis complete\n")

### 10.13 Precious Metals Summary Dashboard

In [ ]:
# ============================================================================
# 10.13 PRECIOUS METALS SUMMARY DASHBOARD
# ============================================================================

# Compile comprehensive summary table
summary_dashboard <- df_metals %>%
  group_by(metal, symbol) %>%
  filter(!is.na(close)) %>%
  summarise(
    # Price info
    current_price = last(close),
    price_52w_high = max(close[date >= max(date) - years(1)]),
    price_52w_low = min(close[date >= max(date) - years(1)]),
    pct_from_52w_high = (last(close) / max(close[date >= max(date) - years(1)]) - 1) * 100,
    
    # Returns
    return_1m = (last(close) / close[date == max(date) - months(1)][1] - 1) * 100,
    return_ytd = (last(close) / close[year(date) == year(max(date)) & date == min(date[year(date) == year(max(date))])][1] - 1) * 100,
    return_1y = (last(close) / close[date == max(date) - years(1)][1] - 1) * 100,
    
    # Technical
    current_rsi = last(RSI_14),
    current_macd = last(MACD_histogram),
    above_ma200 = last(close) > last(MA_200),
    
    # Risk
    volatility_30d = last(volatility_20d) * 100,
    max_drawdown = min(drawdown),
    
    .groups = "drop"
  ) %>%
  mutate(
    # Status indicators
    rsi_status = case_when(
      current_rsi >= 70 ~ "Overbought",
      current_rsi <= 30 ~ "Oversold",
      TRUE ~ "Neutral"
    ),
    macd_status = ifelse(current_macd > 0, "Bullish", "Bearish"),
    trend_status = ifelse(above_ma200, "Uptrend", "Downtrend")
  )

cat("\n" , strrep("=", 70), "\n")
cat("   📊 PRECIOUS METALS SUMMARY DASHBOARD\n")
cat(strrep("=", 70), "\n\n")

# Print formatted summary
for (i in 1:nrow(summary_dashboard)) {
  row <- summary_dashboard[i,]
  cat(sprintf("\n🥇 %s (%s)\n", row$metal, row$symbol))
  cat(strrep("-", 40), "\n")
  cat(sprintf("   Current Price:      $%,.2f\n", row$current_price))
  cat(sprintf("   52-Week Range:      $%,.2f - $%,.2f\n", row$price_52w_low, row$price_52w_high))
  cat(sprintf("   From 52W High:      %.1f%%\n", row$pct_from_52w_high))
  cat(sprintf("\n   📈 Returns:\n"))
  cat(sprintf("      1 Month:         %.1f%%\n", row$return_1m))
  cat(sprintf("      YTD:             %.1f%%\n", row$return_ytd))
  cat(sprintf("      1 Year:          %.1f%%\n", row$return_1y))
  cat(sprintf("\n   📊 Technical:\n"))
  cat(sprintf("      RSI(14):         %.1f (%s)\n", row$current_rsi, row$rsi_status))
  cat(sprintf("      MACD:            %s\n", row$macd_status))
  cat(sprintf("      Trend (MA200):   %s\n", row$trend_status))
  cat(sprintf("\n   ⚠️ Risk:\n"))
  cat(sprintf("      30D Volatility:  %.1f%%\n", row$volatility_30d))
  cat(sprintf("      Max Drawdown:    %.1f%%\n", row$max_drawdown))
}

# Create summary table for report
summary_table <- summary_dashboard %>%
  select(
    Metal = metal,
    Symbol = symbol,
    `Current Price` = current_price,
    `52W High` = price_52w_high,
    `52W Low` = price_52w_low,
    `1M Return` = return_1m,
    `YTD Return` = return_ytd,
    `1Y Return` = return_1y,
    RSI = current_rsi,
    `RSI Status` = rsi_status,
    `MACD Signal` = macd_status,
    Trend = trend_status,
    `Volatility (30d)` = volatility_30d,
    `Max Drawdown` = max_drawdown
  )

add_report_item("table", summary_table, "Precious Metals Summary Dashboard", "7. Precious Metals")

cat("\n\n", strrep("=", 70), "\n")
cat("✅ Precious Metals Analysis Section Complete\n")
cat(strrep("=", 70), "\n")

## Generate PowerPoint Report

Generate a timestamped PowerPoint presentation with all analysis results organized by section.

In [ ]:
# ============================================================================
# GENERATE POWERPOINT REPORT
# ============================================================================
print("\n=== GENERATING POWERPOINT REPORT ===")

# Show what content has been collected
cat(sprintf("\n📊 Total items collected for report: %d\n", length(report_content)))

# List items by section
if (length(report_content) > 0) {
  sections <- unique(sapply(report_content, function(x) x$section))
  for (section in sections) {
    section_items <- report_content[sapply(report_content, function(x) x$section == section)]
    cat(sprintf("\n📁 %s (%d items):\n", section, length(section_items)))
    for (item in section_items) {
      cat(sprintf("   - [%s] %s\n", item$type, item$title))
    }
  }
}

# Generate the PowerPoint report
if (length(report_content) > 0) {
  output_file <- generate_report(
    content_list = report_content,
    template_path = "nn_template.pptx",
    output_dir = getwd(),
    filename_prefix = "Financial_Analysis_Report",
    title = "Financial Data Analysis Report",
    subtitle = paste("Analysis Date:", format(Sys.Date(), "%B %d, %Y")),
    author = "Automated Analysis Pipeline"
  )
  
  cat(sprintf("\n✅ PowerPoint report generated: %s\n", output_file))
} else {
  cat("\n⚠️ No content collected for report. Run the analysis cells first.\n")
}